In [ ]:
!nvidia-smi

In [ ]:
!unzip -q egg.v1i.folder.zip -d data
!ls data

In [ ]:
import os
for root, dirs, files in os.walk("data"):
    print(root, len(files), "files")

In [ ]:
import cv2, glob
import matplotlib.pyplot as plt

paths = sorted(glob.glob("data/**/*.jpg", recursive=True))[:6]
print("Total images:", len(glob.glob("data/**/*.jpg", recursive=True)))

plt.figure(figsize=(15, 8))
for i, p in enumerate(paths):
    img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    plt.subplot(2, 3, i + 1)
    plt.imshow(img)
    plt.title(os.path.basename(p)[:25])
    plt.axis("off")
plt.show()

In [ ]:
import glob, os, collections

paths = sorted(glob.glob("data/**/*.jpg", recursive=True))

def source(p):
    return os.path.basename(p).split(".rf.")[0]

groups = collections.defaultdict(list)
for p in paths:
    groups[source(p)].append(p)

print("Total images:", len(paths))
print("Unique source photos:", len(groups))
print("Copies per source (copies: how many sources):",
      dict(collections.Counter(len(v) for v in groups.values())))

# What kinds of images are there?
kinds = collections.Counter(os.path.basename(p).split("_jpg")[0].rsplit("_", 1)[0] for p in paths)
print(kinds)

In [ ]:
!cat data/README.dataset.txt
!cat data/README.roboflow.txt

In [ ]:
import random, cv2
import matplotlib.pyplot as plt

random.seed(0)
sample = random.sample(paths, 12)

plt.figure(figsize=(16, 12))
for i, p in enumerate(sample):
    img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    plt.subplot(3, 4, i + 1)
    plt.imshow(img)
    plt.title(f"{os.path.basename(p)[:22]}\n{img.shape[1]}x{img.shape[0]}", fontsize=8)
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
!pip -q install git+https://github.com/facebookresearch/segment-anything.git
!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
!ls -lh sam_vit_h_4b8939.pth

In [ ]:
import torch
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

device = "cuda"
sam = sam_model_registry["vit_h"](checkpoint="sam_vit_h_4b8939.pth").to(device)

mask_gen = SamAutomaticMaskGenerator(
    sam,
    points_per_side=32,
    pred_iou_thresh=0.88,
    stability_score_thresh=0.92,
    min_mask_region_area=2000,
)
print("SAM loaded on", device)

In [ ]:
import numpy as np, random

random.seed(1)
test_imgs = random.sample(paths, 3)

plt.figure(figsize=(18, 12))
for r, p in enumerate(test_imgs):
    img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    masks = mask_gen.generate(img)
    masks = sorted(masks, key=lambda m: -m["area"])
    print(os.path.basename(p)[:25], "->", len(masks), "masks")

    overlay = img.copy()
    rng = np.random.default_rng(0)
    for m in masks:
        overlay[m["segmentation"]] = (0.5 * overlay[m["segmentation"]] +
                                      0.5 * rng.integers(0, 255, 3)).astype(np.uint8)

    plt.subplot(3, 2, 2 * r + 1); plt.imshow(img); plt.axis("off")
    plt.subplot(3, 2, 2 * r + 2); plt.imshow(overlay); plt.title(f"{len(masks)} masks"); plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
import pickle, time
from tqdm.auto import tqdm

cache = {}
t0 = time.time()
for p in tqdm(paths):
    img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    ms = mask_gen.generate(img)
    cache[p] = [{"seg": np.packbits(m["segmentation"]),
                 "area": m["area"], "iou": m["predicted_iou"]} for m in ms]
print("done in", round(time.time() - t0), "s")

with open("sam_cache.pkl", "wb") as f:
    pickle.dump(cache, f)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/egg_task
!cp sam_cache.pkl /content/drive/MyDrive/egg_task/

In [ ]:
print("images in cache:", len(cache), "of", len(paths))
print("missing:", len([p for p in paths if p not in cache]))

In [ ]:
import pickle, time
from tqdm.auto import tqdm

todo = [p for p in paths if p not in cache]
print("to process:", len(todo))

t0 = time.time()
for p in tqdm(todo):
    img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    ms = mask_gen.generate(img)
    cache[p] = [{"seg": np.packbits(m["segmentation"]),
                 "area": m["area"], "iou": m["predicted_iou"]} for m in ms]
    if len(cache) % 20 == 0:          # save progress every 20 images
        with open("sam_cache.pkl", "wb") as f:
            pickle.dump(cache, f)

with open("sam_cache.pkl", "wb") as f:
    pickle.dump(cache, f)
print("done in", round(time.time() - t0), "s; cache size:", len(cache))

In [ ]:
print(list(cache.keys())[:2])
print(paths[:2])

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/egg_task
!cp sam_cache.pkl /content/drive/MyDrive/egg_task/
!ls -lh /content/drive/MyDrive/egg_task/

In [ ]:
!pip -q install imagehash
import imagehash
from PIL import Image

hashes = {p: imagehash.phash(Image.open(p).convert("RGB")) for p in paths}
plist = list(hashes)
pairs = sorted(
    ((hashes[plist[i]] - hashes[plist[j]], plist[i], plist[j])
     for i in range(len(plist)) for j in range(i + 1, len(plist))),
    key=lambda x: x[0])

for d, a, b in pairs[:12]:
    print(d, os.path.basename(a)[:28], "|", os.path.basename(b)[:28])

plt.figure(figsize=(14, 14))
for k, (d, a, b) in enumerate(pairs[:4]):
    for m, pth in enumerate([a, b]):
        plt.subplot(4, 2, 2 * k + m + 1)
        plt.imshow(cv2.cvtColor(cv2.imread(pth), cv2.COLOR_BGR2RGB))
        plt.title(f"dist={d}  {os.path.basename(pth)[:25]}", fontsize=9)
        plt.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(16, 22))
for k, (d, a, b) in enumerate(pairs[1:13]):
    for m, pth in enumerate([a, b]):
        plt.subplot(6, 4, 4 * (k // 2) + 2 * (k % 2) + m + 1)
        plt.imshow(cv2.cvtColor(cv2.imread(pth), cv2.COLOR_BGR2RGB))
        plt.title(f"#{k+2} d={d}  {os.path.basename(pth)[14:22]}", fontsize=9)
        plt.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
import json, random

THRESH = 14   # hash distance at or below this = same group (adjust after Step 13)

# union-find to merge near-duplicate images into groups
parent = {p: p for p in paths}
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
for d, a, b in pairs:
    if d > THRESH:
        break
    parent[find(a)] = find(b)

groups = collections.defaultdict(list)
for p in paths:
    groups[find(p)].append(p)
groups = list(groups.values())
print("groups:", len(groups), "| sizes > 1:", [len(g) for g in groups if len(g) > 1])

# assign whole groups to eval until ~20% of images
random.seed(42)
random.shuffle(groups)
target_eval = round(0.2 * len(paths))   # 22
dev, ev = [], []
for g in groups:
    if len(ev) + len(g) <= target_eval:
        ev += g
    else:
        dev += g

print("dev:", len(dev), "eval:", len(ev))
assert not set(dev) & set(ev)

with open("split.json", "w") as f:
    json.dump({"dev": [os.path.basename(p) for p in dev],
               "eval": [os.path.basename(p) for p in ev],
               "hash_threshold": THRESH, "seed": 42}, f, indent=1)
!cp split.json /content/drive/MyDrive/egg_task/

In [ ]:
os.makedirs("outputs/sam_masks", exist_ok=True)
os.makedirs("outputs/sam_overlays", exist_ok=True)

results, flagged = {}, []
for p in paths:
    img, chosen, cands = pick_eggs(p)
    name = os.path.splitext(os.path.basename(p))[0]
    union = np.zeros(img.shape[:2], bool)
    for _, m in chosen:
        union |= m
    cv2.imwrite(f"outputs/sam_masks/{name}.png", union.astype(np.uint8) * 255)
    cv2.imwrite(f"outputs/sam_overlays/{name}.jpg",
                cv2.cvtColor(overlay_red(img, chosen), cv2.COLOR_RGB2BGR))
    best = chosen[0][0] if chosen else 0
    results[p] = (len(chosen), best)
    if len(chosen) != 1 or best < 0.45:
        flagged.append(p)

print("total:", len(paths), "| flagged for review:", len(flagged))
print("egg-count distribution:", collections.Counter(n for n, _ in results.values()))

# contact sheet of flagged images
n = min(len(flagged), 20)
if n:
    cols = 5; rows = (n + cols - 1) // cols
    plt.figure(figsize=(18, 3.6 * rows))
    for i, p in enumerate(flagged[:n]):
        img, chosen, _ = pick_eggs(p)
        plt.subplot(rows, cols, i + 1)
        plt.imshow(overlay_red(img, chosen))
        plt.title(f"{os.path.basename(p)[12:20]} n={results[p][0]} s={results[p][1]:.2f}", fontsize=8)
        plt.axis("off")
    plt.tight_layout(); plt.show()

In [ ]:
os.makedirs("outputs/sam_masks", exist_ok=True)
os.makedirs("outputs/sam_overlays", exist_ok=True)

results, flagged = {}, []
for p in paths:
    img, chosen, cands = pick_eggs(p)
    name = os.path.splitext(os.path.basename(p))[0]
    union = np.zeros(img.shape[:2], bool)
    for _, m in chosen:
        union |= m
    cv2.imwrite(f"outputs/sam_masks/{name}.png", union.astype(np.uint8) * 255)
    cv2.imwrite(f"outputs/sam_overlays/{name}.jpg",
                cv2.cvtColor(overlay_red(img, chosen), cv2.COLOR_RGB2BGR))
    best = chosen[0][0] if chosen else 0
    results[p] = (len(chosen), best)
    if len(chosen) != 1 or best < 0.45:
        flagged.append(p)

print("total:", len(paths), "| flagged for review:", len(flagged))
print("egg-count distribution:", collections.Counter(n for n, _ in results.values()))

# contact sheet of flagged images
n = min(len(flagged), 20)
if n:
    cols = 5; rows = (n + cols - 1) // cols
    plt.figure(figsize=(18, 3.6 * rows))
    for i, p in enumerate(flagged[:n]):
        img, chosen, _ = pick_eggs(p)
        plt.subplot(rows, cols, i + 1)
        plt.imshow(overlay_red(img, chosen))
        plt.title(f"{os.path.basename(p)[12:20]} n={results[p][0]} s={results[p][1]:.2f}", fontsize=8)
        plt.axis("off")
    plt.tight_layout(); plt.show()

In [ ]:
def score_mask2(mask, hsv, min_ell=0.88, min_sol=0.95, min_val=0.50, max_sat=0.40):
    frac = mask.mean()
    if frac < 0.015 or frac > 0.60:
        return 0.0
    cnts, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    c = max(cnts, key=cv2.contourArea)
    if len(c) < 5:
        return 0.0
    sol = cv2.contourArea(c) / (cv2.contourArea(cv2.convexHull(c)) + 1e-6)
    em = np.zeros(mask.shape, np.uint8)
    cv2.ellipse(em, cv2.fitEllipse(c), 1, -1)
    em = em.astype(bool)
    ell_iou = (mask & em).sum() / ((mask | em).sum() + 1e-6)
    sat = hsv[..., 1][mask].mean() / 255
    val = hsv[..., 2][mask].mean() / 255
    ys, xs = np.where(mask)
    touch = int(xs.min() <= 2) + int(xs.max() >= 637) + int(ys.min() <= 2) + int(ys.max() >= 637)
    if ell_iou < min_ell or sol < min_sol or val < min_val or sat > max_sat or touch >= 2:
        return 0.0
    return float(ell_iou * sol * val * (1 - sat) * (0.7 ** touch))

def pick_eggs2(p, rel_thresh=0.7, abs_thresh=0.30, max_overlap=0.2):
    img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    masks = [unpack(m) for m in cache[p]]
    cands = sorted(((score_mask2(m, hsv), m) for m in masks), key=lambda x: -x[0])
    chosen = []
    if cands and cands[0][0] > 0:
        best = cands[0][0]
        for s, m in cands:
            if s < abs_thresh or s < rel_thresh * best:
                break
            # overlap measured against the SMALLER mask, so nested masks are rejected
            if all((m & c).sum() / min(m.sum(), c.sum()) < max_overlap for _, c in chosen):
                chosen.append((s, m))
    return img, chosen, cands

def review_sheet(idx_list):
    cols = 5; rows = (len(idx_list) + cols - 1) // cols
    plt.figure(figsize=(20, 4.2 * rows))
    for k, i in enumerate(idx_list):
        img, chosen, _ = pick_eggs2(paths[i])
        vis = img.copy()
        for _, m in chosen:
            cs, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(vis, cs, -1, (255, 0, 0), 3)   # outline, so edges are visible
        plt.subplot(rows, cols, k + 1)
        plt.imshow(vis)
        plt.title(f"#{i}   eggs={len(chosen)}", fontsize=13)
        plt.axis("off")
    plt.tight_layout(); plt.show()

counts = [len(pick_eggs2(p)[1]) for p in paths]
print("egg-count distribution (v2):", collections.Counter(counts))
print("images with 0 eggs found:", [i for i, c in enumerate(counts) if c == 0])

for start in range(0, len(paths), 20):
    review_sheet(list(range(start, min(start + 20, len(paths)))))

In [ ]:
from segment_anything import SamPredictor
predictor = SamPredictor(sam)

def show_grid(i):
    img = cv2.cvtColor(cv2.imread(paths[i]), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(9, 9))
    plt.imshow(img)
    plt.xticks(range(0, 641, 40)); plt.yticks(range(0, 641, 40))
    plt.grid(True, color="yellow", alpha=0.35, linewidth=0.6)
    plt.title(f"#{i}  (x right, y down)")
    plt.show()

def sam_from_boxes(i, boxes):
    img = cv2.cvtColor(cv2.imread(paths[i]), cv2.COLOR_BGR2RGB)
    predictor.set_image(img)
    out = []
    for b in boxes:
        m, sc, _ = predictor.predict(box=np.array(b), multimask_output=False)
        out.append((float(sc[0]), m[0]))
    return img, out

In [ ]:
show_grid(57)

In [ ]:
overrides = {}                       # keep this line only once, at the start
overrides[57] = [[300, 250, 520, 360]]    # <- replace with your numbers

img, chosen = sam_from_boxes(57, overrides[57])
vis = img.copy()
for _, m in chosen:
    cs, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(vis, cs, -1, (255, 0, 0), 3)
plt.figure(figsize=(7, 7)); plt.imshow(vis); plt.axis("off"); plt.show()

In [ ]:
overrides = globals().get("overrides", {})     # won't wipe earlier fixes if re-run

def fix(i, boxes, neg_points=None):
    img = cv2.cvtColor(cv2.imread(paths[i]), cv2.COLOR_BGR2RGB)
    predictor.set_image(img)
    masks = []
    for b in boxes:
        kw = {}
        if neg_points:
            kw["point_coords"] = np.array(neg_points)
            kw["point_labels"] = np.zeros(len(neg_points), dtype=int)   # 0 = exclude
        m, sc, _ = predictor.predict(box=np.array(b), multimask_output=False, **kw)
        masks.append(m[0])
    overrides[i] = [(1.0, m) for m in masks]
    vis = img.copy()
    for m in masks:
        cs, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(vis, cs, -1, (255, 0, 0), 3)
    plt.figure(figsize=(7, 7)); plt.imshow(vis); plt.axis("off")
    plt.title(f"#{i} override ({len(masks)} egg)"); plt.show()

fix(57, [[185, 170, 540, 380]])

In [ ]:
show_grid(61)


In [ ]:
fix(61, [[250, 165, 400, 300]])

In [ ]:
fix(61, [[250, 165, 400, 300]], neg_points=[[255, 215], [330, 320]])

In [ ]:
fix(61, [[165, 135, 350, 300]])

In [ ]:
fix(61, [[165, 135, 350, 300]], neg_points=[[290, 335]])

In [ ]:
def fix_ellipse(i, cx, cy, ax, ay, angle=0):
    img = cv2.cvtColor(cv2.imread(paths[i]), cv2.COLOR_BGR2RGB)
    m = np.zeros(img.shape[:2], np.uint8)
    cv2.ellipse(m, (cx, cy), (ax, ay), angle, 0, 360, 1, -1)
    overrides[i] = [(1.0, m.astype(bool))]
    vis = img.copy()
    cs, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(vis, cs, -1, (255, 0, 0), 3)
    plt.figure(figsize=(7, 7)); plt.imshow(vis); plt.axis("off"); plt.show()

fix_ellipse(61, 257, 217, 87, 77)

In [ ]:
fix(61, [[165, 135, 350, 300]])

In [ ]:
import pickle, shutil

def final_masks(i):
    """Overrides win; otherwise use the v2 picker."""
    if i in overrides:
        return [m for _, m in overrides[i]]
    _, chosen, _ = pick_eggs2(paths[i])
    return [m for _, m in chosen]

os.makedirs("outputs/task1_masks", exist_ok=True)
os.makedirs("outputs/task1_overlays", exist_ok=True)

summary = {}
for i, p in enumerate(paths):
    img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    ms = final_masks(i)
    union = np.zeros(img.shape[:2], bool)
    for m in ms:
        union |= m
    name = os.path.splitext(os.path.basename(p))[0]
    cv2.imwrite(f"outputs/task1_masks/{name}.png", union.astype(np.uint8) * 255)
    ov = img.copy()
    ov[union] = (0.35 * ov[union] + 0.65 * np.array([255, 40, 40])).astype(np.uint8)
    cv2.imwrite(f"outputs/task1_overlays/{name}.jpg", cv2.cvtColor(ov, cv2.COLOR_RGB2BGR))
    summary[name] = len(ms)

print("saved:", len(summary), "masks | egg-count distribution:", collections.Counter(summary.values()))
print("images with 0 eggs:", [n for n, c in summary.items() if c == 0])
print("manual overrides:", sorted(overrides))

# back up to Drive
shutil.make_archive("task1_outputs", "zip", "outputs")
with open("overrides.pkl", "wb") as f:
    pickle.dump({k: [m for _, m in v] for k, v in overrides.items()}, f)
!cp task1_outputs.zip overrides.pkl /content/drive/MyDrive/egg_task/

In [ ]:
zero_names = ['not_damaged_125_', 'not_damaged_145_', 'not_damaged_10_', 'not_damaged_12_',
              'not_damaged_13_', 'not_damaged_24_', 'not_damaged_44_', 'not_damaged_119_',
              'not_damaged_11_']
zero_idx = [i for i, p in enumerate(paths)
            if any(os.path.basename(p).startswith(z) for z in zero_names)]
print("indices:", zero_idx)

plt.figure(figsize=(20, 20))
for k, i in enumerate(zero_idx):
    img = cv2.cvtColor(cv2.imread(paths[i]), cv2.COLOR_BGR2RGB)
    plt.subplot(3, 3, k + 1)
    plt.imshow(img)
    plt.xticks(range(0, 641, 80)); plt.yticks(range(0, 641, 80))
    plt.grid(True, color="yellow", alpha=0.4, linewidth=0.7)
    plt.title(f"#{i}", fontsize=16)
plt.tight_layout(); plt.show()

In [ ]:
multi = [i for i, p in enumerate(paths) if len(final_masks(i)) >= 2]
print("multi-egg images:", multi)
review_sheet(multi)   # uses pick_eggs2; overrides are not shown here, which is fine

In [ ]:
def sam_masks(i, boxes):
    img = cv2.cvtColor(cv2.imread(paths[i]), cv2.COLOR_BGR2RGB)
    predictor.set_image(img)
    return [predictor.predict(box=np.array(b), multimask_output=False)[0][0] for b in boxes]

def replace(i, boxes):                       # discard picker masks, use these boxes
    overrides[i] = [(1.0, m) for m in sam_masks(i, boxes)]

def add(i, boxes):                           # keep current masks, add missed eggs (run ONCE per image)
    overrides[i] = [(1.0, m) for m in final_masks(i) + sam_masks(i, boxes)]

def check(idx):
    cols = 4; rows = (len(idx) + cols - 1) // cols
    plt.figure(figsize=(20, 5 * rows))
    for k, i in enumerate(idx):
        img = cv2.cvtColor(cv2.imread(paths[i]), cv2.COLOR_BGR2RGB)
        vis = img.copy()
        ms = final_masks(i)
        for m in ms:
            cs, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(vis, cs, -1, (255, 0, 0), 3)
        plt.subplot(rows, cols, k + 1); plt.imshow(vis)
        plt.title(f"#{i}  eggs={len(ms)}", fontsize=14); plt.axis("off")
    plt.tight_layout(); plt.show()

for i, b in {1: [[215,185,375,330]], 12: [[250,280,430,430]], 19: [[135,270,420,440]],
             40: [[295,150,462,260]], 80: [[60,60,640,400]], 92: [[110,205,415,515]],
             93: [[180,140,400,315]], 25: [[330,335,555,515]], 38: [[195,228,498,462]]}.items():
    replace(i, b)

add(37, [[0, 345, 240, 555]])
add(42, [[155, 80, 365, 280]])
add(43, [[165, 15, 395, 150]])

check([1, 12, 19, 40, 80, 92, 93, 25, 38, 37, 42, 43])

In [ ]:
print("Number of masks:", len(final_masks(80)))

In [ ]:
fix(80, [[60, 60, 560, 400]])

In [ ]:
show_grid(93)

In [ ]:
show_grid(34)

In [ ]:
show_grid(64)

In [ ]:
show_grid(79)

In [ ]:
[240, 260, 430, 455]


In [ ]:
replace(34, [[240, 260, 430, 455]])

In [ ]:
replace(64, [[145, 190, 450, 410]])

In [ ]:
replace(34, [[240, 260, 430, 455]])
replace(64, [[145, 190, 450, 410]])

check([34, 64])

In [ ]:
replace(34, [[260, 275, 420, 445]])
check([34])


In [ ]:
replace(34, [[275, 280, 415, 440]])
check([34])

In [ ]:
def fix_with_points(i, box, positive_points, negative_points):
    img = cv2.cvtColor(cv2.imread(paths[i]), cv2.COLOR_BGR2RGB)
    predictor.set_image(img)

    points = np.array(positive_points + negative_points)
    labels = np.array([1] * len(positive_points) + [0] * len(negative_points))

    masks, scores, _ = predictor.predict(
        box=np.array(box),
        point_coords=points,
        point_labels=labels,
        multimask_output=True
    )

    best = int(np.argmax(scores))
    overrides[i] = [(float(scores[best]), masks[best])]

fix_with_points(
    34,
    [250, 260, 435, 455],
    positive_points=[[340, 340], [340, 385]],
    negative_points=[[280, 330], [285, 415], [400, 430]]
)

check([34])

In [ ]:
import matplotlib.pyplot as plt
import cv2
import numpy as np

i = 34
img = cv2.cvtColor(cv2.imread(paths[i]), cv2.COLOR_BGR2RGB)
mask = final_masks(i)

print("Number of masks:", len(mask))

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.imshow(img)
plt.title("Original image #34")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(img)
plt.imshow(mask[0], alpha=0.45, cmap="Reds")
plt.title("Current SAM mask")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

i = 34
img = cv2.cvtColor(cv2.imread(paths[i]), cv2.COLOR_BGR2RGB)

# Get the current mask
mask = final_masks(i)[0].astype(np.uint8)

# Fill small internal gaps
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
refined = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

# Keep the largest connected component
num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(refined)

if num_labels > 1:
    largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    refined = (labels == largest).astype(np.uint8)

# Compare original and refined masks
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(img)
plt.title("Original image")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(mask, cmap="gray")
plt.title("Original SAM mask")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(img)
plt.imshow(refined, alpha=0.45, cmap="Reds")
plt.title("Refined mask")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))

refined2 = cv2.morphologyEx(
    mask,
    cv2.MORPH_CLOSE,
    kernel,
    iterations=1
)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(img)
plt.imshow(mask, alpha=0.4, cmap="Reds")
plt.title("Original SAM mask")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(img)
plt.imshow(refined2, alpha=0.4, cmap="Reds")
plt.title("Larger closing kernel")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
check([19, 25, 37, 38, 42, 43, 80])

In [ ]:
replace(19, [[300, 320, 525, 510]])
check([19])

In [ ]:
replace(25, [[330, 335, 555, 515]])
check([25])


In [ ]:
replace(25, [[330, 335, 555, 515]])
check([25])

In [ ]:
check([1, 12, 25, 34, 37, 38, 40, 42, 43, 64, 79, 80, 92, 93])

In [ ]:
import cv2
import matplotlib.pyplot as plt

i = 19
img = cv2.cvtColor(cv2.imread(paths[i]), cv2.COLOR_BGR2RGB)

# Proposed box around the visible white egg.
box = [100, 250, 400, 490]

preview = img.copy()
x1, y1, x2, y2 = box
cv2.rectangle(preview, (x1, y1), (x2, y2), (255, 0, 0), 3)

plt.figure(figsize=(8, 6))
plt.imshow(preview)
plt.title("Image #19 — verify bounding box")
plt.axis("off")
plt.show()

In [ ]:
replace(19, [[100, 250, 400, 490]])
check([19])

In [ ]:
i = 25
img = cv2.cvtColor(cv2.imread(paths[i]), cv2.COLOR_BGR2RGB)

box = [300, 330, 520, 530]

preview = img.copy()
x1, y1, x2, y2 = box
cv2.rectangle(preview, (x1, y1), (x2, y2), (255, 0, 0), 3)

plt.figure(figsize=(8, 6))
plt.imshow(preview)
plt.title("Image #25 — verify bounding box")
plt.axis("off")
plt.show()

In [ ]:
replace(25, [[300, 330, 520, 530]])
check([25])

In [ ]:
show_grid(79)

In [ ]:
replace(19, [[100, 250, 400, 490]])
check([19])

In [ ]:
replace(25, [[400, 390, 590, 550]])
check([25])

In [ ]:
check([1, 12, 25, 34, 37, 38, 40, 42, 43, 64, 79, 80, 92, 93])

In [ ]:
i = 25

img = cv2.cvtColor(cv2.imread(paths[i]), cv2.COLOR_BGR2RGB)

plt.figure(figsize=(8, 8))
plt.imshow(img)
plt.title("Image #25 — inspect egg boundaries")
plt.axis("on")
plt.show()

print("Image dimensions (height, width):", img.shape[:2])

In [ ]:
show_grid(1)
show_grid(19)
show_grid(34)
show_grid(37)
show_grid(40)
show_grid(64)
show_grid(79)
show_grid(93)

In [ ]:
show_grid(25)

In [ ]:
def sam_masks(i, boxes):
    img = cv2.cvtColor(cv2.imread(paths[i]), cv2.COLOR_BGR2RGB)
    predictor.set_image(img)

    masks = []
    for b in boxes:
        mask, _, _ = predictor.predict(
            box=np.array(b),
            multimask_output=False
        )
        masks.append(mask[0])

    return masks


def replace(i, boxes):
    overrides[i] = [(1.0, m) for m in sam_masks(i, boxes)]


def add(i, boxes):
    overrides[i] = [(1.0, m) for m in final_masks(i) + sam_masks(i, boxes)]


def check(indices):
    cols = 4
    rows = (len(indices) + cols - 1) // cols

    plt.figure(figsize=(20, 5 * rows))

    for k, i in enumerate(indices):
        img = cv2.cvtColor(cv2.imread(paths[i]), cv2.COLOR_BGR2RGB)
        vis = img.copy()

        for m in final_masks(i):
            contours, _ = cv2.findContours(
                m.astype(np.uint8),
                cv2.RETR_EXTERNAL,
                cv2.CHAIN_APPROX_SIMPLE
            )
            cv2.drawContours(vis, contours, -1, (255, 0, 0), 3)

        plt.subplot(rows, cols, k + 1)
        plt.imshow(vis)
        plt.title(f"#{i}  eggs={len(final_masks(i))}")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def sam_masks(i, boxes):
    img = cv2.cvtColor(cv2.imread(paths[i]), cv2.COLOR_BGR2RGB)
    predictor.set_image(img)

    masks = []
    for box in boxes:
        mask, scores, logits = predictor.predict(
            box=np.array(box),
            multimask_output=False
        )
        masks.append(mask[0])

    return masks


def replace(i, boxes):
    overrides[i] = [(1.0, m) for m in sam_masks(i, boxes)]
    print(f"Image #{i} updated with {len(boxes)} mask(s).")

In [ ]:
replace(19, [[140, 275, 415, 470]])

In [ ]:
required = ["paths", "predictor", "overrides", "cv2", "np"]

for name in required:
    print(f"{name}: {'Available' if name in globals() else 'MISSING'}")

In [ ]:
for name in ["paths", "predictor", "overrides", "cv2", "np"]:
    print(f"{name}: {'Available' if name in globals() else 'MISSING'}")

if "paths" in globals():
    print("Number of images:", len(paths))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

folder = "/content/drive/MyDrive/egg_task"

print("Folder exists:", os.path.exists(folder))

if os.path.exists(folder):
    for name in os.listdir(folder):
        print(name)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pickle
import json
import glob

folder = "/content/drive/MyDrive/egg_task"

print("Files available:")
for name in os.listdir(folder):
    print(name)

In [ ]:
import pickle
import os

folder = "/content/drive/MyDrive/egg_task"
override_path = os.path.join(folder, "overrides.pkl")

with open(override_path, "rb") as f:
    overrides = pickle.load(f)

print("Overrides loaded:", len(overrides))
print("Override image indices:", sorted(overrides.keys()))

In [ ]:
import os

for root, dirs, files in os.walk("/content"):
    dirs[:] = [
        d for d in dirs
        if d not in {".git", "__pycache__", "site-packages"}
    ]

    image_files = [
        f for f in files
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    if image_files:
        print("Folder:", root)
        print("Images found:", len(image_files))
        print("Examples:", image_files[:3])

In [ ]:
import os

base = "/content/drive/MyDrive"

for root, dirs, files in os.walk(base):
    dirs[:] = [
        d for d in dirs
        if d not in {".git", "__pycache__"}
    ]

    image_files = [
        f for f in files
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    # Show folders with a substantial number of images
    if len(image_files) >= 10:
        print(f"\nFolder: {root}")
        print(f"Image count: {len(image_files)}")
        print("Examples:", image_files[:5])

In [ ]:
import json

split_path = "/content/drive/MyDrive/egg_task/split.json"

with open(split_path, "r") as f:
    split_data = json.load(f)

print("Split file structure:")
print(type(split_data))

if isinstance(split_data, dict):
    print("Keys:", list(split_data.keys()))

print(str(split_data)[:3000])

In [ ]:
import json
import os

folder = "/content/drive/MyDrive/egg_task"

with open(os.path.join(folder, "split.json"), "r") as f:
    split_data = json.load(f)

dev_names = split_data["dev"]
eval_names = split_data["eval"]

print("Dev images:", len(dev_names))
print("Eval images:", len(eval_names))
print("Total:", len(dev_names) + len(eval_names))

print("\nFirst 5 filenames:")
for name in (dev_names + eval_names)[:5]:
    print(name)

In [ ]:
import os

folder = "/content/drive/MyDrive/egg_task"
all_names = split_data["dev"] + split_data["eval"]

matches = {}

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    file_set = set(files)

    found = [name for name in all_names if name in file_set]

    if found:
        matches[root] = found

print("Matching folders:", len(matches))

for root, found in matches.items():
    print(f"\nFolder: {root}")
    print(f"Matching dataset images: {len(found)}/110")
    print("Examples:", found[:3])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pickle
import json
import glob

folder = "/content/drive/MyDrive/egg_task"

print("Backup files:")
for name in os.listdir(folder):
    print(name)

In [ ]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    dirs[:] = [d for d in dirs if d not in {
        ".git", "__pycache__", "site-packages"
    }]

    images = [
        f for f in files
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    if len(images) >= 80:
        print(f"\nFolder: {root}")
        print(f"Images found: {len(images)}")
        print("Examples:", images[:5])

In [ ]:
import os

search_roots = [
    "/content",
    "/content/drive/MyDrive"
]

for base in search_roots:
    print(f"\nSearching: {base}")

    for root, dirs, files in os.walk(base):
        dirs[:] = [
            d for d in dirs
            if d not in {
                ".git", "__pycache__", "site-packages"
            }
        ]

        images = [
            f for f in files
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ]

        if len(images) >= 100:
            print(f"\nPossible dataset: {root}")
            print(f"Image count: {len(images)}")
            print("Examples:", images[:5])

In [ ]:
import os
import pickle

cache_path = "/content/drive/MyDrive/egg_task/sam_cache.pkl"

print("Cache exists:", os.path.exists(cache_path))
print("Cache size (MB):", round(os.path.getsize(cache_path) / (1024**2), 2))

with open(cache_path, "rb") as f:
    cache = pickle.load(f)

print("Cache type:", type(cache))

if isinstance(cache, dict):
    print("Cached entries:", len(cache))
    print("Example keys:", list(cache.keys())[:5])
elif isinstance(cache, (list, tuple)):
    print("Cached entries:", len(cache))

In [ ]:
# Inspect the first cache entry
first_key = next(iter(cache))
first_value = cache[first_key]

print("First cache key:")
print(first_key)

print("\nCache value type:")
print(type(first_value))

print("\nCache value details:")
if isinstance(first_value, dict):
    print("Keys:", list(first_value.keys()))

    for key, value in first_value.items():
        print(
            f"{key}: type={type(value)}, "
            f"shape={getattr(value, 'shape', None)}"
        )

elif isinstance(first_value, (list, tuple)):
    print("Number of elements:", len(first_value))

    for i, value in enumerate(first_value[:5]):
        print(
            i,
            type(value),
            getattr(value, "shape", None)
        )

else:
    print("Shape:", getattr(first_value, "shape", None))

In [ ]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/egg_task/task1_outputs.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    files = z.namelist()

    print("Total files in ZIP:", len(files))
    print("\nFirst 20 files:")

    for name in files[:20]:
        print(name)

    print("\nTop-level folders:")
    print(sorted(set(name.split("/")[0] for name in files)))

In [ ]:
import os
import zipfile

zip_path = "/content/drive/MyDrive/egg_task/task1_outputs.zip"
output_dir = "/content/egg_task_recovered"

os.makedirs(output_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(output_dir)

print("Extraction complete!")
print("Output directory:", output_dir)

for folder in os.listdir(output_dir):
    path = os.path.join(output_dir, folder)

    if os.path.isdir(path):
        print(f"{folder}: {len(os.listdir(path))} files")

In [ ]:
first_key = next(iter(cache))
first_value = cache[first_key]

print("Cache key:", first_key)
print("\nFirst cache item:")

for key, value in first_value[0].items():
    print(
        f"{key}: type={type(value)}, "
        f"shape={getattr(value, 'shape', None)}"
    )

print("\nAvailable cache keys:", len(cache))

In [ ]:
```python
from google.colab import drive
drive.mount('/content/drive')

import os
import pickle
import json
import zipfile

folder = "/content/drive/MyDrive/egg_task"

print("Backup folder exists:", os.path.exists(folder))
print("\nSaved files:")

for name in os.listdir(folder):
    path = os.path.join(folder, name)
    print(f"{name}: {os.path.getsize(path) / (1024**2):.2f} MB")
```

In [ ]:
```python


In [ ]:

**Keep the Python code between them.**

Your cell should start exactly like this:

```python
from google.colab import drive
drive.mount('/content/drive')

import os
import pickle
import json
import zipfile

folder = "/content/drive/MyDrive/egg_task"

print("Backup folder exists:", os.path.exists(folder))
print("\nSaved files:")

for name in os.listdir(folder):
    path = os.path.join(folder, name)
    print(f"{name}: {os.path.getsize(path) / (1024**2):.2f} MB")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

folder = "/content/drive/MyDrive/egg_task"

print("Backup folder exists:", os.path.exists(folder))

if os.path.exists(folder):
    print("\nSaved files:")
    for name in os.listdir(folder):
        print(name)

In [ ]:
import os
import pickle

folder = "/content/drive/MyDrive/egg_task"

with open(os.path.join(folder, "overrides.pkl"), "rb") as f:
    overrides = pickle.load(f)

with open(os.path.join(folder, "sam_cache.pkl"), "rb") as f:
    cache = pickle.load(f)

print("Overrides loaded:", len(overrides))
print("Override image indices:", sorted(overrides.keys()))
print("Cached images:", len(cache))

In [ ]:
print("Dataset images:", len(paths))
print("Cached images:", len(cache))
print("Existing overrides:", sorted(overrides.keys()))

print("\nAvailable helper functions:")
for name in ["final_masks", "show_grid", "fix"]:
    print(name, "available:", name in globals())

In [ ]:
print("Current variables:")
for name in ["paths", "cache", "overrides", "split_data",
             "predictor", "show_grid", "final_masks"]:
    print(f"{name}: {'Available' if name in globals() else 'Missing'}")

In [ ]:
import os
import json

split_path = "/content/drive/MyDrive/egg_task/split.json"

print("Split file exists:", os.path.exists(split_path))

with open(split_path, "r") as f:
    split_data = json.load(f)

print("Split keys:", split_data.keys())
print("Dev images:", len(split_data["dev"]))
print("Eval images:", len(split_data["eval"]))

In [ ]:
import os
import json

split_path = "/content/drive/MyDrive/egg_task/split.json"

with open(split_path, "r") as f:
    split_data = json.load(f)

# Reconstruct the image paths from the saved split.
all_names = split_data["dev"] + split_data["eval"]

# Search for the original images.
search_roots = [
    "/content/drive/MyDrive",
    "/content/drive",
    "/content"
]

file_lookup = {}

for root in search_roots:
    if not os.path.isdir(root):
        continue

    for current_root, dirs, files in os.walk(root):
        dirs[:] = [
            d for d in dirs
            if d not in {
                ".git", "__pycache__", "site-packages",
                ".ipynb_checkpoints"
            }
        ]

        for filename in files:
            if filename.lower().endswith((".jpg", ".jpeg", ".png")):
                file_lookup.setdefault(filename, os.path.join(current_root, filename))

paths = []

for name in all_names:
    if name not in file_lookup:
        print("MISSING:", name)
    else:
        paths.append(file_lookup[name])

print("\nPaths recovered:", len(paths))
print("Expected:", len(all_names))
print("First 3 paths:")
for p in paths[:3]:
    print(p)

In [ ]:
import cv2

for i in [57, 61]:
    img = cv2.imread(paths[i])
    print(
        f"Image #{i}:",
        os.path.basename(paths[i]),
        "| Shape:",
        None if img is None else img.shape
    )

In [ ]:
import os

print("Checking the recovered paths:\n")

for i in [57, 61]:
    print(f"\nImage #{i}")
    print("Current path:", paths[i])

print("\nSearching for original dataset images...")

for root in [
    "/content/drive/MyDrive",
    "/content/egg_task_recovered",
    "/content"
]:
    if not os.path.isdir(root):
        continue

    for current_root, dirs, files in os.walk(root):
        dirs[:] = [
            d for d in dirs
            if d not in {
                ".git", "__pycache__", "site-packages",
                ".ipynb_checkpoints", "sam_overlays",
                "task1_overlays"
            }
        ]

        image_files = [
            f for f in files
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ]

        if image_files and (
            "egg" in current_root.lower()
            or "test" in current_root.lower()
            or "train" in current_root.lower()
        ):
            print(
                current_root,
                "| image count:",
                len(image_files),
                "| example:",
                image_files[:2]
            )

In [ ]:
import os
import pickle
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from google.colab import drive

drive.mount("/content/drive")

# Locate the recovered project
ROOT = "/content/egg_task_recovered"
BACKUP = "/content/drive/MyDrive/egg_task"

print("Recovered project exists:", os.path.isdir(ROOT))
print("Drive backup exists:", os.path.isdir(BACKUP))

# Find the dataset images
image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
paths = []

for current_root, dirs, files in os.walk(ROOT):
    dirs[:] = [
        d for d in dirs
        if d not in {"task1_masks", "task1_overlays", "sam_masks", "sam_overlays"}
    ]

    for name in files:
        if name.lower().endswith(image_extensions):
            full_path = os.path.join(current_root, name)

            # Exclude generated output images
            if any(
                part in {"task1_masks", "task1_overlays", "sam_masks", "sam_overlays"}
                for part in full_path.split(os.sep)
            ):
                continue

            paths.append(full_path)

# Keep only unique paths and sort them consistently
paths = sorted(set(paths))

print("Image files found:", len(paths))
print("Example paths:")
for p in paths[:5]:
    print(p)

# Load the saved train/evaluation split
split_path = os.path.join(BACKUP, "split.json")

with open(split_path, "r") as f:
    split_data = json.load(f)

print("\nSplit keys:", split_data.keys())
print("Dev images:", len(split_data["dev"]))
print("Evaluation images:", len(split_data["eval"]))

# Load cached SAM masks
cache_path = os.path.join(BACKUP, "sam_cache.pkl")

with open(cache_path, "rb") as f:
    cache = pickle.load(f)

print("\nCached images:", len(cache))
print("Sample cache key:", next(iter(cache)))

In [ ]:
import os
import glob

# Search the recovered dataset for the original images
search_roots = [
    "/content/egg_task_recovered",
    "/content/drive/MyDrive/egg_task",
    "/content"
]

image_extensions = ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG")
image_paths = []

for root in search_roots:
    if not os.path.exists(root):
        continue

    for ext in image_extensions:
        image_paths.extend(
            glob.glob(os.path.join(root, "**", ext), recursive=True)
        )

# Keep only original dataset images, excluding masks and overlays
image_paths = sorted(set(
    p for p in image_paths
    if "not_damaged_" in os.path.basename(p).lower()
    and not any(
        folder in p.lower()
        for folder in ["sam_masks", "sam_overlays", "task1_masks",
                       "task1_overlays", "sample_data"]
    )
))

print("Original images found:", len(image_paths))
print("\nFirst 5 paths:")
for p in image_paths[:5]:
    print(p)

In [ ]:
paths = image_paths

print("Dataset images:", len(paths))
print("Image 57:", os.path.basename(paths[57]))
print("Image 61:", os.path.basename(paths[61]))

In [ ]:
import pickle

BACKUP = "/content/drive/MyDrive/egg_task"

with open(os.path.join(BACKUP, "overrides.pkl"), "rb") as f:
    overrides = pickle.load(f)

with open(os.path.join(BACKUP, "sam_cache.pkl"), "rb") as f:
    cache = pickle.load(f)

print("Cached images:", len(cache))
print("Saved overrides:", sorted(overrides.keys()))

In [ ]:
import os

folders_to_check = [
    "/content/egg_task_recovered",
    "/content/egg_task_recovered/task1_masks",
    "/content/egg_task_recovered/task1_overlays",
    "/content/egg_task_recovered/sam_masks",
    "/content/egg_task_recovered/sam_overlays",
    "/content/drive/MyDrive/egg_task",
]

for folder in folders_to_check:
    print(f"\nFolder: {folder}")
    print("Exists:", os.path.exists(folder))

    if os.path.isdir(folder):
        items = os.listdir(folder)
        print("Number of items:", len(items))
        print("First 8 items:", items[:8])

In [ ]:
import os

root = "/content/drive/MyDrive"

for current_root, dirs, files in os.walk(root):
    # Avoid searching large unrelated folders unnecessarily
    dirs[:] = [
        d for d in dirs
        if d not in [".ipynb_checkpoints", ".git"]
    ]

    image_files = [
        f for f in files
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
        and "not_damaged_" in f.lower()
    ]

    if image_files:
        print("\nFolder:", current_root)
        print("Matching images:", len(image_files))
        print("Examples:", image_files[:5])

In [ ]:
import os
import glob

# Search the recovered project for original images
search_roots = [
    "/content/egg_task_recovered",
    "/content/drive/MyDrive/egg_task",
    "/content",
]

image_lookup = {}

for root in search_roots:
    if not os.path.exists(root):
        continue

    for current_root, dirs, files in os.walk(root):
        # Skip output folders to avoid treating overlays as original images
        dirs[:] = [
            d for d in dirs
            if d not in {".ipynb_checkpoints", ".git"}
        ]

        for name in files:
            lower = name.lower()

            if (
                lower.startswith("not_damaged_")
                and lower.endswith((".jpg", ".jpeg", ".png"))
                and "overlay" not in lower
            ):
                full_path = os.path.join(current_root, name)

                # Keep the first copy of each filename
                image_lookup.setdefault(name, full_path)

# Prefer the original dataset paths recorded in the SAM cache
cache_names = sorted({
    key.split("/")[-1]
    for key in cache.keys()
    if isinstance(key, str) and key.lower().endswith(
        (".jpg", ".jpeg", ".png")
    )
})

paths = [
    image_lookup[name]
    for name in cache_names
    if name in image_lookup
]

print("Images recovered:", len(paths))
print("Expected images: 110")
print("\nFirst five paths:")

for p in paths[:5]:
    print(p)

missing = [name for name in cache_names if name not in image_lookup]
print("\nImages still missing:", len(missing))

if missing:
    print("Missing examples:", missing[:10])

In [ ]:
print("Dataset paths:", len(paths))
print("Cached images:", len(cache))
print("Saved overrides:", sorted(overrides.keys()))

for name in ["predictor", "final_masks", "show_grid", "fix"]:
    print(f"{name}:", "Available" if name in globals() else "Missing")

In [ ]:
import os
import pickle
import json
import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

# Project locations
BACKUP = "/content/drive/MyDrive/egg_task"
ROOT = "/content/egg_task_recovered"

# Make sure Google Drive is mounted
from google.colab import drive
drive.mount("/content/drive")

# Load saved overrides and SAM cache
with open(os.path.join(BACKUP, "overrides.pkl"), "rb") as f:
    overrides = pickle.load(f)

with open(os.path.join(BACKUP, "sam_cache.pkl"), "rb") as f:
    cache = pickle.load(f)

# Load train/evaluation split
with open(os.path.join(BACKUP, "split.json"), "r") as f:
    split_data = json.load(f)

# Locate the recovered dataset images
image_paths = glob.glob(
    "/content/egg_task_recovered/**/*.jpg", recursive=True
)
image_paths += glob.glob(
    "/content/egg_task_recovered/**/*.jpeg", recursive=True
)
image_paths += glob.glob(
    "/content/egg_task_recovered/**/*.png", recursive=True
)

# Keep original dataset images, not generated overlays or masks
image_paths = [
    p for p in image_paths
    if "not_damaged_" in os.path.basename(p).lower()
    and not any(
        folder in p.lower()
        for folder in ["task1_masks", "task1_overlays", "sam_masks", "sam_overlays"]
    )
]

# Remove duplicate paths and sort consistently
paths = sorted(set(image_paths))

print("Recovered image paths:", len(paths))
print("Cached SAM images:", len(cache))
print("Saved overrides:", sorted(overrides.keys()))
print("Development images:", len(split_data["dev"]))
print("Evaluation images:", len(split_data["eval"]))

assert len(paths) == 110, (
    f"Expected 110 images, but found {len(paths)}. "
    "Stop here and send me the output."
)

print("\nFirst three images:")
for p in paths[:3]:
    print(os.path.basename(p))

In [ ]:
import os

ROOT = "/content/egg_task_recovered"

print("Searching for original dataset images...\n")

for current_root, dirs, files in os.walk(ROOT):
    dirs[:] = [
        d for d in dirs
        if d not in [".ipynb_checkpoints", ".git"]
    ]

    matches = [
        f for f in files
        if f.lower().startswith("not_damaged_")
        and f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    if matches:
        print("Folder:", current_root)
        print("Number of images:", len(matches))
        print("Examples:", matches[:5])
        print("-" * 60)


In [ ]:
import pickle
import os

BACKUP = "/content/drive/MyDrive/egg_task"

with open(os.path.join(BACKUP, "sam_cache.pkl"), "rb") as f:
    cache = pickle.load(f)

print("Cache type:", type(cache))
print("Number of cached entries:", len(cache))

print("\nFirst 5 cache keys:")
for key in list(cache.keys())[:5]:
    print(repr(key))

In [ ]:
import os

search_roots = [
    "/content",
    "/content/drive/MyDrive"
]

for root in search_roots:
    print(f"\nSearching under: {root}")

    for current_root, dirs, files in os.walk(root):
        # Avoid searching generated outputs and unnecessary folders
        dirs[:] = [
            d for d in dirs
            if d not in [
                ".ipynb_checkpoints",
                ".git",
                "task1_masks",
                "task1_overlays",
                "sam_masks",
                "sam_overlays",
                "sample_data"
            ]
        ]

        matches = [
            f for f in files
            if f.lower().startswith("not_damaged_")
            and f.lower().endswith((".jpg", ".jpeg"))
        ]

        if matches:
            print("Folder:", current_root)
            print("Images:", len(matches))
            print("Examples:", matches[:3])

In [ ]:
import os

search_roots = [
    "/content/egg_task_recovered",
    "/content/drive/MyDrive/egg_task",
    "/content/drive/MyDrive",
]

for root in search_roots:
    print(f"\nSearching: {root}")

    if not os.path.isdir(root):
        print("Folder not found")
        continue

    found = 0

    for current_root, dirs, files in os.walk(root):
        # Skip generated outputs and hidden folders
        dirs[:] = [
            d for d in dirs
            if not d.startswith(".")
            and d not in [
                "task1_masks",
                "task1_overlays",
                "sam_masks",
                "sam_overlays",
                "sample_data",
            ]
        ]

        images = [
            f for f in files
            if f.lower().endswith((".jpg", ".jpeg"))
            and f.lower().startswith("not_damaged_")
        ]

        if images:
            found += len(images)
            print("Folder:", current_root)
            print("Matching images:", len(images))
            print("Examples:", images[:3])

    print("Total matching images:", found)

In [ ]:
import os

# Search the mounted Google Drive for image files.
# Exclude generated mask and overlay folders.
search_roots = [
    "/content/drive/MyDrive",
    "/content"
]

skip_folders = {
    ".ipynb_checkpoints",
    "sample_data",
    "task1_masks",
    "task1_overlays",
    "sam_masks",
    "sam_overlays"
}

found_images = []

for search_root in search_roots:
    for root, dirs, files in os.walk(search_root):
        dirs[:] = [
            d for d in dirs
            if d not in skip_folders
            and not d.startswith(".")
        ]

        for filename in files:
            if (
                filename.lower().endswith((".jpg", ".jpeg"))
                and filename.lower().startswith("not_damaged_")
            ):
                found_images.append(os.path.join(root, filename))

# Remove duplicate paths
found_images = sorted(set(found_images))

print("Original-style images found:", len(found_images))
print("\nFirst 10 results:")

for path in found_images[:10]:
    print(path)

In [ ]:
import os

# Search likely dataset locations for original egg images
search_roots = [
    "/content/drive/MyDrive",
    "/content/egg_task_recovered",
]

skip_folders = {
    "task1_masks",
    "task1_overlays",
    "sam_masks",
    "sam_overlays",
    ".ipynb_checkpoints",
    "sample_data",
}

for root in search_roots:
    print(f"\nSearching: {root}")

    if not os.path.exists(root):
        print("Folder not found")
        continue

    matches = []

    for current_root, dirs, files in os.walk(root):
        dirs[:] = [
            d for d in dirs
            if d not in skip_folders
            and not d.startswith(".")
        ]

        for filename in files:
            lower = filename.lower()

            if (
                lower.startswith("not_damaged_")
                and lower.endswith((".jpg", ".jpeg", ".png"))
            ):
                matches.append(os.path.join(current_root, filename))

    print("Original-style egg images found:", len(matches))

    for path in matches[:10]:
        print(path)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BACKUP = "/content/drive/MyDrive/egg_task"

# Upload egg.v1i.folder.zip from your laptop once; it is kept on Drive afterwards
from google.colab import files
files.upload()
!cp egg.v1i.folder.zip {BACKUP}/
!unzip -q -o egg.v1i.folder.zip -d data

In [ ]:
paths = sorted(glob.glob("data/**/*.jpg", recursive=True))   # same order as before, so indices still match
assert len(paths) == 110
cache = pickle.load(open(f"{BACKUP}/sam_cache.pkl", "rb"))
split = json.load(open(f"{BACKUP}/split.json"))
raw = pickle.load(open(f"{BACKUP}/overrides.pkl", "rb"))
overrides = {k: [(1.0, m) for m in v] for k, v in raw.items()}   # saved format was masks only; wrap back into tuples

def unpack(m):
    return np.unpackbits(m["seg"])[:640*640].reshape(640, 640).astype(bool)

def score_mask2(mask, hsv, min_ell=0.88, min_sol=0.95, min_val=0.50, max_sat=0.40):
    frac = mask.mean()
    if frac < 0.015 or frac > 0.60: return 0.0
    cnts, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    c = max(cnts, key=cv2.contourArea)
    if len(c) < 5: return 0.0
    sol = cv2.contourArea(c) / (cv2.contourArea(cv2.convexHull(c)) + 1e-6)
    em = np.zeros(mask.shape, np.uint8); cv2.ellipse(em, cv2.fitEllipse(c), 1, -1); em = em.astype(bool)
    ell_iou = (mask & em).sum() / ((mask | em).sum() + 1e-6)
    sat = hsv[..., 1][mask].mean() / 255; val = hsv[..., 2][mask].mean() / 255
    ys, xs = np.where(mask)
    touch = int(xs.min() <= 2) + int(xs.max() >= 637) + int(ys.min() <= 2) + int(ys.max() >= 637)
    if ell_iou < min_ell or sol < min_sol or val < min_val or sat > max_sat or touch >= 2: return 0.0
    return float(ell_iou * sol * val * (1 - sat) * (0.7 ** touch))

def pick_eggs2(p, rel_thresh=0.7, abs_thresh=0.30, max_overlap=0.2):
    img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    masks = [unpack(m) for m in cache[p]]
    cands = sorted(((score_mask2(m, hsv), m) for m in masks), key=lambda x: -x[0])
    chosen = []
    if cands and cands[0][0] > 0:
        best = cands[0][0]
        for s, m in cands:
            if s < abs_thresh or s < rel_thresh * best: break
            if all((m & c).sum() / min(m.sum(), c.sum()) < max_overlap for _, c in chosen):
                chosen.append((s, m))
    return img, chosen, cands

def final_masks(i):
    if i in overrides: return [m for _, m in overrides[i]]
    return [m for _, m in pick_eggs2(paths[i])[1]]

def sam_masks(i, boxes):
    img = cv2.cvtColor(cv2.imread(paths[i]), cv2.COLOR_BGR2RGB)
    predictor.set_image(img)
    return [predictor.predict(box=np.array(b), multimask_output=False)[0][0] for b in boxes]

def replace(i, boxes):
    overrides[i] = [(1.0, m) for m in sam_masks(i, boxes)]

def add(i, boxes):   # now idempotent: always starts from the picker output
    base = [m for _, m in pick_eggs2(paths[i])[1]]
    overrides[i] = [(1.0, m) for m in base + sam_masks(i, boxes)]

def fix_with_points(i, box, pos, neg):
    img = cv2.cvtColor(cv2.imread(paths[i]), cv2.COLOR_BGR2RGB)
    predictor.set_image(img)
    masks, scores, _ = predictor.predict(box=np.array(box), point_coords=np.array(pos + neg),
                                         point_labels=np.array([1]*len(pos) + [0]*len(neg)),
                                         multimask_output=True)
    overrides[i] = [(float(scores[np.argmax(scores)]), masks[np.argmax(scores)])]

def check(idx):
    cols = 4; rows = (len(idx) + cols - 1) // cols
    plt.figure(figsize=(20, 5*rows))
    for k, i in enumerate(idx):
        vis = cv2.cvtColor(cv2.imread(paths[i]), cv2.COLOR_BGR2RGB)
        ms = final_masks(i)
        for m in ms:
            cs, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(vis, cs, -1, (255, 0, 0), 3)
        plt.subplot(rows, cols, k+1); plt.imshow(vis); plt.title(f"#{i} eggs={len(ms)}"); plt.axis("off")
    plt.tight_layout(); plt.show()

def show_grid(i):
    img = cv2.cvtColor(cv2.imread(paths[i]), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(9, 9)); plt.imshow(img)
    plt.xticks(range(0, 641, 40)); plt.yticks(range(0, 641, 40))
    plt.grid(True, color="yellow", alpha=0.35); plt.title(f"#{i}"); plt.show()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BACKUP = "/content/drive/MyDrive/egg_task"
!cp {BACKUP}/egg.v1i.folder.zip .
!unzip -q -o egg.v1i.folder.zip -d data
!ls data

In [ ]:
!pip -q install git+https://github.com/facebookresearch/segment-anything.git imagehash
!wget -q -nc https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
import os, glob, json, pickle, shutil, collections, cv2, numpy as np, matplotlib.pyplot as plt, torch
from segment_anything import sam_model_registry, SamPredictor
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
sam = sam_model_registry["vit_h"](checkpoint="sam_vit_h_4b8939.pth").to(device)
predictor = SamPredictor(sam)

In [ ]:
import time
t = time.time()
replace(1, [[215,185,375,330]])
print("seconds per image:", round(time.time() - t))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BACKUP = "/content/drive/MyDrive/egg_task"
!cp {BACKUP}/egg.v1i.folder.zip .
!unzip -q -o egg.v1i.folder.zip -d data
!pip -q install git+https://github.com/facebookresearch/segment-anything.git imagehash
!wget -q -nc https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
import torch
from segment_anything import sam_model_registry, SamPredictor
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
sam = sam_model_registry["vit_h"](checkpoint="sam_vit_h_4b8939.pth").to(device)
predictor = SamPredictor(sam)

In [ ]:
%%writefile /content/drive/MyDrive/egg_task/helpers.py
import os, glob, json, pickle, shutil, time, collections, cv2, numpy as np, matplotlib.pyplot as plt

paths = sorted(glob.glob("data/**/*.jpg", recursive=True))
assert len(paths) == 110, len(paths)
cache = pickle.load(open(f"{BACKUP}/sam_cache.pkl", "rb"))
split = json.load(open(f"{BACKUP}/split.json"))
_raw = pickle.load(open(f"{BACKUP}/overrides.pkl", "rb"))
overrides = {k: [(1.0, m) for m in v] for k, v in _raw.items()}
print("overrides already saved for:", sorted(overrides))

def unpack(m):
    return np.unpackbits(m["seg"])[:640*640].reshape(640, 640).astype(bool)

def score_mask2(mask, hsv, min_ell=0.88, min_sol=0.95, min_val=0.50, max_sat=0.40):
    frac = mask.mean()
    if frac < 0.015 or frac > 0.60: return 0.0
    cnts, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    c = max(cnts, key=cv2.contourArea)
    if len(c) < 5: return 0.0
    sol = cv2.contourArea(c) / (cv2.contourArea(cv2.convexHull(c)) + 1e-6)
    em = np.zeros(mask.shape, np.uint8); cv2.ellipse(em, cv2.fitEllipse(c), 1, -1); em = em.astype(bool)
    ell_iou = (mask & em).sum() / ((mask | em).sum() + 1e-6)
    sat = hsv[..., 1][mask].mean() / 255; val = hsv[..., 2][mask].mean() / 255
    ys, xs = np.where(mask)
    touch = int(xs.min() <= 2) + int(xs.max() >= 637) + int(ys.min() <= 2) + int(ys.max() >= 637)
    if ell_iou < min_ell or sol < min_sol or val < min_val or sat > max_sat or touch >= 2: return 0.0
    return float(ell_iou * sol * val * (1 - sat) * (0.7 ** touch))

def pick_eggs2(p, rel_thresh=0.7, abs_thresh=0.30, max_overlap=0.2):
    img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    masks = [unpack(m) for m in cache[p]]
    cands = sorted(((score_mask2(m, hsv), m) for m in masks), key=lambda x: -x[0])
    chosen = []
    if cands and cands[0][0] > 0:
        best = cands[0][0]
        for s, m in cands:
            if s < abs_thresh or s < rel_thresh * best: break
            if all((m & c).sum() / min(m.sum(), c.sum()) < max_overlap for _, c in chosen):
                chosen.append((s, m))
    return img, chosen, cands

def final_masks(i):
    if i in overrides: return [m for _, m in overrides[i]]
    return [m for _, m in pick_eggs2(paths[i])[1]]

def sam_masks(i, boxes):
    img = cv2.cvtColor(cv2.imread(paths[i]), cv2.COLOR_BGR2RGB)
    predictor.set_image(img)
    return [predictor.predict(box=np.array(b), multimask_output=False)[0][0] for b in boxes]

def replace(i, boxes):
    overrides[i] = [(1.0, m) for m in sam_masks(i, boxes)]

def add(i, boxes):
    base = [m for _, m in pick_eggs2(paths[i])[1]]
    overrides[i] = [(1.0, m) for m in base + sam_masks(i, boxes)]

def fix_with_points(i, box, pos, neg):
    img = cv2.cvtColor(cv2.imread(paths[i]), cv2.COLOR_BGR2RGB)
    predictor.set_image(img)
    masks, scores, _ = predictor.predict(box=np.array(box), point_coords=np.array(pos + neg),
                                         point_labels=np.array([1]*len(pos) + [0]*len(neg)),
                                         multimask_output=True)
    b = int(np.argmax(scores)); overrides[i] = [(float(scores[b]), masks[b])]

def save_overrides():
    pickle.dump({k: [m for _, m in v] for k, v in overrides.items()}, open(f"{BACKUP}/overrides.pkl", "wb"))

def check(idx):
    cols = 4; rows = (len(idx) + cols - 1) // cols
    plt.figure(figsize=(20, 5*rows))
    for k, i in enumerate(idx):
        vis = cv2.cvtColor(cv2.imread(paths[i]), cv2.COLOR_BGR2RGB); ms = final_masks(i)
        for m in ms:
            cs, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(vis, cs, -1, (255, 0, 0), 3)
        plt.subplot(rows, cols, k+1); plt.imshow(vis); plt.title(f"#{i} eggs={len(ms)}"); plt.axis("off")
    plt.tight_layout(); plt.show()

def show_grid(i):
    img = cv2.cvtColor(cv2.imread(paths[i]), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(9, 9)); plt.imshow(img)
    plt.xticks(range(0, 641, 40)); plt.yticks(range(0, 641, 40))
    plt.grid(True, color="yellow", alpha=0.35); plt.title(f"#{i}"); plt.show()

def export_all():
    for d in ["task1_masks", "task1_instances", "task1_overlays"]:
        os.makedirs(f"outputs/{d}", exist_ok=True)
    for i, p in enumerate(paths):
        img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
        inst = np.zeros(img.shape[:2], np.uint8)
        for k, m in enumerate(final_masks(i), 1): inst[m & (inst == 0)] = k
        name = os.path.splitext(os.path.basename(p))[0]
        cv2.imwrite(f"outputs/task1_masks/{name}.png", (inst > 0).astype(np.uint8) * 255)
        cv2.imwrite(f"outputs/task1_instances/{name}.png", inst)
        ov = img.copy(); u = inst > 0
        ov[u] = (0.35 * ov[u] + 0.65 * np.array([255, 40, 40])).astype(np.uint8)
        cv2.imwrite(f"outputs/task1_overlays/{name}.jpg", cv2.cvtColor(ov, cv2.COLOR_RGB2BGR))
    shutil.make_archive("task1_outputs_v2", "zip", "outputs")
    shutil.copy("task1_outputs_v2.zip", BACKUP)
    print("exported and backed up")

In [ ]:
exec(open(f"{BACKUP}/helpers.py").read())

jobs = [("replace", 1,  [[215,185,375,330]]),
        ("replace", 12, [[250,280,430,430]]), ("replace", 19, [[140,275,415,470]]),
        ("replace", 40, [[295,150,462,260]]), ("replace", 80, [[60,60,560,400]]),
        ("replace", 92, [[110,205,415,515]]), ("replace", 93, [[180,140,400,315]]),
        ("replace", 25, [[400,390,590,550]]), ("replace", 38, [[195,228,498,462]]),
        ("replace", 64, [[145,190,450,410]]),
        ("add", 37, [[0,345,240,555]]), ("add", 42, [[155,80,365,280]]), ("add", 43, [[165,15,395,150]]),
        ("points", 34, None)]

for kind, i, boxes in jobs:
    if i in overrides:
        print(f"#{i} already saved, skipping"); continue
    t = time.time()
    if kind == "replace": replace(i, boxes)
    elif kind == "add":   add(i, boxes)
    else: fix_with_points(34, [250,260,435,455], pos=[[340,340],[340,385]], neg=[[280,330],[285,415],[400,430]])
    save_overrides()
    print(f"#{i} done in {round(time.time()-t)}s")

counts = [len(final_masks(i)) for i in range(110)]
print(collections.Counter(counts), "zero-egg:", [i for i, c in enumerate(counts) if c == 0])

In [ ]:
export_all()
check([1,12,19,25,34,37,38,40,42,43,64,79,80,92,93])

In [ ]:
name2path = {os.path.basename(p): p for p in paths}

def gt_mask(name):
    return cv2.imread(f"outputs/task1_masks/{os.path.splitext(name)[0]}.png", 0) > 0

def iou_dice(pred, gt):
    inter = (pred & gt).sum(); union = (pred | gt).sum(); s = pred.sum() + gt.sum()
    return (inter / union if union else 1.0), (2 * inter / s if s else 1.0)

def evaluate(algo, names, **kw):
    r = np.array([iou_dice(algo(cv2.imread(name2path[n]), **kw), gt_mask(n)) for n in names])
    return {"mIoU": round(r[:, 0].mean(), 4), "medIoU": round(float(np.median(r[:, 0])), 4),
            "mDice": round(r[:, 1].mean(), 4), "IoU<0.5": int((r[:, 0] < 0.5).sum()), "n": len(r)}, r

def show_worst(algo, names, scores, k=8, **kw):
    order = np.argsort(scores[:, 0])[:k]
    plt.figure(figsize=(20, 5 * ((k + 3) // 4)))
    for j, idx in enumerate(order):
        img = cv2.cvtColor(cv2.imread(name2path[names[idx]]), cv2.COLOR_BGR2RGB)
        pred = algo(cv2.imread(name2path[names[idx]]), **kw); gt = gt_mask(names[idx])
        vis = img.copy()
        vis[pred & gt] = (0.5 * vis[pred & gt] + 0.5 * np.array([0, 255, 0])).astype(np.uint8)    # green = correct
        vis[pred & ~gt] = (0.5 * vis[pred & ~gt] + 0.5 * np.array([255, 0, 0])).astype(np.uint8)  # red = false positive
        vis[~pred & gt] = (0.5 * vis[~pred & gt] + 0.5 * np.array([0, 0, 255])).astype(np.uint8)  # blue = missed
        plt.subplot((k + 3) // 4, 4, j + 1); plt.imshow(vis)
        plt.title(f"{names[idx][:20]}  IoU={scores[idx,0]:.2f}"); plt.axis("off")
    plt.tight_layout(); plt.show()

dev = split["dev"]          # tune ONLY on dev; do not touch split["eval"] until the very end
log = []

def algo_v1(img, blur=5):
    g = cv2.GaussianBlur(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), (blur, blur), 0)
    _, m = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return m > 0

stats, scores = evaluate(algo_v1, dev, blur=5)
print("Trial 1:", stats)
log.append({"trial": 1, "change": "grayscale + Gaussian blur + Otsu", "why": "simplest baseline",
            "params": "blur=5", **stats, "decision": "TODO after looking at failures"})
show_worst(algo_v1, dev, scores, k=8, blur=5)

In [ ]:
log[0]["decision"] = ("Discarded. Global brightness threshold also selects white plates/walls/paper (false positives) "
                      "and misses eggs held against dark scenes; 64/88 dev images have IoU<0.5.")

def run_trial(n, algo, change, why, **kw):
    stats, sc = evaluate(algo, dev, **kw)
    log.append({"trial": n, "change": change, "why": why,
                "params": ", ".join(f"{k}={v}" for k, v in kw.items()), **stats})
    print(f"Trial {n}:", stats)
    return sc

def clean(m, k=7):
    ker = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    m = cv2.morphologyEx(m.astype(np.uint8), cv2.MORPH_OPEN, ker)
    return cv2.morphologyEx(m, cv2.MORPH_CLOSE, ker)

def largest_cc(m):
    n, lab, st, _ = cv2.connectedComponentsWithStats(m.astype(np.uint8))
    return (lab == 1 + np.argmax(st[1:, cv2.CC_STAT_AREA])) if n > 1 else (m > 0)

# Trial 2: colour mask (low saturation AND bright) + morphology + largest component
def algo_v2(img, s_max=70, v_min=120, k=7):
    hsv = cv2.cvtColor(cv2.GaussianBlur(img, (5, 5), 0), cv2.COLOR_BGR2HSV)
    return largest_cc(clean((hsv[..., 1] < s_max) & (hsv[..., 2] > v_min), k))

sc2 = run_trial(2, algo_v2, "HSV mask S<70 & V>120, open/close, largest component",
                "egg shell is bright and unsaturated; skin and most backgrounds are not", s_max=70, v_min=120, k=7)

# Trial 3: many candidate masks, then pick by SHAPE (convex, ellipse-like)
def shape_score(mask):
    cnts, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    c = max(cnts, key=cv2.contourArea)
    if len(c) < 5: return 0.0
    sol = cv2.contourArea(c) / (cv2.contourArea(cv2.convexHull(c)) + 1e-6)
    em = np.zeros(mask.shape, np.uint8); cv2.ellipse(em, cv2.fitEllipse(c), 1, -1); em = em.astype(bool)
    return float(sol * (mask & em).sum() / ((mask | em).sum() + 1e-6))

def components(m, min_frac=0.008, max_frac=0.6):
    n, lab, st, _ = cv2.connectedComponentsWithStats(m.astype(np.uint8)); H, W = m.shape
    for j in range(1, n):
        x, y, w, h, a = [int(v) for v in st[j]]
        if a < min_frac * H * W or a > max_frac * H * W: continue
        if int(x <= 2) + int(y <= 2) + int(x + w >= W - 2) + int(y + h >= H - 2) >= 2: continue
        yield lab == j

def algo_v3(img, s_list=(40, 70, 100), v_list=(100, 140, 180, 220), k=7, rel=0.9, abs_t=0.75):
    hsv = cv2.cvtColor(cv2.GaussianBlur(img, (5, 5), 0), cv2.COLOR_BGR2HSV)
    cands = [(shape_score(c), c) for s in s_list for v in v_list
             for c in components(clean((hsv[..., 1] < s) & (hsv[..., 2] > v), k))]
    out = np.zeros(img.shape[:2], bool)
    if not cands: return out
    cands.sort(key=lambda x: -x[0]); best = cands[0][0]; chosen = []
    for s, c in cands:
        if s < abs_t or s < rel * best: break
        if all((c & o).sum() / min(c.sum(), o.sum()) < 0.2 for o in chosen):
            chosen.append(c); out |= c
    return out if chosen else cands[0][1]

sc3 = run_trial(3, algo_v3, "multi-threshold candidates, choose components by ellipse fit x solidity",
                "fixes bright backgrounds: only convex egg-shaped blobs survive; handles multiple eggs",
                s_list=(40, 70, 100), v_list=(100, 140, 180, 220), k=7, rel=0.9, abs_t=0.75)
show_worst(algo_v3, dev, sc3, k=8)

In [ ]:
import pandas as pd
pd.DataFrame(log).to_csv(f"{BACKUP}/experiment_log.csv", index=False)

In [ ]:
log[2]["decision"] = ("Partial/discarded. Colour thresholds merge the egg with pale backgrounds (white-on-white, "
                      "bright light), so the egg component fails the shape test and a wrong small blob is chosen "
                      "(IoU=0 on the worst 8 dev images).")
# log[1]["decision"] = "..."   # fill in after you see the trial 2 numbers

def enclosed_regions(edges):
    """Pixels fully surrounded by edges (not connected to the image border)."""
    n, lab = cv2.connectedComponents((edges == 0).astype(np.uint8), connectivity=4)
    border = set(np.unique(np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]])).tolist())
    keep = [l for l in range(1, n) if l not in border]
    return np.isin(lab, keep)

def algo_v4(img, canny=((20, 60), (35, 100), (50, 140)), close_ks=(5, 9, 15), blur=7,
            abs_t=0.80, rel=0.85, area_pow=0.25, hull=True):
    H, W = img.shape[:2]
    g = cv2.GaussianBlur(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), (blur, blur), 0)
    cands = []
    for lo, hi in canny:
        e0 = cv2.Canny(g, lo, hi)
        for k in close_ks:
            e = cv2.morphologyEx(e0, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k)))
            reg = enclosed_regions(e)
            reg = cv2.dilate(reg.astype(np.uint8), np.ones((3, 3), np.uint8), iterations=2) > 0  # take back the edge pixels
            for c in components(reg):
                s = shape_score(c)
                if s >= abs_t:
                    cands.append((s * (c.sum() / (H * W)) ** area_pow, c))
    out = np.zeros((H, W), bool)
    if not cands: return out
    cands.sort(key=lambda x: -x[0]); best = cands[0][0]; chosen = []
    for s, c in cands:
        if s < rel * best: break
        if all((c & o).sum() / min(c.sum(), o.sum()) < 0.2 for o in chosen):
            chosen.append(c)
    for c in chosen:
        if hull:
            cs, _ = cv2.findContours(c.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            tmp = np.zeros((H, W), np.uint8); cv2.fillConvexPoly(tmp, cv2.convexHull(max(cs, key=cv2.contourArea)), 1)
            out |= tmp > 0
        else:
            out |= c
    return out

sc4 = run_trial(4, algo_v4, "Canny edges + closing -> enclosed regions -> pick egg-shaped by ellipse fit x solidity x area prior",
                "eggs on pale backgrounds are separated by edge/shadow, not colour",
                canny="3 pairs", close_ks=(5, 9, 15), abs_t=0.80, rel=0.85)
pd.DataFrame(log).to_csv(f"{BACKUP}/experiment_log.csv", index=False)
show_worst(algo_v4, dev, sc4, k=8)

In [ ]:
sc4 = run_trial(4, algo_v4,
                "Canny edges (3 threshold pairs) + closing -> enclosed regions -> pick egg-shaped by ellipse fit x solidity x area prior",
                "eggs on pale backgrounds are separated by edge/shadow, not colour",
                close_ks=(5, 9, 15), abs_t=0.80, rel=0.85)
pd.DataFrame(log).to_csv(f"{BACKUP}/experiment_log.csv", index=False)
show_worst(algo_v4, dev, sc4, k=8)

In [ ]:
pd.set_option("display.width", 200, "display.max_colwidth", 60)
print(pd.DataFrame(log)[["trial", "change", "mIoU", "medIoU", "mDice", "IoU<0.5"]])

In [ ]:
# ---- log decisions so far ----
log[1]["decision"] = "Discarded. Marginal gain over Otsu; pale/bright backgrounds still pass the S<70 & V>120 test."
log[2]["decision"] = ("KEPT as best so far (mIoU 0.495). Shape-based selection of candidate blobs removed most bright-"
                      "background false positives; still fails when the egg merges with a pale background.")
log[3]["decision"] = ("Discarded (mIoU 0.178, median 0). Returns an empty mask for most images: shadow side of pale eggs "
                      "leaves gaps in the edge ring, so the 'enclosed' region leaks to the border and is rejected.")

# ---- GrabCut building blocks ----
def grabcut_init_center(H, W, r_prob=0.65, r_core=0.12, border=0.03):
    yy, xx = np.mgrid[0:H, 0:W]
    r = np.sqrt(((xx - W/2)/(W/2))**2 + ((yy - H/2)/(H/2))**2)
    gc = np.full((H, W), cv2.GC_PR_BGD, np.uint8)
    gc[r < r_prob] = cv2.GC_PR_FGD
    gc[r < r_core] = cv2.GC_FGD
    b = max(2, int(border * min(H, W)))
    gc[:b] = cv2.GC_BGD; gc[-b:] = cv2.GC_BGD; gc[:, :b] = cv2.GC_BGD; gc[:, -b:] = cv2.GC_BGD
    return gc

def seed_to_gc(seed, grow=25, core=15):
    k = lambda r: cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2*r+1, 2*r+1))
    s = seed.astype(np.uint8)
    gc = np.full(s.shape, cv2.GC_BGD, np.uint8)
    gc[cv2.dilate(s, k(grow)) > 0] = cv2.GC_PR_BGD
    gc[s > 0] = cv2.GC_PR_FGD
    gc[cv2.erode(s, k(core)) > 0] = cv2.GC_FGD
    return gc

def run_grabcut(img, gc, iters=5):
    bgd = np.zeros((1, 65), np.float64); fgd = np.zeros((1, 65), np.float64)
    cv2.grabCut(img, gc, None, bgd, fgd, iters, cv2.GC_INIT_WITH_MASK)
    return (gc == cv2.GC_FGD) | (gc == cv2.GC_PR_FGD)

def postprocess(m, k=9, keep_rel=0.3):
    m = clean(m, k)
    n, lab, st, _ = cv2.connectedComponentsWithStats(m.astype(np.uint8))
    if n <= 1: return np.zeros(m.shape, bool)
    areas = st[1:, cv2.CC_STAT_AREA]
    keep = [j + 1 for j, a in enumerate(areas) if a >= keep_rel * areas.max()]
    out = np.isin(lab, keep)
    cnts, _ = cv2.findContours(out.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    filled = np.zeros(out.shape, np.uint8); cv2.drawContours(filled, cnts, -1, 1, -1)
    return filled > 0

# ---- Trial 5: GrabCut, centre prior ----
def algo_v5(img, iters=5, r_prob=0.65, r_core=0.12, k=9):
    H, W = img.shape[:2]
    return postprocess(run_grabcut(img, grabcut_init_center(H, W, r_prob, r_core), iters), k)

sc5 = run_trial(5, algo_v5, "GrabCut initialised with a centre prior (core=FG, ellipse=probable FG, border=BG), "
                "hole-fill, keep components >=30% of the largest",
                "eggs are large and roughly central; GrabCut uses edges+colour models, not a global threshold",
                iters=5, r_prob=0.65, r_core=0.12, k=9)

# ---- Trial 6: hybrid ----
def algo_v6(img, shape_min=0.90, grow=25, core=15, iters=5, k=9):
    H, W = img.shape[:2]
    seed = algo_v3(img)
    ok = False
    if seed.sum() > 0.01 * H * W:
        ys, xs = np.nonzero(seed)
        cx, cy = xs.mean() / W, ys.mean() / H
        ok = shape_score(seed) >= shape_min and 0.2 < cx < 0.8 and 0.2 < cy < 0.8
    gc = seed_to_gc(seed, grow, core) if ok else grabcut_init_center(H, W)
    return postprocess(run_grabcut(img, gc, iters), k)

sc6 = run_trial(6, algo_v6, "Hybrid: trial-3 candidate seeds GrabCut if egg-shaped (>=0.90) and central, else centre prior",
                "trial 3 is precise when it works; GrabCut refines its boundary and rescues its failures",
                shape_min=0.90, grow=25, core=15, iters=5, k=9)

pd.DataFrame(log).to_csv(f"{BACKUP}/experiment_log.csv", index=False)
print(pd.DataFrame(log)[["trial", "mIoU", "medIoU", "mDice", "IoU<0.5"]])
show_worst(algo_v6, dev, sc6, k=8)

In [ ]:
import time
def run_grabcut(img, gc, iters=5, scale=0.5):
    H, W = img.shape[:2]
    small = cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    gcs = cv2.resize(gc, (small.shape[1], small.shape[0]), interpolation=cv2.INTER_NEAREST)
    bgd = np.zeros((1, 65), np.float64); fgd = np.zeros((1, 65), np.float64)
    cv2.grabCut(small, gcs, None, bgd, fgd, iters, cv2.GC_INIT_WITH_MASK)
    m = ((gcs == cv2.GC_FGD) | (gcs == cv2.GC_PR_FGD)).astype(np.uint8)
    return cv2.resize(m, (W, H), interpolation=cv2.INTER_NEAREST) > 0

img0 = cv2.imread(name2path[dev[0]]); H, W = img0.shape[:2]
for s in (0.5, 0.35, 0.25):
    t = time.time(); run_grabcut(img0, grabcut_init_center(H, W), iters=5, scale=s)
    print(f"scale {s}: {time.time()-t:.1f}s per image  -> ~{(time.time()-t)*88/60:.1f} min for 88 dev images")

In [ ]:
SCALE = 0.35        # <- set from Step 2

_rg = run_grabcut
def run_grabcut(img, gc, iters=5, scale=None):
    return _rg(img, gc, iters, SCALE if scale is None else scale)

sc5 = run_trial(5, algo_v5, f"GrabCut (scale {SCALE}) initialised with a centre prior (core=FG, ellipse=probable FG, border=BG), "
                "hole-fill, keep components >=30% of the largest",
                "eggs are large and roughly central; GrabCut uses edges+colour models, not a global threshold",
                iters=5, r_prob=0.65, r_core=0.12, k=9)
pd.DataFrame(log).to_csv(f"{BACKUP}/experiment_log.csv", index=False)
print(pd.DataFrame(log)[["trial", "mIoU", "medIoU", "mDice", "IoU<0.5"]])
show_worst(algo_v5, dev, sc5, k=8)

In [ ]:
log = [r for j, r in enumerate(log) if j not in (4, 5)]      # drop the first, duplicate pair
log[4]["decision"] = ("Partial. Same mIoU as trial 3 (0.496) but different failures: finds eggs that colour "
                      "thresholds miss, but swallows the whole container/tray when the egg sits in or on one.")
log[5]["decision"] = ("KEPT as current best (mIoU 0.572, median 0.632). Trial 3 gives a trustworthy egg shape, "
                      "GrabCut refines the boundary and rescues trial 3's failures.")
print(pd.DataFrame(log)[["trial", "mIoU", "medIoU", "IoU<0.5"]])

In [ ]:
def algo_v7(img, radii=((0.65, 0.12), (0.45, 0.10), (0.30, 0.08)),
            shape_ok=0.90, max_area=0.45, iters=5, k=9):
    H, W = img.shape[:2]
    def quality(m):
        if m.sum() == 0: return 0.0
        a = m.sum() / (H * W)
        return shape_score(m) * (1.0 if a <= max_area else max_area / a)
    best = algo_v6(img, iters=iters, k=k); best_q = quality(best)
    if best_q >= shape_ok: return best
    for r_prob, r_core in radii[1:]:
        m = postprocess(run_grabcut(img, grabcut_init_center(H, W, r_prob, r_core), iters), k)
        q = quality(m)
        if q > best_q: best, best_q = m, q
        if q >= shape_ok: break
    return best

sc7 = run_trial(7, algo_v7, "Hybrid + validation: if result is not egg-shaped or covers >45% of the image, retry GrabCut "
                "with a tighter centre prior (r=0.45, then 0.30) and keep the most egg-shaped result",
                "container/bowl failures: the centre prior was so large that GrabCut absorbed the whole container",
                shape_ok=0.90, max_area=0.45, iters=5, k=9)
pd.DataFrame(log).to_csv(f"{BACKUP}/experiment_log.csv", index=False)
print(pd.DataFrame(log)[["trial", "mIoU", "medIoU", "mDice", "IoU<0.5"]])
show_worst(algo_v7, dev, sc7, k=8)

In [ ]:
print("Number of trials logged:", len(log))

print(pd.DataFrame(log)[
    ["trial", "change", "mIoU", "medIoU", "mDice", "IoU<0.5"]
])

In [ ]:
pd.DataFrame(log).to_csv(
    f"{BACKUP}/experiment_log.csv",
    index=False
)

print("Experiment log saved.")

In [ ]:
import os
import json
import pandas as pd

os.makedirs(BACKUP, exist_ok=True)

# Save the complete experiment history
log_path = f"{BACKUP}/experiment_log.csv"
pd.DataFrame(log).to_csv(log_path, index=False)

# Record the selected algorithm and development results
final_result = {
    "selected_trial": 7,
    "algorithm": "Hybrid segmentation with shape and area validation",
    "development_images": 88,
    "mIoU": 0.6405,
    "median_IoU": 0.8057,
    "mean_Dice": 0.7022,
    "images_with_IoU_below_0.5": 26,
    "status": "Frozen; no further development-set tuning"
}

with open(f"{BACKUP}/final_algorithm_result.json", "w") as f:
    json.dump(final_result, f, indent=2)

print("Final algorithm results saved.")
print("Experiment log:", log_path)
print(pd.DataFrame(log)[
    ["trial", "mIoU", "medIoU", "mDice", "IoU<0.5"]
])

In [ ]:
print("Split keys:", split.keys())
print("Development images:", len(split["dev"]))
print("Evaluation images:", len(split["eval"]))

print("\nFirst 3 evaluation entries:")
for i, item in enumerate(split["eval"][:3]):
    print(f"{i}: {repr(item)}")

print("\nDevelopment sample type:", type(split["dev"][0]))
print("Development sample:", repr(split["dev"][0]))

In [ ]:
import os

print("Current working directory:", os.getcwd())
print("BACKUP:", BACKUP)

# Inspect common dataset locations
for path in [
    "/content",
    "/content/drive/MyDrive",
    BACKUP
]:
    print(f"\n--- {path} ---")
    if os.path.exists(path):
        print(os.listdir(path)[:25])
    else:
        print("Path does not exist")

In [ ]:
import os

target = split["eval"][0]
print("Target filename:", target)

matches = []

for root, dirs, files in os.walk("/content"):
    # Skip large or irrelevant directories
    dirs[:] = [
        d for d in dirs
        if d not in {"node_modules", ".git", "__pycache__"}
    ]

    if target in files:
        matches.append(os.path.join(root, target))

print("\nMatches:")
for path in matches:
    print(path)

In [ ]:
import os

print("Dataset directory:")
for root, dirs, files in os.walk("/content/data"):
    level = root.replace("/content/data", "").count(os.sep)
    if level > 3:
        dirs[:] = []
        continue

    print("  " * level + os.path.basename(root) + "/")
    for f in files[:5]:
        print("  " * (level + 1) + f)

print("\nTask 1 output ZIP exists:",
      os.path.exists("/content/drive/MyDrive/egg_task/task1_outputs_v2.zip"))

print("\nTask 1 output directory:")
for path in [
    "/content/outputs",
    "/content/outputs/task1_overlays",
    "/content/drive/MyDrive/egg_task"
]:
    print(f"\n{path}:")
    if os.path.exists(path):
        print(os.listdir(path)[:20])
    else:
        print("Not found")

In [ ]:
import os
import pickle

pkl_path = "/content/drive/MyDrive/egg_task/overrides.pkl"

print("Overrides file exists:", os.path.exists(pkl_path))

if os.path.exists(pkl_path):
    with open(pkl_path, "rb") as f:
        overrides = pickle.load(f)

    print("Overrides type:", type(overrides))
    print("Number of entries:", len(overrides))

    if isinstance(overrides, dict):
        print("Sample keys:", list(overrides.keys())[:5])
        first_key = next(iter(overrides), None)
        if first_key is not None:
            print("First entry type:", type(overrides[first_key]))
            print("First entry:", repr(overrides[first_key])[:500])

In [ ]:
import os
import cv2

MASK_DIR = "/content/outputs/task1_masks"
INSTANCE_DIR = "/content/outputs/task1_instances"

for folder in [MASK_DIR, INSTANCE_DIR]:
    print(f"\n--- {folder} ---")

    if os.path.isdir(folder):
        files = os.listdir(folder)
        print("Number of files:", len(files))

        for name in files[:10]:
            path = os.path.join(folder, name)
            print(name, "|", os.path.getsize(path), "bytes")
    else:
        print("Directory not found")

In [ ]:
import os

mask_files = set(os.listdir(MASK_DIR))
instance_files = set(os.listdir(INSTANCE_DIR))

print("Evaluation images:", len(split["eval"]))
print("Evaluation images with matching mask filenames:",
      sum(name in mask_files for name in split["eval"]))
print("Evaluation images with matching instance filenames:",
      sum(name in instance_files for name in split["eval"]))

print("\nFirst 5 evaluation images and their mask availability:")
for name in split["eval"][:5]:
    print(
        name,
        "| mask:", name in mask_files,
        "| instances:", name in instance_files
    )

In [ ]:
import os

MASK_DIR = "/content/outputs/task1_masks"
INSTANCE_DIR = "/content/outputs/task1_instances"

print("Evaluation images:", len(split["eval"]))

mask_matches = []
missing_masks = []

for image_name in split["eval"]:
    stem = os.path.splitext(image_name)[0]
    mask_path = os.path.join(MASK_DIR, stem + ".png")

    if os.path.exists(mask_path):
        mask_matches.append((image_name, mask_path))
    else:
        missing_masks.append(image_name)

print("Matching PNG masks:", len(mask_matches))
print("Missing masks:", len(missing_masks))

print("\nFirst 5 matches:")
for image_name, mask_path in mask_matches[:5]:
    print(image_name, "->", mask_path)

print("\nFirst 5 missing:")
for name in missing_masks[:5]:
    print(name)

In [ ]:
import os
from collections import Counter

mask_stems = {
    os.path.splitext(name)[0]
    for name in os.listdir(MASK_DIR)
    if name.lower().endswith(".png")
}

dev_stems = {os.path.splitext(name)[0] for name in split["dev"]}
eval_stems = {os.path.splitext(name)[0] for name in split["eval"]}

print("Development images with masks:", len(dev_stems & mask_stems))
print("Evaluation images with masks:", len(eval_stems & mask_stems))
print("Masks overlapping development:", len(mask_stems & dev_stems))
print("Masks overlapping evaluation:", len(mask_stems & eval_stems))

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from IPython.display import display

DATA_ROOT = "/content/data"
MASK_DIR = "/content/outputs/task1_masks"
RESULT_DIR = f"{BACKUP}/final_evaluation"

os.makedirs(RESULT_DIR, exist_ok=True)

def find_image(filename):
    for folder in ["train", "valid", "test"]:
        for subfolder in ["egg", "Unlabeled"]:
            path = os.path.join(DATA_ROOT, folder, subfolder, filename)
            if os.path.isfile(path):
                return path
    return None

def calculate_metrics(pred, gt):
    pred = np.asarray(pred).astype(bool)
    gt = np.asarray(gt).astype(bool)

    intersection = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()

    iou = intersection / union if union else 1.0
    denom = pred.sum() + gt.sum()
    dice = (2 * intersection / denom) if denom else 1.0

    return float(iou), float(dice)

results = []

for idx, filename in enumerate(split["eval"], start=1):
    image_path = find_image(filename)
    mask_path = os.path.join(
        MASK_DIR, os.path.splitext(filename)[0] + ".png"
    )

    if image_path is None:
        raise FileNotFoundError(f"Image not found in train/valid/test: {filename}")

    if not os.path.isfile(mask_path):
        raise FileNotFoundError(f"Reference mask missing: {mask_path}")

    image = cv2.imread(image_path)
    gt = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    if image is None or gt is None:
        raise ValueError(f"Could not read image or mask: {filename}")

    # Frozen Trial 7 — do not tune on the evaluation split
    pred = np.asarray(algo_v7(image)).astype(bool)

    if pred.shape != gt.shape:
        pred = cv2.resize(
            pred.astype(np.uint8),
            (gt.shape[1], gt.shape[0]),
            interpolation=cv2.INTER_NEAREST
        ).astype(bool)

    gt = gt > 0
    iou, dice = calculate_metrics(pred, gt)

    results.append({
        "filename": filename,
        "source_path": image_path,
        "IoU": iou,
        "Dice": dice,
        "predicted_pixels": int(pred.sum()),
        "ground_truth_pixels": int(gt.sum())
    })

    print(
        f"[{idx:02d}/{len(split['eval'])}] "
        f"IoU={iou:.4f}, Dice={dice:.4f} | {filename}"
    )

df_eval = pd.DataFrame(results)
csv_path = os.path.join(RESULT_DIR, "evaluation_metrics.csv")
df_eval.to_csv(csv_path, index=False)

print("\n===== FINAL EVALUATION SUMMARY =====")
print("Evaluation images:", len(df_eval))
print("Mean IoU:", round(df_eval["IoU"].mean(), 4))
print("Median IoU:", round(df_eval["IoU"].median(), 4))
print("Mean Dice:", round(df_eval["Dice"].mean(), 4))
print("Images with IoU < 0.5:", int((df_eval["IoU"] < 0.5).sum()))
print("Results saved to:", csv_path)

display(df_eval.sort_values("IoU").head(5))

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Use the evaluation results already calculated
worst = df_eval.nsmallest(5, "IoU")

for _, row in worst.iterrows():
    filename = row["filename"]
    image_path = row["source_path"]

    mask_name = os.path.splitext(filename)[0] + ".png"
    gt_path = os.path.join(MASK_DIR, mask_name)

    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    gt = cv2.imread(gt_path, cv2.IMREAD_GRAYSCALE) > 0

    # Re-run the frozen algorithm for visualization only
    pred = np.asarray(algo_v7(cv2.imread(image_path))).astype(bool)

    if pred.shape != gt.shape:
        pred = cv2.resize(
            pred.astype(np.uint8),
            (gt.shape[1], gt.shape[0]),
            interpolation=cv2.INTER_NEAREST
        ).astype(bool)

    # Overlay: green = reference, red = prediction
    overlay = image.copy()
    overlay[gt] = (
        0.5 * overlay[gt] + 0.5 * np.array([0, 255, 0])
    ).astype(np.uint8)
    overlay[pred] = (
        0.5 * overlay[pred] + 0.5 * np.array([255, 0, 0])
    ).astype(np.uint8)

    fig, axes = plt.subplots(1, 4, figsize=(18, 5))
    axes[0].imshow(image)
    axes[0].set_title("Original image")
    axes[1].imshow(gt, cmap="gray")
    axes[1].set_title("Reference mask")
    axes[2].imshow(pred, cmap="gray")
    axes[2].set_title("Trial 7 prediction")
    axes[3].imshow(overlay)
    axes[3].set_title("Overlay: green=GT, red=prediction")

    for ax in axes:
        ax.axis("off")

    fig.suptitle(f"IoU={row['IoU']:.4f} | Dice={row['Dice']:.4f}")
    plt.tight_layout()
    plt.show()

In [ ]:
import os
import json
import pandas as pd

RESULT_DIR = "/content/drive/MyDrive/egg_task/final_evaluation"
os.makedirs(RESULT_DIR, exist_ok=True)

# Save per-image metrics
df_eval.to_csv(
    os.path.join(RESULT_DIR, "evaluation_metrics.csv"),
    index=False
)

# Save aggregate summary
summary = {
    "evaluation_images": int(len(df_eval)),
    "mean_IoU": float(df_eval["IoU"].mean()),
    "median_IoU": float(df_eval["IoU"].median()),
    "mean_Dice": float(df_eval["Dice"].mean()),
    "images_with_IoU_below_0.5": int((df_eval["IoU"] < 0.5).sum()),
    "images_with_IoU_at_least_0.5": int((df_eval["IoU"] >= 0.5).sum()),
    "development_best_trial": 7,
    "development_best_mIoU": 0.6405
}

summary_path = os.path.join(RESULT_DIR, "evaluation_summary.json")

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("\nSaved:", summary_path)

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

VIS_DIR = os.path.join(RESULT_DIR, "worst_cases")
os.makedirs(VIS_DIR, exist_ok=True)

# Identify the five worst evaluation images
worst = df_eval.nsmallest(5, "IoU")

for _, row in worst.iterrows():
    filename = row["filename"]

    image_path = os.path.join(IMAGE_DIR, filename)
    mask_name = os.path.splitext(filename)[0] + ".png"
    mask_path = os.path.join(MASK_DIR, mask_name)

    image = cv2.imread(image_path)
    gt = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    if image is None or gt is None:
        print("Could not load:", filename)
        continue

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Use the frozen algorithm without changing its parameters
    pred = np.asarray(algo_v7(cv2.imread(image_path))).astype(bool)

    if pred.shape != gt.shape:
        pred = cv2.resize(
            pred.astype(np.uint8),
            (gt.shape[1], gt.shape[0]),
            interpolation=cv2.INTER_NEAREST
        ).astype(bool)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    axes[0].imshow(image)
    axes[0].set_title("Original")

    axes[1].imshow(gt > 0, cmap="gray")
    axes[1].set_title("Reference mask")

    axes[2].imshow(pred, cmap="gray")
    axes[2].set_title(
        f"Trial 7 prediction\nIoU={row['IoU']:.3f}"
    )

    for ax in axes:
        ax.axis("off")

    fig.suptitle(filename)
    plt.tight_layout()

    safe_name = filename.rsplit(".", 1)[0] + "_comparison.png"
    fig.savefig(
        os.path.join(VIS_DIR, safe_name),
        dpi=150,
        bbox_inches="tight"
    )
    plt.close(fig)

print("Worst-case visualizations saved to:", VIS_DIR)

In [ ]:
import json
import pandas as pd

base = "/content/drive/MyDrive/egg_task"

with open(os.path.join(base, "final_algorithm_result.json")) as f:
    print("Final algorithm record:")
    print(json.dumps(json.load(f), indent=2))

log_df = pd.read_csv(os.path.join(base, "experiment_log.csv"))

print("\nExperiment log:")
print(log_df[["trial", "mIoU", "medIoU", "mDice", "IoU<0.5"]].to_string(index=False))

In [ ]:
import os

DATA_ROOT = "/content/data"

def find_image_path(filename):
    for split_name in ["train", "valid", "test"]:
        for category in ["egg", "Unlabeled"]:
            candidate = os.path.join(
                DATA_ROOT, split_name, category, filename
            )
            if os.path.isfile(candidate):
                return candidate
    return None

In [ ]:
image_path = os.path.join(IMAGE_DIR, filename)

In [ ]:
import os
import cv2
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# 1. PATHS
# ============================================================

DATA_ROOT = "/content/data"
MASK_DIR = "/content/outputs/task1_masks"

RESULT_DIR = "/content/drive/MyDrive/egg_task/final_evaluation"
VIS_DIR = os.path.join(RESULT_DIR, "worst_cases")

os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(VIS_DIR, exist_ok=True)

print("Dataset directory:", DATA_ROOT)
print("Mask directory:", MASK_DIR)
print("Visualization directory:", VIS_DIR)

# ============================================================
# 2. FIND AN IMAGE ACROSS ALL DATASET SPLITS
# ============================================================

def find_image_path(filename):
    """
    Search for an image in train, valid, and test,
    including egg and Unlabeled directories.
    """
    for split_name in ["train", "valid", "test"]:
        for category in ["egg", "Unlabeled"]:
            candidate = os.path.join(
                DATA_ROOT,
                split_name,
                category,
                filename
            )

            if os.path.isfile(candidate):
                return candidate

    return None


# ============================================================
# 3. VERIFY REQUIRED VARIABLES AND FILES
# ============================================================

if "df_eval" not in globals():
    raise NameError(
        "df_eval is missing. Run the evaluation cell first."
    )

if "algo_v7" not in globals():
    raise NameError(
        "algo_v7 is missing. Run the cell defining your frozen "
        "Trial 7 algorithm first."
    )

required_columns = {"filename", "IoU", "Dice"}
missing_columns = required_columns - set(df_eval.columns)

if missing_columns:
    raise ValueError(
        f"df_eval is missing columns: {missing_columns}"
    )

if not os.path.isdir(DATA_ROOT):
    raise FileNotFoundError(
        f"Dataset directory not found: {DATA_ROOT}"
    )

if not os.path.isdir(MASK_DIR):
    raise FileNotFoundError(
        f"Mask directory not found: {MASK_DIR}"
    )

print("\nEvaluation records:", len(df_eval))

# ============================================================
# 4. SELECT THE FIVE WORST EVALUATION IMAGES
# ============================================================

worst = df_eval.nsmallest(5, "IoU").copy()

print("\nFive worst evaluation cases:")
print(
    worst[["filename", "IoU", "Dice"]].to_string(index=False)
)

# ============================================================
# 5. GENERATE COMPARISON IMAGES
# ============================================================

saved_count = 0
failed_files = []

for _, row in worst.iterrows():

    filename = row["filename"]

    # Find the original image across all dataset splits
    image_path = find_image_path(filename)

    if image_path is None:
        print("IMAGE NOT FOUND:", filename)
        failed_files.append(filename)
        continue

    # Corresponding reference mask
    mask_name = os.path.splitext(filename)[0] + ".png"
    mask_path = os.path.join(MASK_DIR, mask_name)

    if not os.path.isfile(mask_path):
        print("MASK NOT FOUND:", mask_path)
        failed_files.append(filename)
        continue

    # Load original image and ground-truth mask
    image_bgr = cv2.imread(image_path)
    gt = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    if image_bgr is None or gt is None:
        print("IMAGE OR MASK COULD NOT BE READ:", filename)
        failed_files.append(filename)
        continue

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

    # --------------------------------------------------------
    # Run the already-selected, frozen Trial 7 algorithm.
    # Do not change its parameters using evaluation results.
    # --------------------------------------------------------

    pred = np.asarray(algo_v7(image_bgr)).astype(bool)

    # Match prediction dimensions to the reference mask
    if pred.shape != gt.shape:
        pred = cv2.resize(
            pred.astype(np.uint8),
            (gt.shape[1], gt.shape[0]),
            interpolation=cv2.INTER_NEAREST
        ).astype(bool)

    gt_binary = gt > 0

    # --------------------------------------------------------
    # Calculate IoU and Dice
    # --------------------------------------------------------

    intersection = np.logical_and(pred, gt_binary).sum()
    union = np.logical_or(pred, gt_binary).sum()

    iou = (
        float(intersection / union)
        if union > 0
        else 1.0
    )

    denominator = pred.sum() + gt_binary.sum()

    dice = (
        float(2 * intersection / denominator)
        if denominator > 0
        else 1.0
    )

    # --------------------------------------------------------
    # Create prediction overlay
    #
    # Green = reference mask only
    # Red   = predicted mask only
    # Yellow = overlap between reference and prediction
    # --------------------------------------------------------

    overlay = image_rgb.copy()

    # Reference mask: green
    overlay[gt_binary] = (
        0.5 * overlay[gt_binary]
        + 0.5 * np.array([0, 255, 0])
    ).astype(np.uint8)

    # Prediction mask: red
    overlay[pred] = (
        0.5 * overlay[pred]
        + 0.5 * np.array([255, 0, 0])
    ).astype(np.uint8)

    # Overlap: yellow
    overlap = np.logical_and(pred, gt_binary)

    overlay[overlap] = (
        0.5 * image_rgb[overlap]
        + 0.5 * np.array([255, 255, 0])
    ).astype(np.uint8)

    # --------------------------------------------------------
    # Plot four panels
    # --------------------------------------------------------

    fig, axes = plt.subplots(1, 4, figsize=(16, 5))

    axes[0].imshow(image_rgb)
    axes[0].set_title("Original image")

    axes[1].imshow(gt_binary, cmap="gray")
    axes[1].set_title("Reference mask")

    axes[2].imshow(pred, cmap="gray")
    axes[2].set_title("Trial 7 prediction")

    axes[3].imshow(overlay)
    axes[3].set_title(
        f"Overlay\nIoU={iou:.4f}, Dice={dice:.4f}"
    )

    for ax in axes:
        ax.axis("off")

    fig.suptitle(filename, fontsize=9)
    plt.tight_layout()

    # Save comparison image
    safe_name = (
        os.path.splitext(filename)[0] + "_comparison.png"
    )

    output_path = os.path.join(VIS_DIR, safe_name)

    fig.savefig(
        output_path,
        dpi=150,
        bbox_inches="tight"
    )

    plt.close(fig)

    saved_count += 1

    print(
        f"[{saved_count}/5] Saved: {output_path}\n"
        f"    IoU={iou:.4f}, Dice={dice:.4f}"
    )

# ============================================================
# 6. SAVE VISUALIZATION SUMMARY
# ============================================================

visualization_summary = {
    "requested_images": int(len(worst)),
    "successfully_saved": int(saved_count),
    "failed_images": failed_files,
    "visualization_directory": VIS_DIR
}

summary_path = os.path.join(
    RESULT_DIR,
    "worst_cases_summary.json"
)

with open(summary_path, "w") as f:
    json.dump(visualization_summary, f, indent=2)

print("\n" + "=" * 55)
print("WORST-CASE VISUALIZATION SUMMARY")
print("=" * 55)
print(json.dumps(visualization_summary, indent=2))

# ============================================================
# 7. DISPLAY SAVED COMPARISONS IN COLAB
# ============================================================

if saved_count > 0:

    print("\nDisplaying saved comparisons...")

    saved_files = sorted([
        os.path.join(VIS_DIR, name)
        for name in os.listdir(VIS_DIR)
        if name.endswith("_comparison.png")
    ])

    for path in saved_files:
        display(plt.imread(path))
        print(os.path.basename(path))

print("\nAll available visualizations are saved in:")
print(VIS_DIR)

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# PATHS
# ============================================================

DATA_ROOT = "/content/data"
MASK_DIR = "/content/outputs/task1_masks"

RESULT_DIR = (
    "/content/drive/MyDrive/egg_task/final_evaluation"
)

WORST_DIR = os.path.join(RESULT_DIR, "worst_cases")
os.makedirs(WORST_DIR, exist_ok=True)

METRICS_CSV = os.path.join(
    RESULT_DIR, "evaluation_metrics.csv"
)

# ============================================================
# FIND IMAGE IN TRAIN / VALID / TEST
# ============================================================

def find_image_path(filename):
    for split_name in ["train", "valid", "test"]:
        for category in ["egg", "Unlabeled"]:
            candidate = os.path.join(
                DATA_ROOT,
                split_name,
                category,
                filename
            )

            if os.path.isfile(candidate):
                return candidate

    return None


# ============================================================
# METRICS
# ============================================================

def calculate_metrics(pred, gt):
    pred = np.asarray(pred, dtype=bool)
    gt = np.asarray(gt, dtype=bool)

    intersection = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()

    iou = intersection / union if union else 1.0

    denominator = pred.sum() + gt.sum()

    dice = (
        2.0 * intersection / denominator
        if denominator else 1.0
    )

    return float(iou), float(dice)


# ============================================================
# LOAD EXISTING EVALUATION RESULTS
# ============================================================

if not os.path.isfile(METRICS_CSV):
    raise FileNotFoundError(
        f"Evaluation metrics CSV not found: {METRICS_CSV}"
    )

df_eval = pd.read_csv(METRICS_CSV)

required_columns = {"filename", "IoU", "Dice"}
missing_columns = required_columns - set(df_eval.columns)

if missing_columns:
    raise ValueError(
        f"Missing columns in evaluation CSV: {missing_columns}"
    )

print("Evaluation records:", len(df_eval))

# ============================================================
# SELECT FIVE WORST CASES
# ============================================================

worst_cases = (
    df_eval.sort_values("IoU", ascending=True)
    .head(5)
    .reset_index(drop=True)
)

print("\nFive worst evaluation cases:")
display(worst_cases[["filename", "IoU", "Dice"]])


# ============================================================
# VISUALIZE EACH CASE
# ============================================================

visualization_results = []

for index, row in worst_cases.iterrows():

    filename = row["filename"]
    image_path = find_image_path(filename)

    if image_path is None:
        print(f"[{index + 1}/5] Image not found: {filename}")
        continue

    mask_name = os.path.splitext(filename)[0] + ".png"
    mask_path = os.path.join(MASK_DIR, mask_name)

    if not os.path.isfile(mask_path):
        print(f"[{index + 1}/5] Mask not found: {mask_path}")
        continue

    # Read original image and reference mask
    image_bgr = cv2.imread(image_path, cv2.IMREAD_COLOR)
    gt_raw = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    if image_bgr is None or gt_raw is None:
        print(f"[{index + 1}/5] Could not read image or mask: {filename}")
        continue

    image_rgb = cv2.cvtColor(
        image_bgr, cv2.COLOR_BGR2RGB
    )

    gt = gt_raw > 0

    # --------------------------------------------------------
    # Run the frozen Trial 7 algorithm
    # algo_v7 must already be defined in your notebook.
    # --------------------------------------------------------

    pred = np.asarray(
        algo_v7(image_bgr),
        dtype=bool
    )

    # Match prediction dimensions to reference mask
    if pred.shape != gt.shape:
        pred = cv2.resize(
            pred.astype(np.uint8),
            (gt.shape[1], gt.shape[0]),
            interpolation=cv2.INTER_NEAREST
        ).astype(bool)

    iou, dice = calculate_metrics(pred, gt)

    # --------------------------------------------------------
    # Build overlay
    # Green = ground truth only
    # Red = prediction only
    # Yellow = overlap
    # --------------------------------------------------------

    overlay = image_rgb.copy().astype(np.float32)

    gt_only = gt & ~pred
    pred_only = pred & ~gt
    overlap = gt & pred

    overlay[gt_only] = (
        0.5 * overlay[gt_only]
        + 0.5 * np.array([0, 255, 0])
    )

    overlay[pred_only] = (
        0.5 * overlay[pred_only]
        + 0.5 * np.array([255, 0, 0])
    )

    overlay[overlap] = (
        0.5 * overlay[overlap]
        + 0.5 * np.array([255, 255, 0])
    )

    overlay = np.clip(overlay, 0, 255).astype(np.uint8)

    # --------------------------------------------------------
    # Save comparison figure
    # --------------------------------------------------------

    fig, axes = plt.subplots(1, 4, figsize=(16, 5))

    axes[0].imshow(image_rgb)
    axes[0].set_title("Original image")

    axes[1].imshow(gt, cmap="gray")
    axes[1].set_title("Reference mask")

    axes[2].imshow(pred, cmap="gray")
    axes[2].set_title(
        f"Trial 7 prediction\nIoU={iou:.4f}, Dice={dice:.4f}"
    )

    axes[3].imshow(overlay)
    axes[3].set_title(
        "Overlay\nGreen=GT, Red=prediction"
    )

    for ax in axes:
        ax.axis("off")

    fig.suptitle(filename, fontsize=9)
    plt.tight_layout()

    output_name = (
        os.path.splitext(filename)[0]
        + "_comparison.png"
    )

    output_path = os.path.join(
        WORST_DIR, output_name
    )

    fig.savefig(
        output_path,
        dpi=150,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    visualization_results.append({
        "filename": filename,
        "IoU": iou,
        "Dice": dice,
        "image_path": image_path,
        "visualization_path": output_path
    })

    print(
        f"[{index + 1}/5] Saved: {output_path}\n"
        f"    IoU={iou:.4f}, Dice={dice:.4f}"
    )


# ============================================================
# SUMMARY
# ============================================================

df_visualizations = pd.DataFrame(visualization_results)

print("\nVisualization complete.")
print("Successfully visualized:", len(df_visualizations))
print("Output directory:", WORST_DIR)

if not df_visualizations.empty:
    display(df_visualizations)

In [ ]:
import os

WORST_DIR = "/content/drive/MyDrive/egg_task/final_evaluation/worst_cases"

files = [
    f for f in os.listdir(WORST_DIR)
    if f.endswith("_comparison.png")
]

print("Saved comparison images:", len(files))

for f in sorted(files):
    print(os.path.join(WORST_DIR, f))

In [ ]:
import os
import json
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/egg_task"
RESULT_DIR = os.path.join(BASE_DIR, "final_evaluation")
os.makedirs(RESULT_DIR, exist_ok=True)

metrics_path = os.path.join(
    RESULT_DIR, "evaluation_metrics.csv"
)

df = pd.read_csv(metrics_path)

summary = {
    "evaluation_images": int(len(df)),
    "mean_IoU": float(df["IoU"].mean()),
    "median_IoU": float(df["IoU"].median()),
    "mean_Dice": float(df["Dice"].mean()),
    "images_with_IoU_below_0.5": int(
        (df["IoU"] < 0.5).sum()
    ),
    "images_with_IoU_at_least_0.5": int(
        (df["IoU"] >= 0.5).sum()
    ),
    "development_best_trial": 7,
    "development_best_mIoU": 0.6405,
    "algorithm_status": "Frozen; no further tuning",
}

summary_path = os.path.join(
    RESULT_DIR, "evaluation_summary.json"
)

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

worst_path = os.path.join(
    RESULT_DIR, "worst_evaluation_cases.csv"
)

df.sort_values("IoU").head(5).to_csv(
    worst_path, index=False
)

print("Final evaluation summary:")
print(json.dumps(summary, indent=2))
print("\nSaved:", summary_path)
print("Saved:", worst_path)

In [ ]:
import os
import json
import zipfile
import shutil
import pandas as pd
from datetime import datetime

# ============================================================
# CONFIGURATION
# ============================================================

BASE_DIR = "/content/drive/MyDrive/egg_task"
RESULT_DIR = os.path.join(BASE_DIR, "final_evaluation")

OUTPUT_DIR = "/content/drive/MyDrive/egg_task/submission"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ZIP_PATH = os.path.join(
    BASE_DIR,
    "egg_segmentation_final_submission.zip"
)

# ============================================================
# VERIFY REQUIRED FILES
# ============================================================

required_files = [
    os.path.join(BASE_DIR, "experiment_log.csv"),
    os.path.join(BASE_DIR, "final_algorithm_result.json"),
    os.path.join(RESULT_DIR, "evaluation_metrics.csv"),
    os.path.join(RESULT_DIR, "evaluation_summary.json"),
    os.path.join(RESULT_DIR, "worst_evaluation_cases.csv"),
]

missing = [
    path for path in required_files
    if not os.path.isfile(path)
]

if missing:
    print("Missing required files:")
    for path in missing:
        print(" -", path)
    raise FileNotFoundError(
        "Please fix the missing files before packaging."
    )

# ============================================================
# LOAD FINAL RESULTS
# ============================================================

with open(
    os.path.join(RESULT_DIR, "evaluation_summary.json")
) as f:
    summary = json.load(f)

with open(
    os.path.join(BASE_DIR, "final_algorithm_result.json")
) as f:
    algorithm = json.load(f)

# ============================================================
# COPY PROJECT OUTPUTS INTO SUBMISSION DIRECTORY
# ============================================================

files_to_copy = [
    (
        os.path.join(BASE_DIR, "experiment_log.csv"),
        "experiment_log.csv",
    ),
    (
        os.path.join(BASE_DIR, "final_algorithm_result.json"),
        "final_algorithm_result.json",
    ),
    (
        os.path.join(RESULT_DIR, "evaluation_metrics.csv"),
        "evaluation_metrics.csv",
    ),
    (
        os.path.join(RESULT_DIR, "evaluation_summary.json"),
        "evaluation_summary.json",
    ),
    (
        os.path.join(RESULT_DIR, "worst_evaluation_cases.csv"),
        "worst_evaluation_cases.csv",
    ),
]

for source, filename in files_to_copy:
    shutil.copy2(
        source,
        os.path.join(OUTPUT_DIR, filename)
    )

# ============================================================
# COPY WORST-CASE VISUALIZATIONS
# ============================================================

worst_source = os.path.join(
    RESULT_DIR, "worst_cases"
)

worst_destination = os.path.join(
    OUTPUT_DIR, "worst_cases"
)

if os.path.isdir(worst_source):
    shutil.copytree(
        worst_source,
        worst_destination,
        dirs_exist_ok=True
    )

# ============================================================
# COPY TASK 1 MASKS
# ============================================================

MASK_DIR = "/content/outputs/task1_masks"

if os.path.isdir(MASK_DIR):
    masks_destination = os.path.join(
        OUTPUT_DIR, "task1_masks"
    )

    shutil.copytree(
        MASK_DIR,
        masks_destination,
        dirs_exist_ok=True
    )

    mask_count = len([
        f for f in os.listdir(masks_destination)
        if f.lower().endswith(".png")
    ])
else:
    mask_count = 0
    print("Warning: Task 1 masks directory not found.")

# ============================================================
# CREATE README
# ============================================================

readme = f"""
EGG SEGMENTATION PROJECT — FINAL SUBMISSION
============================================

Algorithm
---------
Selected development trial: {algorithm.get("selected_trial", 7)}
Algorithm description: {algorithm.get("algorithm", "Hybrid segmentation with shape and area validation")}
Development images: {algorithm.get("development_images", 88)}
Development mIoU: {algorithm.get("mIoU", 0.6405)}
Development median IoU: {algorithm.get("median_IoU", 0.8057)}
Development mean Dice: {algorithm.get("mean_Dice", 0.7022)}

Evaluation
----------
Evaluation images: {summary["evaluation_images"]}
Mean IoU: {summary["mean_IoU"]:.4f}
Median IoU: {summary["median_IoU"]:.4f}
Mean Dice: {summary["mean_Dice"]:.4f}
Images with IoU below 0.5: {summary["images_with_IoU_below_0.5"]}
Images with IoU at least 0.5: {summary["images_with_IoU_at_least_0.5"]}

Task 1
------
PNG masks included: {mask_count}

Contents
--------
experiment_log.csv
    Results of the recorded segmentation experiments.

final_algorithm_result.json
    Selected algorithm and development-set results.

evaluation_metrics.csv
    Per-image evaluation metrics.

evaluation_summary.json
    Aggregate evaluation metrics.

worst_evaluation_cases.csv
    Five lowest-IoU evaluation cases.

worst_cases/
    Saved visual comparisons for the worst evaluation cases.

task1_masks/
    Generated Task 1 masks, when available.

Important
---------
The selected algorithm is frozen. No further development-set tuning
was performed after selecting Trial 7.

Evaluation results are reported separately from development results.
The evaluation set contains only {summary["evaluation_images"]} images.

Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
"""

readme_path = os.path.join(
    OUTPUT_DIR, "README.txt"
)

with open(readme_path, "w") as f:
    f.write(readme.strip() + "\n")

# ============================================================
# CREATE ZIP
# ============================================================

if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zf:

    for root, dirs, files in os.walk(OUTPUT_DIR):
        for filename in files:
            full_path = os.path.join(root, filename)

            relative_path = os.path.relpath(
                full_path, OUTPUT_DIR
            )

            zf.write(
                full_path,
                arcname=os.path.join(
                    "egg_segmentation_submission",
                    relative_path
                )
            )

# ============================================================
# VERIFY PACKAGE
# ============================================================

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    bad_file = zf.testzip()
    zip_contents = zf.namelist()

if bad_file is not None:
    raise RuntimeError(
        f"ZIP integrity check failed: {bad_file}"
    )

print("FINAL SUBMISSION PACKAGE CREATED")
print("=" * 45)
print("ZIP:", ZIP_PATH)
print("ZIP size (MB):", round(os.path.getsize(ZIP_PATH) / (1024**2), 2))
print("Mask PNGs included:", mask_count)
print("ZIP integrity:", "PASSED")
print("\nFiles included:")
for name in zip_contents:
    print(name)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import zipfile
import shutil

BASE = "/content/drive/MyDrive/egg_task"
PACKAGE = os.path.join(BASE, "egg_segmentation_final_submission")
ZIP_PATH = os.path.join(BASE, "egg_segmentation_final_submission.zip")

# Re-create the package folder
if os.path.exists(PACKAGE):
    shutil.rmtree(PACKAGE)
os.makedirs(PACKAGE, exist_ok=True)

def copy_item(source, destination):
    if os.path.exists(source):
        if os.path.isdir(source):
            shutil.copytree(source, destination, dirs_exist_ok=True)
        else:
            os.makedirs(os.path.dirname(destination), exist_ok=True)
            shutil.copy2(source, destination)
        print("Included:", os.path.basename(source))
    else:
        print("WARNING: Missing:", source)

# Experiment results
copy_item(
    os.path.join(BASE, "experiment_log.csv"),
    os.path.join(PACKAGE, "experiment_log.csv")
)
copy_item(
    os.path.join(BASE, "final_algorithm_result.json"),
    os.path.join(PACKAGE, "final_algorithm_result.json")
)

# Evaluation results
EVAL = os.path.join(BASE, "final_evaluation")

for filename in [
    "evaluation_metrics.csv",
    "evaluation_summary.json",
    "worst_evaluation_cases.csv"
]:
    copy_item(
        os.path.join(EVAL, filename),
        os.path.join(PACKAGE, "evaluation", filename)
    )

copy_item(
    os.path.join(EVAL, "worst_cases"),
    os.path.join(PACKAGE, "evaluation", "worst_cases")
)

# Task 1 masks
copy_item(
    "/content/outputs/task1_masks",
    os.path.join(PACKAGE, "task1_masks")
)

# Project README
readme = """EGG SEGMENTATION PROJECT

Selected algorithm: Trial 7
Algorithm: Hybrid segmentation with shape and area validation

Development results:
Images: 88
Mean IoU: 0.6405
Median IoU: 0.8057
Mean Dice: 0.7022

Final evaluation results:
Images: 22
Mean IoU: 0.5320
Median IoU: 0.5818
Mean Dice: 0.6039
Images with IoU >= 0.5: 12

The algorithm was frozen before final evaluation.
Refer to the original task requirements for all required deliverables.
"""

with open(os.path.join(PACKAGE, "README.txt"), "w") as f:
    f.write(readme)

# Create the ZIP archive
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

with zipfile.ZipFile(
    ZIP_PATH, "w", zipfile.ZIP_DEFLATED
) as archive:
    for root, dirs, files in os.walk(PACKAGE):
        for filename in files:
            full_path = os.path.join(root, filename)
            relative_path = os.path.relpath(full_path, BASE)
            archive.write(full_path, relative_path)

# Verify the archive
with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    bad_file = archive.testzip()
    contents = archive.namelist()

print("\nZIP CREATED:", ZIP_PATH)
print("SIZE: {:.2f} MB".format(os.path.getsize(ZIP_PATH) / (1024 * 1024)))
print("FILE COUNT:", len(contents))
print("ZIP INTEGRITY:", "PASS" if bad_file is None else "FAIL")

print("\nCONTENTS:")
for item in contents:
    print(item)

In [ ]:
!pip -q install --upgrade yt-dlp
!yt-dlp --version

In [ ]:
import subprocess
import os

VIDEO_URL = "https://www.youtube.com/watch?v=Y4JyV94y2_A"
VIDEO_DIR = "/content/egg_video"
os.makedirs(VIDEO_DIR, exist_ok=True)

result = subprocess.run(
    [
        "yt-dlp",
        "--no-playlist",
        "-f", "best[ext=mp4]/best",
        "--merge-output-format", "mp4",
        "-o", f"{VIDEO_DIR}/input_video.%(ext)s",
        VIDEO_URL
    ],
    capture_output=True,
    text=True
)

print(result.stdout[-2000:])
print(result.stderr[-3000:])
print("Return code:", result.returncode)
print("Files:", os.listdir(VIDEO_DIR))

In [ ]:
!pip install -q --upgrade "yt-dlp[default]"
!curl -fsSL https://deno.land/install.sh | sh

import os
os.environ["PATH"] = os.path.expanduser("~/.deno/bin") + ":" + os.environ["PATH"]

!deno --version
!yt-dlp --version

In [ ]:
import os
import subprocess

VIDEO_URL = "https://www.youtube.com/watch?v=Y4JyV94y2_A"
VIDEO_DIR = "/content/egg_video"
os.makedirs(VIDEO_DIR, exist_ok=True)

result = subprocess.run(
    [
        "yt-dlp",
        "--js-runtimes", "deno",
        "--remote-components", "ejs:npm",
        "--no-playlist",
        "-f", "best[ext=mp4]/best",
        "-o", f"{VIDEO_DIR}/input_video.%(ext)s",
        VIDEO_URL
    ],
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print(result.stdout[-2000:])
print(result.stderr[-3000:])
print("Files:", os.listdir(VIDEO_DIR))

In [ ]:
import cv2
import os
from IPython.display import display, Image

VIDEO_PATH = "/content/egg_video/input_video.mp4"

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise RuntimeError("Could not open the downloaded video.")

fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print("VIDEO INFORMATION")
print("-" * 35)
print(f"Resolution: {width} x {height}")
print(f"FPS: {fps:.2f}")
print(f"Frame count: {frame_count}")
print(f"Duration: {frame_count / fps:.2f} seconds" if fps > 0 else "Duration: unknown")

os.makedirs("/content/egg_video/previews", exist_ok=True)

for label, position in [
    ("start", 0),
    ("middle", frame_count // 2),
    ("end", max(0, frame_count - 1))
]:
    cap.set(cv2.CAP_PROP_POS_FRAMES, position)
    success, frame = cap.read()

    if success:
        path = f"/content/egg_video/previews/{label}.jpg"
        cv2.imwrite(path, frame)
        print(f"\n{label.upper()} FRAME")
        display(Image(filename=path))

cap.release()

In [ ]:
import cv2
import matplotlib.pyplot as plt
import os

VIDEO_PATH = "/content/egg_video/input_video.mp4"
OUTPUT_DIR = "/content/egg_video/previews"
os.makedirs(OUTPUT_DIR, exist_ok=True)

cap = cv2.VideoCapture(VIDEO_PATH)

# Sample one frame every second for the first 10 seconds
times = list(range(0, 10))
frames = []

for t in times:
    cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000)
    success, frame = cap.read()

    if success:
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append((t, frame))

cap.release()

fig, axes = plt.subplots(2, 5, figsize=(18, 8))

for ax, (t, frame) in zip(axes.flat, frames):
    ax.imshow(frame)
    ax.set_title(f"Time: {t}s")
    ax.axis("off")

for ax in axes.flat[len(frames):]:
    ax.axis("off")

plt.tight_layout()

contact_path = f"{OUTPUT_DIR}/contact_sheet.png"
plt.savefig(contact_path, dpi=120, bbox_inches="tight")
plt.show()

print("Contact sheet saved to:", contact_path)

In [ ]:
import cv2
import matplotlib.pyplot as plt

VIDEO_PATH = "/content/egg_video/input_video.mp4"

cap = cv2.VideoCapture(VIDEO_PATH)
times = list(range(10, 60, 5)) + [59]
frames = []

for t in times:
    cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000)
    success, frame = cap.read()
    if success:
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append((t, frame))

cap.release()

fig, axes = plt.subplots(2, 6, figsize=(20, 7))

for ax, (t, frame) in zip(axes.flat, frames):
    ax.imshow(frame)
    ax.set_title(f"{t} seconds")
    ax.axis("off")

for ax in axes.flat[len(frames):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import numpy as np
import os
import csv
from scipy.optimize import linear_sum_assignment
from IPython.display import display, Video

VIDEO_PATH = "/content/egg_video/input_video.mp4"
OUTPUT_DIR = "/content/egg_video/tracking"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_VIDEO = f"{OUTPUT_DIR}/egg_tracking_preview.mp4"
OUTPUT_CSV = f"{OUTPUT_DIR}/egg_tracks.csv"

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise RuntimeError("Could not open input video.")

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Analyze the section where eggs are visible.
START_SEC = 10
END_SEC = 43

start_frame = int(START_SEC * fps)
end_frame = min(int(END_SEC * fps), int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))

cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

writer = cv2.VideoWriter(
    OUTPUT_VIDEO,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

if not writer.isOpened():
    cap.release()
    raise RuntimeError("Could not create output video.")

# Tracking state
tracks = {}
next_id = 1
MAX_MATCH_DISTANCE = 100
MAX_MISSED_FRAMES = 12
csv_rows = []

frame_index = start_frame

while frame_index < end_frame:
    ok, frame = cap.read()
    if not ok:
        break

    # Resize for faster segmentation.
    scale = 0.5
    small = cv2.resize(frame, None, fx=scale, fy=scale)
    hsv = cv2.cvtColor(small, cv2.COLOR_BGR2HSV)

    # White/bright objects. Tune after inspecting the output.
    lower = np.array([0, 0, 145], dtype=np.uint8)
    upper = np.array([180, 110, 255], dtype=np.uint8)
    mask = cv2.inRange(hsv, lower, upper)

    # Reduce speckle noise and fill small gaps.
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(
        mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    detections = []

    for contour in contours:
        area = cv2.contourArea(contour)
        if not (180 <= area <= 9000):
            continue

        perimeter = cv2.arcLength(contour, True)
        if perimeter <= 0:
            continue

        circularity = 4 * np.pi * area / (perimeter * perimeter)
        x, y, w, h = cv2.boundingRect(contour)

        # Eggs are approximately oval; avoid very thin regions.
        aspect = w / max(h, 1)
        if circularity < 0.35 or not (0.45 <= aspect <= 2.2):
            continue

        # Ignore regions near the outer video edges where the watermark sits.
        cx, cy = x + w / 2, y + h / 2
        if cx > small.shape[1] * 0.90 and cy > small.shape[0] * 0.85:
            continue

        detections.append({
            "center": (cx / scale, cy / scale),
            "bbox": (
                int(x / scale), int(y / scale),
                int(w / scale), int(h / scale)
            ),
            "area": area / (scale * scale)
        })

    # Match current detections to existing tracks using nearest-centroid distance.
    track_ids = list(tracks.keys())
    matched_detections = set()
    matched_tracks = set()

    if track_ids and detections:
        cost = np.zeros((len(track_ids), len(detections)), dtype=np.float32)

        for i, tid in enumerate(track_ids):
            old_cx, old_cy = tracks[tid]["center"]
            for j, det in enumerate(detections):
                new_cx, new_cy = det["center"]
                cost[i, j] = np.hypot(new_cx - old_cx, new_cy - old_cy)

        rows, cols = linear_sum_assignment(cost)

        for i, j in zip(rows, cols):
            if cost[i, j] <= MAX_MATCH_DISTANCE:
                tid = track_ids[i]
                tracks[tid]["center"] = detections[j]["center"]
                tracks[tid]["bbox"] = detections[j]["bbox"]
                tracks[tid]["missed"] = 0
                tracks[tid]["age"] += 1
                matched_detections.add(j)
                matched_tracks.add(tid)

    # Create tracks for unmatched detections.
    for j, det in enumerate(detections):
        if j not in matched_detections:
            tracks[next_id] = {
                "center": det["center"],
                "bbox": det["bbox"],
                "missed": 0,
                "age": 1
            }
            next_id += 1

    # Age tracks not matched in this frame.
    for tid in list(tracks.keys()):
        if tid not in matched_tracks and tracks[tid]["center"] not in [
            d["center"] for d in detections
        ]:
            tracks[tid]["missed"] += 1

        if tracks[tid]["missed"] > MAX_MISSED_FRAMES:
            del tracks[tid]

    # Draw detections with track IDs.
    for tid, track in tracks.items():
        if track["missed"] > 0:
            continue

        x, y, w, h = track["bbox"]
        cx, cy = map(int, track["center"])

        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 3)
        cv2.circle(frame, (cx, cy), 5, (0, 0, 255), -1)
        cv2.putText(
            frame, f"Egg {tid}", (x, max(30, y - 10)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2
        )

        csv_rows.append({
            "frame": frame_index,
            "time_seconds": round(frame_index / fps, 3),
            "track_id": tid,
            "center_x": cx,
            "center_y": cy,
            "bbox_x": x,
            "bbox_y": y,
            "bbox_width": w,
            "bbox_height": h
        })

    cv2.putText(
        frame, f"Time: {frame_index / fps:.1f}s",
        (30, 45), cv2.FONT_HERSHEY_SIMPLEX, 1,
        (0, 255, 255), 2
    )

    writer.write(frame)
    frame_index += 1

cap.release()
writer.release()

with open(OUTPUT_CSV, "w", newline="") as f:
    fieldnames = [
        "frame", "time_seconds", "track_id", "center_x", "center_y",
        "bbox_x", "bbox_y", "bbox_width", "bbox_height"
    ]
    csv_writer = csv.DictWriter(f, fieldnames=fieldnames)
    csv_writer.writeheader()
    csv_writer.writerows(csv_rows)

print("Preview video:", OUTPUT_VIDEO)
print("Tracking CSV:", OUTPUT_CSV)
print("Detection records:", len(csv_rows))
print("Track IDs created:", next_id - 1)

display(Video(OUTPUT_VIDEO, embed=True, width=800))

In [ ]:
import os
import subprocess
import cv2
import matplotlib.pyplot as plt

INPUT_VIDEO = "/content/egg_video/tracking/egg_tracking_preview.mp4"
OUTPUT_VIDEO = "/content/egg_video/tracking/egg_tracking_preview_h264.mp4"

# Convert to browser-friendly H.264
result = subprocess.run(
    [
        "ffmpeg", "-y",
        "-i", INPUT_VIDEO,
        "-vcodec", "libx264",
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
        "-an",
        OUTPUT_VIDEO
    ],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print(result.stderr[-2500:])
    raise RuntimeError("Video conversion failed.")

print("Converted video:", OUTPUT_VIDEO)
print("File size (MB):", round(os.path.getsize(OUTPUT_VIDEO) / (1024 * 1024), 2))

# Inspect four frames from the annotated output
cap = cv2.VideoCapture(OUTPUT_VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS)

times = [2, 8, 15, 25]
frames = []

for t in times:
    cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000)
    ok, frame = cap.read()
    if ok:
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append((t, frame))

cap.release()

fig, axes = plt.subplots(2, 2, figsize=(16, 9))

for ax, (t, frame) in zip(axes.flat, frames):
    ax.imshow(frame)
    ax.set_title(f"Tracking output at {t}s")
    ax.axis("off")

plt.tight_layout()
plt.show()

# Display the converted video
from IPython.display import display, Video
display(Video(OUTPUT_VIDEO, embed=True, width=800))

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

VIDEO_PATH = "/content/egg_video/input_video.mp4"

cap = cv2.VideoCapture(VIDEO_PATH)

def detect_eggs(frame):
    h, w = frame.shape[:2]

    # Restrict processing to the tabletop region.
    roi_y1 = int(h * 0.40)
    roi_y2 = int(h * 0.88)
    roi_x1 = int(w * 0.03)
    roi_x2 = int(w * 0.94)

    roi = frame[roi_y1:roi_y2, roi_x1:roi_x2]
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

    # Bright, low-saturation white egg pixels.
    mask = cv2.inRange(
        hsv,
        np.array([0, 0, 155], dtype=np.uint8),
        np.array([180, 105, 255], dtype=np.uint8)
    )

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(
        mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    output = frame.copy()
    detections = []

    for contour in contours:
        area = cv2.contourArea(contour)
        if not (2500 <= area <= 50000):
            continue

        perimeter = cv2.arcLength(contour, True)
        if perimeter <= 0:
            continue

        circularity = 4 * np.pi * area / (perimeter * perimeter)
        x, y, bw, bh = cv2.boundingRect(contour)
        aspect = bw / max(bh, 1)

        # Eggs are relatively large, rounded objects.
        if circularity < 0.35 or not (0.65 <= aspect <= 1.8):
            continue

        x += roi_x1
        y += roi_y1
        detections.append((x, y, bw, bh, area))

        cv2.rectangle(output, (x, y), (x + bw, y + bh), (0, 255, 0), 3)

    return output, mask, detections

times = [12, 15, 20, 25, 30, 35, 40]

fig, axes = plt.subplots(len(times), 2, figsize=(14, 4 * len(times)))

for i, t in enumerate(times):
    cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000)
    ok, frame = cap.read()

    if not ok:
        continue

    annotated, mask, detections = detect_eggs(frame)

    axes[i, 0].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    axes[i, 0].set_title(f"{t}s — {len(detections)} candidate eggs")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(mask, cmap="gray")
    axes[i, 1].set_title("Threshold mask")
    axes[i, 1].axis("off")

    print(f"{t}s: {len(detections)} candidate detections")

cap.release()
plt.tight_layout()
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

VIDEO_PATH = "/content/egg_video/input_video.mp4"

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)

def detect_eggs(frame):
    h, w = frame.shape[:2]

    # Restrict processing to the tabletop.
    x1, x2 = int(w * 0.03), int(w * 0.94)
    y1, y2 = int(h * 0.42), int(h * 0.82)

    roi = frame[y1:y2, x1:x2]
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

    # White objects: high brightness and relatively low saturation.
    mask = cv2.inRange(
        hsv,
        np.array([0, 0, 155]),
        np.array([180, 95, 255])
    )

    # Remove small noise without excessively expanding objects.
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

    contours, _ = cv2.findContours(
        mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    detections = []

    for contour in contours:
        area = cv2.contourArea(contour)

        if area < 1800 or area > 18000:
            continue

        bx, by, bw, bh = cv2.boundingRect(contour)

        # Reject objects touching the ROI boundary.
        if bx <= 2 or by <= 2:
            continue
        if bx + bw >= mask.shape[1] - 2:
            continue
        if by + bh >= mask.shape[0] - 2:
            continue

        aspect = bw / max(bh, 1)
        if not (0.85 <= aspect <= 2.0):
            continue

        perimeter = cv2.arcLength(contour, True)
        if perimeter == 0:
            continue

        circularity = 4 * np.pi * area / (perimeter * perimeter)

        if circularity < 0.50:
            continue

        # Convert ROI coordinates to original-frame coordinates.
        detections.append(
            (x1 + bx, y1 + by, bw, bh, area, circularity)
        )

    # Draw detections on a copy of the full frame.
    annotated = frame.copy()

    for i, (x, y, bw, bh, area, circularity) in enumerate(detections, 1):
        cv2.rectangle(
            annotated, (x, y), (x + bw, y + bh), (0, 255, 0), 3
        )
        cv2.putText(
            annotated, f"Egg candidate {i}",
            (x, max(25, y - 8)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2
        )

    return annotated, mask, detections


times_to_check = [12, 15, 20, 25, 30, 35, 40]

fig, axes = plt.subplots(len(times_to_check), 2, figsize=(14, 28))

for row, sec in enumerate(times_to_check):
    cap.set(cv2.CAP_PROP_POS_MSEC, sec * 1000)
    ok, frame = cap.read()

    if not ok:
        continue

    annotated, mask, detections = detect_eggs(frame)

    axes[row, 0].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    axes[row, 0].set_title(
        f"{sec}s — {len(detections)} detections"
    )
    axes[row, 0].axis("off")

    axes[row, 1].imshow(mask, cmap="gray")
    axes[row, 1].set_title("Threshold mask")
    axes[row, 1].axis("off")

plt.tight_layout()
plt.show()

cap.release()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

VIDEO_PATH = "/content/egg_video/input_video.mp4"

cap = cv2.VideoCapture(VIDEO_PATH)

def inspect_egg_candidates(frame):
    h, w = frame.shape[:2]

    # Table region
    x1, x2 = int(w * 0.03), int(w * 0.94)
    y1, y2 = int(h * 0.42), int(h * 0.82)

    roi = frame[y1:y2, x1:x2]
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

    mask = cv2.inRange(
        hsv,
        np.array([0, 0, 155]),
        np.array([180, 105, 255])
    )

    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (5, 5)
    )
    mask = cv2.morphologyEx(
        mask, cv2.MORPH_OPEN, kernel
    )

    contours, _ = cv2.findContours(
        mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    annotated = frame.copy()
    measurements = []

    for contour in contours:
        area = cv2.contourArea(contour)

        if area < 500:
            continue

        x, y, bw, bh = cv2.boundingRect(contour)
        aspect = bw / max(bh, 1)

        perimeter = cv2.arcLength(contour, True)
        circularity = (
            4 * np.pi * area / (perimeter * perimeter)
            if perimeter > 0 else 0
        )

        measurements.append({
            "area": round(area),
            "width": bw,
            "height": bh,
            "aspect": round(aspect, 2),
            "circularity": round(circularity, 2)
        })

        # Display larger contours for visual inspection.
        if area >= 1500:
            cv2.rectangle(
                annotated,
                (x1 + x, y1 + y),
                (x1 + x + bw, y1 + y + bh),
                (0, 255, 0),
                3
            )

    return annotated, mask, measurements


for sec in [12, 20, 25, 35, 40]:
    cap.set(cv2.CAP_PROP_POS_MSEC, sec * 1000)
    ok, frame = cap.read()

    if not ok:
        continue

    annotated, mask, measurements = inspect_egg_candidates(frame)

    print(f"\nTime: {sec} seconds")
    for item in sorted(
        measurements, key=lambda d: d["area"], reverse=True
    )[:10]:
        print(item)

    plt.figure(figsize=(13, 4))

    plt.subplot(1, 2, 1)
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.title(f"Candidates at {sec}s")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(mask, cmap="gray")
    plt.title("Threshold mask")
    plt.axis("off")

    plt.show()

cap.release()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

VIDEO_PATH = "/content/egg_video/input_video.mp4"

def detect_eggs(frame):
    h, w = frame.shape[:2]

    # Table region
    x1, x2 = int(w * 0.03), int(w * 0.94)
    y1, y2 = int(h * 0.42), int(h * 0.82)

    roi = frame[y1:y2, x1:x2]
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

    mask = cv2.inRange(
        hsv,
        np.array([0, 0, 155]),
        np.array([180, 105, 255])
    )

    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (5, 5)
    )
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

    contours, _ = cv2.findContours(
        mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    detections = []

    for contour in contours:
        area = cv2.contourArea(contour)

        # Egg-sized contours
        if not 18000 <= area <= 65000:
            continue

        x, y, bw, bh = cv2.boundingRect(contour)
        aspect = bw / max(bh, 1)

        if not 0.90 <= aspect <= 1.65:
            continue

        perimeter = cv2.arcLength(contour, True)
        circularity = (
            4 * np.pi * area / (perimeter * perimeter)
            if perimeter > 0 else 0
        )

        if circularity < 0.40:
            continue

        # Convert coordinates back to the full frame.
        full_x = x1 + x
        full_y = y1 + y

        detections.append({
            "bbox": (full_x, full_y, bw, bh),
            "center": (
                full_x + bw // 2,
                full_y + bh // 2
            ),
            "area": round(area),
            "circularity": round(circularity, 2)
        })

    # Show the candidate detections.
    annotated = frame.copy()

    for i, det in enumerate(
        sorted(detections, key=lambda d: d["center"][0]), 1
    ):
        x, y, bw, bh = det["bbox"]

        cv2.rectangle(
            annotated,
            (x, y),
            (x + bw, y + bh),
            (0, 255, 0),
            3
        )

        cv2.putText(
            annotated,
            f"Egg {i}",
            (x, max(y - 10, 25)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 0),
            2
        )

    return annotated, mask, detections


cap = cv2.VideoCapture(VIDEO_PATH)

for sec in [12, 20, 25, 30, 35, 40]:
    cap.set(cv2.CAP_PROP_POS_MSEC, sec * 1000)
    ok, frame = cap.read()

    if not ok:
        continue

    annotated, mask, detections = detect_eggs(frame)

    print(f"{sec}s: {len(detections)} detections")
    print([
        {
            "center": d["center"],
            "area": d["area"],
            "circularity": d["circularity"]
        }
        for d in detections
    ])

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.title(f"Egg candidates — {sec}s")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(mask, cmap="gray")
    plt.title("Threshold mask")
    plt.axis("off")

    plt.show()

cap.release()

In [ ]:
import cv2
import numpy as np

def detect_eggs(frame):
    """
    Detect white eggs on the brown tabletop.
    Returns:
        annotated_frame, full_frame_mask, detections
    """

    h, w = frame.shape[:2]
    annotated = frame.copy()
    full_mask = np.zeros((h, w), dtype=np.uint8)

    # Restrict processing to the tabletop, excluding the shirt and watermark.
    x1, x2 = int(w * 0.03), int(w * 0.94)
    y1, y2 = int(h * 0.53), int(h * 0.86)

    roi = frame[y1:y2, x1:x2]

    # White eggs have high brightness and relatively low saturation.
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

    mask = cv2.inRange(
        hsv,
        np.array([0, 0, 155], dtype=np.uint8),
        np.array([180, 110, 255], dtype=np.uint8)
    )

    # Remove small noise without excessively merging nearby objects.
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(
        mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    detections = []

    for contour in contours:
        area = cv2.contourArea(contour)

        if area < 9000 or area > 65000:
            continue

        perimeter = cv2.arcLength(contour, True)
        if perimeter == 0:
            continue

        circularity = 4 * np.pi * area / (perimeter * perimeter)

        x, y, bw, bh = cv2.boundingRect(contour)
        aspect = bw / max(bh, 1)

        # Reject long text fragments and very irregular shapes.
        if not (0.75 <= aspect <= 1.65):
            continue

        if circularity < 0.45:
            continue

        # Translate ROI coordinates back to full-frame coordinates.
        X, Y = x + x1, y + y1
        cx, cy = X + bw // 2, Y + bh // 2

        detections.append({
            "bbox": (X, Y, bw, bh),
            "center": (cx, cy),
            "area": float(area),
            "circularity": float(circularity)
        })

        cv2.rectangle(
            annotated,
            (X, Y),
            (X + bw, Y + bh),
            (0, 255, 0),
            3
        )

        cv2.circle(annotated, (cx, cy), 5, (0, 0, 255), -1)

    # Place the ROI mask into a full-size mask for visualization.
    full_mask[y1:y2, x1:x2] = mask

    return annotated, full_mask, detections

In [ ]:
import cv2
import matplotlib.pyplot as plt

VIDEO_PATH = "/content/egg_video/input_video.mp4"

test_times = [12, 20, 25, 30, 35, 40]

cap = cv2.VideoCapture(VIDEO_PATH)

fig, axes = plt.subplots(len(test_times), 2, figsize=(14, 24))

for row, sec in enumerate(test_times):
    cap.set(cv2.CAP_PROP_POS_MSEC, sec * 1000)
    ok, frame = cap.read()

    if not ok:
        print(f"Could not read frame at {sec}s")
        continue

    annotated, mask, detections = detect_eggs(frame)

    axes[row, 0].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    axes[row, 0].set_title(f"{sec}s — {len(detections)} detections")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(mask, cmap="gray")
    axes[row, 1].set_title("Egg threshold mask")
    axes[row, 1].axis("off")

    print(
        f"{sec}s:",
        [
            {
                "center": d["center"],
                "area": round(d["area"]),
                "circularity": round(d["circularity"], 2)
            }
            for d in detections
        ]
    )

cap.release()
plt.tight_layout()
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

VIDEO_PATH = "/content/egg_video/input_video.mp4"

cap = cv2.VideoCapture(VIDEO_PATH)

def detect_eggs(frame):
    h, w = frame.shape[:2]

    # Restrict processing to the tabletop
    y1, y2 = int(h * 0.53), int(h * 0.82)
    x1, x2 = int(w * 0.02), int(w * 0.96)

    roi = frame[y1:y2, x1:x2]
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

    # Bright, low-saturation regions
    mask = cv2.inRange(
        hsv,
        np.array([0, 0, 165]),
        np.array([180, 100, 255])
    )

    # Remove small noise without joining nearby objects
    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (5, 5)
    )
    mask = cv2.morphologyEx(
        mask, cv2.MORPH_OPEN, kernel
    )

    contours, _ = cv2.findContours(
        mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    detections = []

    for contour in contours:
        area = cv2.contourArea(contour)

        if area < 5000 or area > 65000:
            continue

        perimeter = cv2.arcLength(contour, True)
        if perimeter == 0:
            continue

        circularity = 4 * np.pi * area / (perimeter * perimeter)

        x, y, bw, bh = cv2.boundingRect(contour)
        aspect = bw / max(bh, 1)

        # Filter out long text fragments and irregular shapes
        if circularity < 0.45:
            continue

        if not (0.75 <= aspect <= 1.8):
            continue

        # Convert ROI coordinates to full-frame coordinates
        x += x1
        y += y1

        detections.append((x, y, bw, bh, round(area)))

    # Draw candidate detections
    annotated = frame.copy()

    for i, (x, y, bw, bh, area) in enumerate(
        sorted(detections, key=lambda d: d[0])
    ):
        cv2.rectangle(
            annotated, (x, y), (x + bw, y + bh),
            (0, 255, 0), 3
        )
        cv2.putText(
            annotated, f"Egg {i+1}",
            (x, max(25, y - 10)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.8,
            (0, 255, 0), 2
        )

    return annotated, mask, detections


# Test representative frames
for sec in [12, 20, 25, 30, 35, 40]:
    cap.set(cv2.CAP_PROP_POS_MSEC, sec * 1000)
    ok, frame = cap.read()

    if not ok:
        print(f"Could not read frame at {sec}s")
        continue

    annotated, mask, detections = detect_eggs(frame)

    print(f"{sec}s: {len(detections)} candidate(s)", detections)

    plt.figure(figsize=(13, 4))

    plt.subplot(1, 2, 1)
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.title(f"Detection at {sec}s")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(mask, cmap="gray")
    plt.title("Threshold mask")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

cap.release()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

VIDEO_PATH = "/content/egg_video/input_video.mp4"

cap = cv2.VideoCapture(VIDEO_PATH)

def detect_eggs(frame):
    h, w = frame.shape[:2]

    # Keep the tabletop; include the left and right edges where eggs appear
    x1, x2 = int(w * 0.02), int(w * 0.98)
    y1, y2 = int(h * 0.42), int(h * 0.84)

    roi = frame[y1:y2, x1:x2]
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

    # White eggs: low saturation and high brightness
    mask = cv2.inRange(
        hsv,
        np.array([0, 0, 150], dtype=np.uint8),
        np.array([180, 115, 255], dtype=np.uint8)
    )

    # Remove tiny noise without excessively shrinking the eggs
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(
        mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    detections = []

    for contour in contours:
        area = cv2.contourArea(contour)

        if area < 3000 or area > 55000:
            continue

        x, y, bw, bh = cv2.boundingRect(contour)

        # Reject extremely narrow or elongated objects
        aspect = bw / max(bh, 1)
        if not 0.65 <= aspect <= 1.9:
            continue

        perimeter = cv2.arcLength(contour, True)
        circularity = (
            4 * np.pi * area / (perimeter * perimeter)
            if perimeter > 0 else 0
        )

        if circularity < 0.45:
            continue

        # Convert ROI coordinates back to full-frame coordinates
        detections.append({
            "bbox": (x + x1, y + y1, bw, bh),
            "center": (x + x1 + bw // 2, y + y1 + bh // 2),
            "area": round(area),
            "circularity": round(circularity, 2)
        })

    return detections, mask, (x1, y1, x2, y2)


# Inspect representative frames
times_to_check = [12, 20, 25, 30, 35, 40]

fig, axes = plt.subplots(len(times_to_check), 2, figsize=(14, 24))

for row, seconds in enumerate(times_to_check):
    cap.set(cv2.CAP_PROP_POS_MSEC, seconds * 1000)
    ok, frame = cap.read()

    if not ok:
        continue

    detections, mask, _ = detect_eggs(frame)

    preview = frame.copy()

    for i, detection in enumerate(detections, start=1):
        x, y, bw, bh = detection["bbox"]

        cv2.rectangle(
            preview, (x, y), (x + bw, y + bh),
            (0, 255, 0), 3
        )

        cv2.putText(
            preview, f"Egg {i}",
            (x, max(30, y - 10)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.8,
            (0, 255, 0), 2
        )

    axes[row, 0].imshow(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB))
    axes[row, 0].set_title(f"{seconds}s — {len(detections)} detections")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(mask, cmap="gray")
    axes[row, 1].set_title("Tabletop threshold mask")
    axes[row, 1].axis("off")

    print(f"{seconds}s:", detections)

cap.release()
plt.tight_layout()
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

VIDEO_PATH = "/content/egg_video/input_video.mp4"

def detect_eggs_v2(frame):
    h, w = frame.shape[:2]

    x1, x2 = int(w * 0.02), int(w * 0.98)
    y1, y2 = int(h * 0.42), int(h * 0.84)

    roi = frame[y1:y2, x1:x2]
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

    mask = cv2.inRange(
        hsv,
        np.array([0, 0, 160], dtype=np.uint8),
        np.array([180, 100, 255], dtype=np.uint8)
    )

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

    contours, _ = cv2.findContours(
        mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    detections = []

    for contour in contours:
        area = cv2.contourArea(contour)

        if not 4000 <= area <= 55000:
            continue

        x, y, bw, bh = cv2.boundingRect(contour)
        aspect = bw / max(bh, 1)

        if not 0.75 <= aspect <= 1.85:
            continue

        perimeter = cv2.arcLength(contour, True)
        circularity = (
            4 * np.pi * area / (perimeter ** 2)
            if perimeter > 0 else 0
        )

        if circularity < 0.50:
            continue

        detections.append({
            "bbox": (x + x1, y + y1, bw, bh),
            "center": (x + x1 + bw // 2, y + y1 + bh // 2),
            "area": round(area),
            "circularity": round(circularity, 2)
        })

    return detections, mask


cap = cv2.VideoCapture(VIDEO_PATH)
times = [12, 20, 25, 30, 35, 40]

fig, axes = plt.subplots(len(times), 2, figsize=(14, 24))

for row, seconds in enumerate(times):
    cap.set(cv2.CAP_PROP_POS_MSEC, seconds * 1000)
    ok, frame = cap.read()

    if not ok:
        continue

    detections, mask = detect_eggs_v2(frame)
    preview = frame.copy()

    for i, d in enumerate(detections, 1):
        x, y, bw, bh = d["bbox"]
        cv2.rectangle(preview, (x, y), (x+bw, y+bh), (0, 255, 0), 3)
        cv2.putText(
            preview, f"Candidate {i}", (x, max(y-10, 30)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2
        )

    axes[row, 0].imshow(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB))
    axes[row, 0].set_title(f"{seconds}s — {len(detections)} candidates")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(mask, cmap="gray")
    axes[row, 1].set_title("Threshold mask")
    axes[row, 1].axis("off")

    print(f"{seconds}s:", detections)

cap.release()
plt.tight_layout()
plt.show()

In [ ]:
import os

print("Checking video folder:")
print(os.path.exists("/content/egg_video"))

if os.path.exists("/content/egg_video"):
    print(os.listdir("/content/egg_video"))

print("\nSearching /content for video files:")

for root, dirs, files in os.walk("/content"):
    dirs[:] = [d for d in dirs if d not in [
        "node_modules", ".cache", ".git"
    ]]
    for file in files:
        if file.lower().endswith((".mp4", ".mkv", ".avi", ".mov", ".webm")):
            print(os.path.join(root, file))

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    dirs[:] = [d for d in dirs if d not in [
        ".git", "node_modules"
    ]]
    for file in files:
        if file.lower().endswith((".mp4", ".mkv", ".avi", ".mov", ".webm")):
            print(os.path.join(root, file))

In [ ]:
!pip -q install --upgrade "yt-dlp[default]"

!curl -fsSL https://deno.land/install.sh | sh

import os
os.environ["PATH"] = os.path.expanduser("~/.deno/bin") + ":" + os.environ["PATH"]

!deno --version
!yt-dlp --version

In [ ]:
import os
import subprocess

VIDEO_URL = "https://www.youtube.com/watch?v=Y4JyV94y2_A"
VIDEO_DIR = "/content/egg_video"

os.makedirs(VIDEO_DIR, exist_ok=True)

result = subprocess.run(
    [
        "yt-dlp",
        "--js-runtimes", "deno",
        "--remote-components", "ejs:npm",
        "--no-playlist",
        "-f", "best[ext=mp4]/best",
        "-o", f"{VIDEO_DIR}/input_video.%(ext)s",
        VIDEO_URL
    ],
    capture_output=True,
    text=True
)

print(result.stdout[-4000:])
print(result.stderr[-2000:])
print("Return code:", result.returncode)

print("\nDownloaded files:")
print(os.listdir(VIDEO_DIR))

In [ ]:
import os
import shutil

from google.colab import drive
drive.mount("/content/drive")

SOURCE = "/content/egg_video/input_video.mp4"
DRIVE_DIR = "/content/drive/MyDrive/egg_task/video"

os.makedirs(DRIVE_DIR, exist_ok=True)

if not os.path.exists(SOURCE):
    raise FileNotFoundError(f"Video not found: {SOURCE}")

DESTINATION = os.path.join(DRIVE_DIR, "input_video.mp4")
shutil.copy2(SOURCE, DESTINATION)

print("Video saved to:", DESTINATION)
print("File size (MB):", round(os.path.getsize(DESTINATION) / (1024 * 1024), 2))

In [ ]:
import cv2
import os

VIDEO_PATH = "/content/drive/MyDrive/egg_task/video/input_video.mp4"

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise RuntimeError("Could not open the saved video.")

print("Video found:", os.path.exists(VIDEO_PATH))
print("Resolution:",
      int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
      "x",
      int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
print("FPS:", cap.get(cv2.CAP_PROP_FPS))
print("Duration (seconds):",
      round(cap.get(cv2.CAP_PROP_FRAME_COUNT) /
            cap.get(cv2.CAP_PROP_FPS), 2))

cap.release()

In [ ]:
VIDEO_PATH = "/content/drive/MyDrive/egg_task/video/input_video.mp4"

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

VIDEO_PATH = "/content/drive/MyDrive/egg_task/video/input_video.mp4"

if not os.path.exists(VIDEO_PATH):
    raise FileNotFoundError(
        "Video not found. Run the download and Google Drive backup steps first."
    )

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise RuntimeError("OpenCV could not open the video.")

fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print("Video:", VIDEO_PATH)
print("Resolution:", width, "x", height)
print("FPS:", fps)
print("Frames:", total_frames)
print("Duration:", round(total_frames / fps, 2), "seconds")

cap.set(cv2.CAP_PROP_POS_MSEC, 25_000)
ret, frame = cap.read()
cap.release()

if ret:
    plt.figure(figsize=(12, 6))
    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.title("Input video at 25 seconds")
    plt.axis("off")
    plt.show()
else:
    print("Could not read the sample frame.")

In [ ]:
import os
import cv2
import csv
import json
import numpy as np
from collections import defaultdict

VIDEO_PATH = "/content/drive/MyDrive/egg_task/video/input_video.mp4"
OUTPUT_DIR = "/content/drive/MyDrive/egg_task/video/tracking"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_VIDEO = os.path.join(OUTPUT_DIR, "egg_tracking.mp4")
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "egg_tracks.csv")
OUTPUT_SUMMARY = os.path.join(OUTPUT_DIR, "tracking_summary.json")

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise RuntimeError(f"Cannot open video: {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS)
fps = fps if fps and fps > 0 else 24.0

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Work on the egg/tabletop portion of the video.
# These bounds exclude most of the person's shirt and the video title card.
X1 = int(frame_width * 0.03)
X2 = int(frame_width * 0.94)
Y1 = int(frame_height * 0.40)
Y2 = int(frame_height * 0.88)

MIN_AREA = 2500
MAX_AREA = 50000
MAX_MATCH_DISTANCE = 100
MAX_MISSED_FRAMES = int(fps * 1.5)

kernel = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE, (5, 5)
)

# Each tracker entry stores centroid, box, missed-frame count,
# and the number of times the track has been detected.
tracks = {}
next_track_id = 1
rows = []
track_detection_counts = defaultdict(int)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(
    OUTPUT_VIDEO, fourcc, fps, (frame_width, frame_height)
)

if not writer.isOpened():
    cap.release()
    raise RuntimeError("Could not create the output video.")

frame_index = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_index += 1
    timestamp = (frame_index - 1) / fps

    # Only process frames where eggs are expected to be visible.
    # Intro/outro cards are still preserved in the output video.
    active_scene = 10 <= timestamp <= 43

    detections = []

    if active_scene:
        roi = frame[Y1:Y2, X1:X2]
        hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

        mask = cv2.inRange(
            hsv,
            np.array([0, 0, 155], dtype=np.uint8),
            np.array([180, 105, 255], dtype=np.uint8)
        )

        mask = cv2.morphologyEx(
            mask, cv2.MORPH_OPEN, kernel
        )
        mask = cv2.morphologyEx(
            mask, cv2.MORPH_CLOSE, kernel
        )

        contours, _ = cv2.findContours(
            mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )

        for contour in contours:
            area = cv2.contourArea(contour)

            if not MIN_AREA <= area <= MAX_AREA:
                continue

            perimeter = cv2.arcLength(contour, True)
            if perimeter <= 0:
                continue

            circularity = 4 * np.pi * area / (perimeter * perimeter)

            x, y, w, h = cv2.boundingRect(contour)
            aspect_ratio = w / max(h, 1)

            if circularity < 0.35:
                continue

            if not 0.65 <= aspect_ratio <= 1.8:
                continue

            # Ignore contours touching the top ROI boundary,
            # where the person's clothing can enter the mask.
            if y <= 2:
                continue

            x += X1
            y += Y1

            cx = x + w // 2
            cy = y + h // 2

            detections.append({
                "bbox": (x, y, w, h),
                "centroid": (cx, cy),
                "area": float(area),
                "circularity": float(circularity)
            })

    # Mark tracks as not detected in this frame.
    for track in tracks.values():
        track["missed"] += 1

    # Match detections to existing tracks by centroid distance.
    # Greedy nearest matching is suitable as a simple baseline.
    candidate_matches = []

    for track_id, track in tracks.items():
        if track["missed"] > MAX_MISSED_FRAMES:
            continue

        old_cx, old_cy = track["centroid"]

        for det_idx, det in enumerate(detections):
            new_cx, new_cy = det["centroid"]
            distance = np.hypot(
                new_cx - old_cx, new_cy - old_cy
            )

            if distance <= MAX_MATCH_DISTANCE:
                candidate_matches.append(
                    (distance, track_id, det_idx)
                )

    candidate_matches.sort(key=lambda item: item[0])

    used_tracks = set()
    used_detections = set()
    assignments = {}

    for distance, track_id, det_idx in candidate_matches:
        if track_id in used_tracks or det_idx in used_detections:
            continue

        used_tracks.add(track_id)
        used_detections.add(det_idx)
        assignments[det_idx] = track_id

    # Update matched tracks and create tracks for unmatched detections.
    for det_idx, det in enumerate(detections):
        if det_idx in assignments:
            track_id = assignments[det_idx]
            tracks[track_id]["centroid"] = det["centroid"]
            tracks[track_id]["bbox"] = det["bbox"]
            tracks[track_id]["missed"] = 0
            tracks[track_id]["hits"] += 1
        else:
            track_id = next_track_id
            next_track_id += 1

            tracks[track_id] = {
                "centroid": det["centroid"],
                "bbox": det["bbox"],
                "missed": 0,
                "hits": 1,
                "first_seen": timestamp
            }

            assignments[det_idx] = track_id

    # Draw visible detections and write per-frame records.
    for det_idx, det in enumerate(detections):
        track_id = assignments[det_idx]
        x, y, w, h = det["bbox"]
        cx, cy = det["centroid"]

        track_detection_counts[track_id] += 1

        cv2.rectangle(
            frame, (x, y), (x + w, y + h), (0, 255, 0), 2
        )
        cv2.putText(
            frame,
            f"Egg {track_id}",
            (x, max(y - 8, 25)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2
        )

        rows.append({
            "frame": frame_index,
            "timestamp_seconds": round(timestamp, 3),
            "track_id": track_id,
            "center_x": cx,
            "center_y": cy,
            "x": x,
            "y": y,
            "width": w,
            "height": h,
            "area": round(det["area"], 2),
            "circularity": round(det["circularity"], 3)
        })

    # Remove tracks missing for too long.
    expired = [
        track_id for track_id, track in tracks.items()
        if track["missed"] > MAX_MISSED_FRAMES
    ]

    for track_id in expired:
        del tracks[track_id]

    # Inform viewers about the processing stage.
    cv2.putText(
        frame,
        "Egg tracking | Classical CV",
        (30, 45),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 255),
        2
    )

    writer.write(frame)

cap.release()
writer.release()

# Export detection history.
with open(OUTPUT_CSV, "w", newline="") as f:
    fieldnames = [
        "frame", "timestamp_seconds", "track_id",
        "center_x", "center_y", "x", "y",
        "width", "height", "area", "circularity"
    ]
    csv_writer = csv.DictWriter(f, fieldnames=fieldnames)
    csv_writer.writeheader()
    csv_writer.writerows(rows)

summary = {
    "input_video": VIDEO_PATH,
    "output_video": OUTPUT_VIDEO,
    "fps": fps,
    "total_frames": total_frames,
    "duration_seconds": round(total_frames / fps, 2),
    "detections_recorded": len(rows),
    "unique_track_ids": len(set(row["track_id"] for row in rows)),
    "tracking_method": "HSV thresholding, contour filtering, centroid matching",
    "note": (
        "Track IDs are approximate. Occlusions, reflections, "
        "and false detections may cause identity switches."
    )
}

with open(OUTPUT_SUMMARY, "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("\nSaved video:", OUTPUT_VIDEO)
print("Saved CSV:", OUTPUT_CSV)
print("Saved summary:", OUTPUT_SUMMARY)

In [ ]:
import cv2
import matplotlib.pyplot as plt

OUTPUT_VIDEO = "/content/drive/MyDrive/egg_task/video/tracking/egg_tracking.mp4"

cap = cv2.VideoCapture(OUTPUT_VIDEO)

if not cap.isOpened():
    raise RuntimeError("Could not open the tracking output video.")

fps = cap.get(cv2.CAP_PROP_FPS)
cap.set(cv2.CAP_PROP_POS_MSEC, 25_000)

ret, frame = cap.read()
cap.release()

if ret:
    plt.figure(figsize=(14, 8))
    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.title("Egg Tracking Output — 25 seconds")
    plt.axis("off")
    plt.show()
else:
    print("Could not read the preview frame.")

In [ ]:
import os

paths = [
    "/content/egg_video/input_video.mp4",
    "/content/egg_video/tracking/egg_tracking_preview_h264.mp4",
    "/content/drive/MyDrive/egg_task"
]

for path in paths:
    print(f"\n{path}")
    print("Exists:", os.path.exists(path))
    if os.path.exists(path) and os.path.isfile(path):
        print("Size (MB):", round(os.path.getsize(path) / (1024 * 1024), 2))
    elif os.path.exists(path):
        print("Contents:", os.listdir(path)[:15])

In [ ]:
import os
import cv2
import csv
import json
import numpy as np
from scipy.optimize import linear_sum_assignment
from IPython.display import Video, display

# ---------- Paths ----------
VIDEO_PATH = "/content/egg_video/input_video.mp4"
OUTPUT_DIR = "/content/drive/MyDrive/egg_task/video/final_tracking"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_VIDEO = os.path.join(OUTPUT_DIR, "egg_tracking_final.mp4")
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "egg_tracks_final.csv")
OUTPUT_SUMMARY = os.path.join(OUTPUT_DIR, "tracking_summary.json")

if not os.path.exists(VIDEO_PATH):
    raise FileNotFoundError(
        f"Video not found: {VIDEO_PATH}. Check that the Colab runtime is connected."
    )

# ---------- Video settings ----------
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise RuntimeError("OpenCV could not open the input video.")

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print("Video:", width, "x", height)
print("FPS:", fps)
print("Duration:", round(total_frames / fps, 2), "seconds")

# Track eggs during the main tabletop sequence only.
START_TIME = 10
END_TIME = 44

START_FRAME = int(START_TIME * fps)
END_FRAME = min(int(END_TIME * fps), total_frames)

# Tabletop region: exclude the person above and the bottom watermark.
X1 = int(width * 0.03)
X2 = int(width * 0.94)
Y1 = int(height * 0.40)
Y2 = int(height * 0.82)

# ---------- Detection settings ----------
MIN_AREA = 10000
MAX_AREA = 85000
MIN_CIRCULARITY = 0.48
MIN_SOLIDITY = 0.78

# ---------- Tracking settings ----------
MAX_MATCH_DISTANCE = 150
MAX_MISSED_FRAMES = 12

tracks = {}
next_id = 1
csv_rows = []

kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (width, height))

if not writer.isOpened():
    cap.release()
    raise RuntimeError("Could not create the output video.")

frame_index = 0
frames_processed = 0
frames_with_detections = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    annotated = frame.copy()
    detections = []

    # Only search for eggs in the selected video interval.
    if START_FRAME <= frame_index < END_FRAME:

        # Crop to tabletop.
        roi = frame[Y1:Y2, X1:X2]
        hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

        # White objects have high brightness and relatively low saturation.
        mask = cv2.inRange(
            hsv,
            np.array([0, 0, 155], dtype=np.uint8),
            np.array([180, 105, 255], dtype=np.uint8)
        )

        mask = cv2.morphologyEx(
            mask, cv2.MORPH_OPEN, kernel_open
        )

        contours, _ = cv2.findContours(
            mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )

        for contour in contours:
            area = cv2.contourArea(contour)

            if area < MIN_AREA or area > MAX_AREA:
                continue

            x, y, w, h = cv2.boundingRect(contour)

            if h == 0 or w == 0:
                continue

            aspect = w / float(h)
            perimeter = cv2.arcLength(contour, True)

            if perimeter <= 0:
                continue

            circularity = 4 * np.pi * area / (perimeter * perimeter)
            hull = cv2.convexHull(contour)
            hull_area = cv2.contourArea(hull)

            if hull_area <= 0:
                continue

            solidity = area / hull_area

            # Reject thin text fragments and irregular objects.
            if not (0.60 <= aspect <= 1.80):
                continue

            if circularity < MIN_CIRCULARITY:
                continue

            if solidity < MIN_SOLIDITY:
                continue

            # Convert ROI coordinates back to full-frame coordinates.
            full_x = x + X1
            full_y = y + Y1
            center_x = full_x + w // 2
            center_y = full_y + h // 2

            detections.append({
                "bbox": (full_x, full_y, w, h),
                "center": (center_x, center_y),
                "area": float(area),
                "circularity": float(circularity)
            })

        # Update the age of tracks before matching.
        for track in tracks.values():
            track["missed"] += 1

        # Match existing tracks to current detections.
        track_ids = list(tracks.keys())

        if track_ids and detections:
            cost = np.zeros((len(track_ids), len(detections)), dtype=float)

            for i, tid in enumerate(track_ids):
                old_x, old_y = tracks[tid]["center"]

                for j, detection in enumerate(detections):
                    new_x, new_y = detection["center"]

                    cost[i, j] = np.hypot(
                        new_x - old_x,
                        new_y - old_y
                    )

            row_indices, col_indices = linear_sum_assignment(cost)

            matched_tracks = set()
            matched_detections = set()

            for i, j in zip(row_indices, col_indices):
                if cost[i, j] > MAX_MATCH_DISTANCE:
                    continue

                tid = track_ids[i]
                detection = detections[j]

                tracks[tid]["center"] = detection["center"]
                tracks[tid]["bbox"] = detection["bbox"]
                tracks[tid]["missed"] = 0
                tracks[tid]["hits"] += 1

                matched_tracks.add(tid)
                matched_detections.add(j)

            # Create new tracks for unmatched detections.
            for j, detection in enumerate(detections):
                if j in matched_detections:
                    continue

                tid = next_id
                next_id += 1

                tracks[tid] = {
                    "center": detection["center"],
                    "bbox": detection["bbox"],
                    "missed": 0,
                    "hits": 1
                }

        elif detections:
            for detection in detections:
                tid = next_id
                next_id += 1

                tracks[tid] = {
                    "center": detection["center"],
                    "bbox": detection["bbox"],
                    "missed": 0,
                    "hits": 1
                }

        # Remove tracks that have been missing for too long.
        expired = [
            tid for tid, track in tracks.items()
            if track["missed"] > MAX_MISSED_FRAMES
        ]

        for tid in expired:
            del tracks[tid]

        # Draw only tracks detected in the current frame.
        for tid, track in tracks.items():
            if track["missed"] != 0:
                continue

            x, y, w, h = track["bbox"]
            cx, cy = track["center"]

            cv2.rectangle(
                annotated,
                (x, y),
                (x + w, y + h),
                (0, 255, 0),
                3
            )

            cv2.putText(
                annotated,
                f"Egg {tid}",
                (x, max(25, y - 10)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 0),
                2
            )

            csv_rows.append({
                "frame": frame_index,
                "time_seconds": round(frame_index / fps, 3),
                "egg_id": tid,
                "center_x": cx,
                "center_y": cy,
                "x": x,
                "y": y,
                "width": w,
                "height": h
            })

        if detections:
            frames_with_detections += 1

        # Mark the tracking interval and detection count.
        cv2.putText(
            annotated,
            f"Classical CV Egg Tracking | Time: {frame_index / fps:.1f}s",
            (30, 45),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.9,
            (0, 255, 255),
            2
        )

        cv2.putText(
            annotated,
            f"Current detections: {len(detections)}",
            (30, 80),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 255),
            2
        )

    writer.write(annotated)

    frame_index += 1
    frames_processed += 1

    if frames_processed % 200 == 0:
        print("Processed frames:", frames_processed)

cap.release()
writer.release()

# ---------- Save tracking CSV ----------
with open(OUTPUT_CSV, "w", newline="") as f:
    fieldnames = [
        "frame", "time_seconds", "egg_id",
        "center_x", "center_y", "x", "y", "width", "height"
    ]

    csv_writer = csv.DictWriter(f, fieldnames=fieldnames)
    csv_writer.writeheader()
    csv_writer.writerows(csv_rows)

# ---------- Save summary ----------
summary = {
    "input_video": VIDEO_PATH,
    "output_video": OUTPUT_VIDEO,
    "tracking_csv": OUTPUT_CSV,
    "tracking_interval_seconds": [START_TIME, END_TIME],
    "video_fps": fps,
    "frames_processed": frames_processed,
    "frames_with_detections": frames_with_detections,
    "detection_records": len(csv_rows),
    "unique_track_ids": sorted(set(
        row["egg_id"] for row in csv_rows
    )),
    "unique_track_count": len(set(
        row["egg_id"] for row in csv_rows
    )),
    "method": "HSV thresholding, contour shape filtering, Hungarian assignment",
    "note": "Track IDs are provisional and require visual validation."
}

with open(OUTPUT_SUMMARY, "w") as f:
    json.dump(summary, f, indent=2)

print("\nTracking completed.")
print(json.dumps(summary, indent=2))
print("\nSaved files:")
print(OUTPUT_VIDEO)
print(OUTPUT_CSV)
print(OUTPUT_SUMMARY)

In [ ]:
from IPython.display import Video, display
import os

video_path = "/content/drive/MyDrive/egg_task/video/final_tracking/egg_tracking_final.mp4"

print("Video exists:", os.path.exists(video_path))

if os.path.exists(video_path):
    print("Size (MB):", round(os.path.getsize(video_path) / (1024 * 1024), 2))
    display(Video(video_path, embed=True, width=800))

In [ ]:
 import cv2
import os

video_path = "/content/drive/MyDrive/egg_task/video/final_tracking/egg_tracking_final.mp4"

cap = cv2.VideoCapture(video_path)

print("File exists:", os.path.exists(video_path))
print("Video opens:", cap.isOpened())

if cap.isOpened():
    print("Width:", int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)))
    print("Height:", int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
    print("FPS:", cap.get(cv2.CAP_PROP_FPS))
    print("Frames:", int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))

    cap.set(cv2.CAP_PROP_POS_MSEC, 25000)
    success, frame = cap.read()

    if success:
        from google.colab.patches import cv2_imshow
        cv2_imshow(frame)
    else:
        print("Could not read the frame at 25 seconds.")

cap.release()

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os

root = "/content/drive/MyDrive/egg_task"

print("Project folder exists:", os.path.exists(root))

for folder, dirs, files in os.walk(root):
    dirs[:] = [d for d in dirs if d not in [".git", "node_modules"]]

    for name in files:
        if (
            name.endswith((".mp4", ".csv", ".json"))
            and ("egg" in name.lower() or "track" in name.lower())
        ):
            path = os.path.join(folder, name)
            print(path, "|", round(os.path.getsize(path) / (1024 * 1024), 2), "MB")

In [ ]:
import os
import pandas as pd
import json

VIDEO_DIR = "/content/drive/MyDrive/egg_task/video"
TRACK_DIR = os.path.join(VIDEO_DIR, "tracking")

print("Video folder exists:", os.path.isdir(VIDEO_DIR))
print("Tracking folder exists:", os.path.isdir(TRACK_DIR))

print("\nFiles in tracking folder:")
if os.path.isdir(TRACK_DIR):
    for name in sorted(os.listdir(TRACK_DIR)):
        path = os.path.join(TRACK_DIR, name)
        print(f"{name} — {os.path.getsize(path) / (1024*1024):.2f} MB")

csv_path = os.path.join(TRACK_DIR, "egg_tracks.csv")
summary_path = os.path.join(TRACK_DIR, "tracking_summary.json")

if os.path.isfile(csv_path):
    tracks = pd.read_csv(csv_path)
    print("\nTracking CSV loaded successfully.")
    print("Rows:", len(tracks))
    print("Columns:", tracks.columns.tolist())
    display(tracks.head())
else:
    print("\nNo egg_tracks.csv found in the tracking folder.")

if os.path.isfile(summary_path):
    with open(summary_path, "r") as f:
        summary = json.load(f)
    print("\nTracking summary:")
    print(json.dumps(summary, indent=2))

In [ ]:
import os
import cv2
from google.colab.patches import cv2_imshow
from IPython.display import Video, display

VIDEO_PATH = "/content/drive/MyDrive/egg_task/video/tracking/egg_tracking.mp4"

if not os.path.exists(VIDEO_PATH):
    raise FileNotFoundError(f"Video not found: {VIDEO_PATH}")

cap = cv2.VideoCapture(VIDEO_PATH)

print("Video opened:", cap.isOpened())
print("FPS:", cap.get(cv2.CAP_PROP_FPS))
print("Frame count:", int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))

# Inspect frames at 12, 20, 25, 30, and 40 seconds
fps = cap.get(cv2.CAP_PROP_FPS)

for seconds in [12, 20, 25, 30, 40]:
    cap.set(cv2.CAP_PROP_POS_MSEC, seconds * 1000)
    success, frame = cap.read()

    if success:
        print(f"Tracking output at {seconds} seconds")
        cv2_imshow(cv2.resize(frame, (960, 540)))

cap.release()

In [ ]:
import os
import cv2
import json
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from IPython.display import Video, display

# Input and output paths
INPUT_VIDEO = "/content/drive/MyDrive/egg_task/video/input_video.mp4"
OUTPUT_DIR = "/content/drive/MyDrive/egg_task/video/improved_tracking"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_VIDEO = os.path.join(OUTPUT_DIR, "egg_tracking_improved.mp4")
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "egg_tracks_improved.csv")
OUTPUT_JSON = os.path.join(OUTPUT_DIR, "tracking_summary.json")

if not os.path.exists(INPUT_VIDEO):
    raise FileNotFoundError(f"Video not found: {INPUT_VIDEO}")

cap = cv2.VideoCapture(INPUT_VIDEO)

if not cap.isOpened():
    raise RuntimeError("Could not open input video.")

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print("Video:", width, "x", height)
print("FPS:", fps)
print("Frames:", total_frames)
print("Duration:", round(total_frames / fps, 2), "seconds")

# H.264 output is easier to preview in Colab.
writer = cv2.VideoWriter(
    OUTPUT_VIDEO,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

if not writer.isOpened():
    cap.release()
    raise RuntimeError("Could not create output video.")

# Tracking state
tracks = {}
next_track_id = 1
records = []
frame_number = 0

# Track matching parameters
MAX_DISTANCE = 120
MAX_MISSED_FRAMES = 12

# Detect eggs only in the tabletop area.
# The bounds are normalized to the original frame dimensions.
X1 = int(width * 0.03)
X2 = int(width * 0.97)
Y1 = int(height * 0.39)
Y2 = int(height * 0.86)

kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

while True:
    success, frame = cap.read()
    if not success:
        break

    frame_number += 1
    timestamp = (frame_number - 1) / fps

    annotated = frame.copy()
    roi = frame[Y1:Y2, X1:X2]

    # White eggs have high brightness and relatively low saturation.
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(
        hsv,
        np.array([0, 0, 155], dtype=np.uint8),
        np.array([180, 100, 255], dtype=np.uint8)
    )

    # Remove isolated noise.
    mask = cv2.morphologyEx(
        mask, cv2.MORPH_OPEN, kernel, iterations=1
    )

    contours, _ = cv2.findContours(
        mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    detections = []

    for contour in contours:
        area = cv2.contourArea(contour)

        # Reject tiny highlights and huge merged regions.
        if not 5000 <= area <= 65000:
            continue

        x, y, w, h = cv2.boundingRect(contour)

        if h == 0 or w == 0:
            continue

        aspect_ratio = w / h
        perimeter = cv2.arcLength(contour, True)

        if perimeter == 0:
            continue

        circularity = 4 * np.pi * area / (perimeter * perimeter)

        # Allow oval eggs, but reject very elongated objects.
        if not 0.55 <= aspect_ratio <= 2.0:
            continue

        if circularity < 0.35:
            continue

        # Convert ROI coordinates to original frame coordinates.
        x += X1
        y += Y1

        detections.append({
            "x": x,
            "y": y,
            "w": w,
            "h": h,
            "cx": x + w // 2,
            "cy": y + h // 2,
            "area": float(area),
            "circularity": float(circularity)
        })

    # Match detections to previous tracks by centroid distance.
    track_ids = list(tracks.keys())
    matched_tracks = set()
    matched_detections = set()

    if track_ids and detections:
        cost = np.zeros((len(track_ids), len(detections)), dtype=float)

        for i, tid in enumerate(track_ids):
            old = tracks[tid]

            for j, det in enumerate(detections):
                distance = np.hypot(
                    old["cx"] - det["cx"],
                    old["cy"] - det["cy"]
                )
                cost[i, j] = distance

        rows, cols = linear_sum_assignment(cost)

        for i, j in zip(rows, cols):
            if cost[i, j] > MAX_DISTANCE:
                continue

            tid = track_ids[i]
            det = detections[j]

            tracks[tid].update({
                "cx": det["cx"],
                "cy": det["cy"],
                "x": det["x"],
                "y": det["y"],
                "w": det["w"],
                "h": det["h"],
                "area": det["area"],
                "circularity": det["circularity"],
                "last_seen": frame_number,
                "missed": 0
            })

            det["track_id"] = tid
            matched_tracks.add(tid)
            matched_detections.add(j)

    # Create new tracks for unmatched detections.
    for j, det in enumerate(detections):
        if j in matched_detections:
            continue

        tid = next_track_id
        next_track_id += 1

        tracks[tid] = {
            "cx": det["cx"],
            "cy": det["cy"],
            "x": det["x"],
            "y": det["y"],
            "w": det["w"],
            "h": det["h"],
            "area": det["area"],
            "circularity": det["circularity"],
            "last_seen": frame_number,
            "missed": 0
        }

        det["track_id"] = tid

    # Age tracks not detected in this frame.
    for tid in list(tracks.keys()):
        if tid not in matched_tracks:
            # A newly created track should not be aged immediately.
            if tracks[tid]["last_seen"] != frame_number:
                tracks[tid]["missed"] += 1

        if tracks[tid]["missed"] > MAX_MISSED_FRAMES:
            del tracks[tid]

    # Draw the tabletop ROI boundary for debugging.
    cv2.rectangle(
        annotated, (X1, Y1), (X2, Y2), (255, 180, 0), 2
    )

    # Draw detected eggs and save frame-level observations.
    for det in detections:
        tid = det["track_id"]

        x, y = det["x"], det["y"]
        w, h = det["w"], det["h"]

        cv2.rectangle(
            annotated, (x, y), (x + w, y + h), (0, 255, 0), 3
        )

        cv2.putText(
            annotated,
            f"Egg {tid}",
            (x, max(25, y - 8)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 0),
            2
        )

        records.append({
            "frame": frame_number,
            "timestamp_seconds": round(timestamp, 3),
            "track_id": tid,
            "center_x": det["cx"],
            "center_y": det["cy"],
            "x": x,
            "y": y,
            "width": w,
            "height": h,
            "area": round(det["area"], 2),
            "circularity": round(det["circularity"], 3)
        })

    cv2.putText(
        annotated,
        f"Egg detections: {len(detections)}",
        (30, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 255),
        2
    )

    writer.write(annotated)

cap.release()
writer.release()

# Save tracking observations.
df = pd.DataFrame(records)
df.to_csv(OUTPUT_CSV, index=False)

summary = {
    "input_video": INPUT_VIDEO,
    "output_video": OUTPUT_VIDEO,
    "fps": fps,
    "total_frames": total_frames,
    "duration_seconds": round(total_frames / fps, 2),
    "detections_recorded": len(df),
    "unique_track_ids": int(df["track_id"].nunique()) if not df.empty else 0,
    "tracking_method": "HSV thresholding, contour filtering, Hungarian centroid matching",
    "limitations": (
        "Classical thresholding can still confuse eggs with hands, "
        "reflections, and other bright objects. Track IDs may switch "
        "during occlusion or when detections disappear."
    )
}

with open(OUTPUT_JSON, "w") as f:
    json.dump(summary, f, indent=2)

print("\nTRACKING SUMMARY")
print(json.dumps(summary, indent=2))
print("\nSaved video:", OUTPUT_VIDEO)
print("Saved CSV:", OUTPUT_CSV)
print("Saved summary:", OUTPUT_JSON)

In [ ]:
from IPython.display import Video, display

display(Video(
    "/content/drive/MyDrive/egg_task/video/improved_tracking/egg_tracking_improved.mp4",
    embed=True,
    width=800
))

In [ ]:
import os
import cv2

video_path = "/content/drive/MyDrive/egg_task/video/improved_tracking/egg_tracking_improved.mp4"

print("File exists:", os.path.exists(video_path))

if os.path.exists(video_path):
    print("File size (MB):", round(os.path.getsize(video_path) / (1024 * 1024), 2))

    cap = cv2.VideoCapture(video_path)
    print("Video opened:", cap.isOpened())
    print("Frame count:", int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
    print("FPS:", cap.get(cv2.CAP_PROP_FPS))
    cap.release()

In [ ]:
import cv2
import matplotlib.pyplot as plt

video_path = "/content/drive/MyDrive/egg_task/video/improved_tracking/egg_tracking_improved.mp4"

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Cannot open video. Check the file path and output from Step 1.")
else:
    fps = cap.get(cv2.CAP_PROP_FPS)

    for seconds in [12, 25, 40]:
        cap.set(cv2.CAP_PROP_POS_MSEC, seconds * 1000)
        success, frame = cap.read()

        if success:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            plt.figure(figsize=(12, 6))
            plt.imshow(frame)
            plt.title(f"Egg tracking at {seconds} seconds")
            plt.axis("off")
            plt.show()
        else:
            print(f"Could not read frame at {seconds} seconds.")

    cap.release()

In [ ]:
from google.colab import files

files.download(
    "/content/drive/MyDrive/egg_task/video/improved_tracking/egg_tracking_improved.mp4"
)

In [ ]:
import os

root = "/content/drive/MyDrive/egg_task"

for folder, dirs, files in os.walk(root):
    dirs[:] = [d for d in dirs if d not in [".git", "__pycache__"]]
    for file in files:
        if file.endswith(".py"):
            path = os.path.join(folder, file)
            try:
                with open(path, "r", encoding="utf-8") as f:
                    lines = f.readlines()
                matches = [
                    (i + 1, line.strip())
                    for i, line in enumerate(lines)
                    if "cv2.rectangle" in line
                ]
                if matches:
                    print(f"\nFILE: {path}")
                    for line_no, line in matches:
                        print(f"Line {line_no}: {line}")
            except Exception:
                pass

In [ ]:
import os

root = "/content/drive/MyDrive/egg_task"

for folder, dirs, files in os.walk(root):
    dirs[:] = [d for d in dirs if d not in [".git", "__pycache__"]]

    for filename in files:
        if filename.endswith(".py"):
            path = os.path.join(folder, filename)

            try:
                with open(path, "r", encoding="utf-8") as f:
                    content = f.read()

                if (
                    "egg_tracking_improved" in content
                    or "cv2.VideoWriter" in content
                    or "cv2.rectangle" in content
                ):
                    print("\nFILE:", path)

                    for number, line in enumerate(content.splitlines(), 1):
                        if any(term in line for term in [
                            "egg_tracking_improved",
                            "cv2.VideoWriter",
                            "cv2.rectangle",
                            "ROI",
                            "region"
                        ]):
                            print(f"{number}: {line.strip()}")

            except Exception as e:
                print("Could not read:", path, e)

In [ ]:
import os
import cv2
import subprocess

# Existing tracking video
INPUT_VIDEO = "/content/drive/MyDrive/egg_task/video/tracking/egg_tracking.mp4"

# New output video
OUTPUT_VIDEO = "/content/drive/MyDrive/egg_task/video/tracking/egg_tracking_final.mp4"

print("Input exists:", os.path.exists(INPUT_VIDEO))

if not os.path.exists(INPUT_VIDEO):
    print("Searching for available tracking videos...")
    for root, dirs, files in os.walk("/content/drive/MyDrive/egg_task"):
        for file in files:
            if file.endswith(".mp4"):
                print(os.path.join(root, file))
else:
    cap = cv2.VideoCapture(INPUT_VIDEO)
    print("Video opened:", cap.isOpened())
    print("Resolution:", int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
          "x", int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
    cap.release()

In [ ]:
import os

ROOT = "/content/drive/MyDrive/egg_task"

print("SEARCHING FOR IMPROVED VIDEOS AND TRACKING FILES\n")

for folder, dirs, files in os.walk(ROOT):
    dirs[:] = [d for d in dirs if d not in [".git", "__pycache__"]]

    for name in files:
        if (
            name.endswith((".mp4", ".csv", ".json", ".py"))
            and any(word in name.lower() for word in [
                "improved", "tracking", "track", "egg"
            ])
        ):
            path = os.path.join(folder, name)
            size_mb = os.path.getsize(path) / (1024 * 1024)
            print(f"{path}  ({size_mb:.2f} MB)")

In [ ]:
import os
import pandas as pd
import json

BASE = "/content/drive/MyDrive/egg_task/video"

VIDEO = f"{BASE}/input_video.mp4"
if not os.path.exists(VIDEO):
    VIDEO = "/content/egg_video/input_video.mp4"

CSV = f"{BASE}/improved_tracking/egg_tracks_improved.csv"
SUMMARY = f"{BASE}/improved_tracking/tracking_summary.json"

print("Original video exists:", os.path.exists(VIDEO))
print("Improved tracking CSV exists:", os.path.exists(CSV))

if os.path.exists(CSV):
    df = pd.read_csv(CSV)

    print("\nRows:", len(df))
    print("Columns:", df.columns.tolist())

    if "track_id" in df.columns:
        print("Unique track IDs:", df["track_id"].nunique())
        print("\nObservations per track:")
        print(df.groupby("track_id").size().sort_values(ascending=False).head(15))

if os.path.exists(SUMMARY):
    with open(SUMMARY) as f:
        print("\nTracking summary:")
        print(json.dumps(json.load(f), indent=2))

In [ ]:
import os
import cv2
import pandas as pd
import numpy as np
import subprocess
from scipy.optimize import linear_sum_assignment
from collections import defaultdict

# --------------------------------------------------
# 1. File paths
# --------------------------------------------------
BASE = "/content/drive/MyDrive/egg_task/video"
INPUT_VIDEO = f"{BASE}/input_video.mp4"
INPUT_CSV = f"{BASE}/improved_tracking/egg_tracks_improved.csv"

OUTPUT_DIR = f"{BASE}/final_tracking"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TEMP_VIDEO = f"{OUTPUT_DIR}/egg_tracking_temp.mp4"
FINAL_VIDEO = f"{OUTPUT_DIR}/egg_tracking_final_v2.mp4"
OUTPUT_CSV = f"{OUTPUT_DIR}/egg_tracks_final_v2.csv"

assert os.path.exists(INPUT_VIDEO), f"Missing video: {INPUT_VIDEO}"
assert os.path.exists(INPUT_CSV), f"Missing CSV: {INPUT_CSV}"

# --------------------------------------------------
# 2. Load detections
# --------------------------------------------------
df = pd.read_csv(INPUT_CSV)

# Retain detections likely to represent eggs.
df = df[
    (df["area"] >= 15000) &
    (df["area"] <= 60000) &
    (df["circularity"] >= 0.55) &
    (df["width"] >= 100) &
    (df["width"] <= 320) &
    (df["height"] >= 80) &
    (df["height"] <= 280) &
    (df["center_y"] >= 400) &
    (df["center_y"] <= 880)
].copy()

df["aspect"] = df["width"] / df["height"]
df = df[(df["aspect"] >= 0.65) & (df["aspect"] <= 1.8)]

# Use at most two candidate detections per frame.
# Prefer contours with a plausible egg shape and size.
df["quality"] = (
    df["circularity"] -
    0.00001 * abs(df["area"] - 40000)
)

df = (
    df.sort_values("quality", ascending=False)
      .groupby("frame", group_keys=False)
      .head(2)
      .copy()
)

detections_by_frame = {
    int(frame): group.to_dict("records")
    for frame, group in df.groupby("frame")
}

print("Filtered detections:", len(df))
print("Frames with detections:", len(detections_by_frame))

# --------------------------------------------------
# 3. Open video and prepare output
# --------------------------------------------------
cap = cv2.VideoCapture(INPUT_VIDEO)

if not cap.isOpened():
    raise RuntimeError("Could not open the original video.")

fps = cap.get(cv2.CAP_PROP_FPS) or 24.0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

writer = cv2.VideoWriter(
    TEMP_VIDEO,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

if not writer.isOpened():
    cap.release()
    raise RuntimeError("Could not create temporary output video.")

# --------------------------------------------------
# 4. Centroid-based tracking
# --------------------------------------------------
active_tracks = {}
next_id = 1
records = []

# Maximum matching distance in pixels.
MAX_DISTANCE = 150

# Retain an ID through short periods of missed detection.
MAX_MISSED_FRAMES = 18

frame_index = 0

while True:
    ok, frame = cap.read()
    if not ok:
        break

    frame_number = frame_index + 1
    candidates = detections_by_frame.get(frame_number, [])

    # Remove tracks that have been missing too long.
    active_tracks = {
        tid: track
        for tid, track in active_tracks.items()
        if frame_index - track["last_seen"] <= MAX_MISSED_FRAMES
    }

    track_ids = list(active_tracks.keys())
    assigned = {}
    used_candidates = set()

    # Match existing tracks to current detections.
    if track_ids and candidates:
        cost = np.zeros((len(track_ids), len(candidates)))

        for i, tid in enumerate(track_ids):
            old_x, old_y = active_tracks[tid]["center"]

            for j, det in enumerate(candidates):
                dx = old_x - float(det["center_x"])
                dy = old_y - float(det["center_y"])
                cost[i, j] = np.hypot(dx, dy)

        rows, cols = linear_sum_assignment(cost)

        for i, j in zip(rows, cols):
            if cost[i, j] <= MAX_DISTANCE:
                tid = track_ids[i]
                assigned[j] = tid
                used_candidates.add(j)

    # Assign new IDs to unmatched detections.
    for j in range(len(candidates)):
        if j not in used_candidates:
            assigned[j] = next_id
            next_id += 1

    # Draw detections and update track positions.
    for j, det in enumerate(candidates):
        tid = assigned[j]

        x = int(det["x"])
        y = int(det["y"])
        w = int(det["width"])
        h = int(det["height"])
        cx = float(det["center_x"])
        cy = float(det["center_y"])

        active_tracks[tid] = {
            "center": (cx, cy),
            "last_seen": frame_index
        }

        cv2.rectangle(
            frame, (x, y), (x + w, y + h),
            (0, 255, 0), 3
        )

        cv2.putText(
            frame, f"Egg {tid}",
            (x, max(25, y - 8)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8, (0, 255, 0), 2
        )

        records.append({
            "frame": frame_number,
            "timestamp_seconds": round(frame_index / fps, 3),
            "track_id": tid,
            "center_x": cx,
            "center_y": cy,
            "x": x,
            "y": y,
            "width": w,
            "height": h,
            "area": det["area"],
            "circularity": det["circularity"]
        })

    # Deliberately do not draw the blue ROI rectangle.
    cv2.putText(
        frame, f"Egg detections: {len(candidates)}",
        (35, 55), cv2.FONT_HERSHEY_SIMPLEX,
        1.0, (0, 255, 255), 2
    )

    writer.write(frame)
    frame_index += 1

    if frame_index % 240 == 0:
        print(f"Processed {frame_index}/{total_frames} frames")

cap.release()
writer.release()

# --------------------------------------------------
# 5. Convert to broadly compatible H.264 MP4
# --------------------------------------------------
result = subprocess.run([
    "ffmpeg", "-y",
    "-i", TEMP_VIDEO,
    "-c:v", "libx264",
    "-preset", "fast",
    "-pix_fmt", "yuv420p",
    "-movflags", "+faststart",
    "-an",
    FINAL_VIDEO
], capture_output=True, text=True)

if result.returncode != 0:
    print(result.stderr[-3000:])
    raise RuntimeError("Video conversion failed.")

pd.DataFrame(records).to_csv(OUTPUT_CSV, index=False)

print("\nVIDEO CREATED")
print("Output:", FINAL_VIDEO)
print("Tracking CSV:", OUTPUT_CSV)
print("Frames processed:", frame_index)
print("Detection records:", len(records))
print("Track IDs:", len(set(r["track_id"] for r in records)))
print("Video size (MB):", round(os.path.getsize(FINAL_VIDEO) / 1024**2, 2))

In [ ]:
from IPython.display import Video, display

display(Video(
    "/content/drive/MyDrive/egg_task/video/final_tracking/egg_tracking_final_v2.mp4",
    embed=True,
    width=900
))

In [ ]:
import os

root = "/content/drive/MyDrive/egg_task"

for folder, dirs, files in os.walk(root):
    dirs[:] = [d for d in dirs if d not in [".git", "__pycache__"]]

    for filename in files:
        if filename.endswith(".py"):
            path = os.path.join(folder, filename)

            try:
                with open(path, "r", encoding="utf-8") as f:
                    lines = f.readlines()

                matches = [
                    (i + 1, line.rstrip())
                    for i, line in enumerate(lines)
                    if any(term in line.lower() for term in [
                        "next_id", "track_id", "centroid",
                        "hungarian", "max_disappeared",
                        "max_distance", "assign"
                    ])
                ]

                if matches:
                    print(f"\n--- {path} ---")
                    for line_no, line in matches[:35]:
                        print(f"{line_no}: {line}")

            except Exception as e:
                print("Could not read:", path, e)

In [ ]:
import os
import cv2
import pandas as pd
import numpy as np
import subprocess
from scipy.optimize import linear_sum_assignment

# --------------------------------------------------
# Paths
# --------------------------------------------------
BASE = "/content/drive/MyDrive/egg_task/video"

INPUT_VIDEO = f"{BASE}/input_video.mp4"
INPUT_CSV = f"{BASE}/improved_tracking/egg_tracks_improved.csv"

OUTPUT_DIR = f"{BASE}/final_tracking"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TEMP_VIDEO = f"{OUTPUT_DIR}/egg_tracking_two_ids_temp.mp4"
OUTPUT_VIDEO = f"{OUTPUT_DIR}/egg_tracking_two_ids.mp4"
OUTPUT_CSV = f"{OUTPUT_DIR}/egg_tracks_two_ids.csv"

assert os.path.exists(INPUT_VIDEO), f"Missing: {INPUT_VIDEO}"
assert os.path.exists(INPUT_CSV), f"Missing: {INPUT_CSV}"

# --------------------------------------------------
# Load and filter detections
# --------------------------------------------------
df = pd.read_csv(INPUT_CSV)

df["aspect"] = df["width"] / df["height"].clip(lower=1)

df = df[
    (df["area"] >= 15000) &
    (df["area"] <= 60000) &
    (df["circularity"] >= 0.55) &
    (df["width"].between(100, 320)) &
    (df["height"].between(80, 280)) &
    (df["aspect"].between(0.65, 1.8)) &
    (df["center_y"].between(400, 880))
].copy()

# Rank candidates by shape and plausible egg size.
df["quality"] = (
    df["circularity"]
    - abs(df["area"] - 40000) / 100000
)

# Retain at most two detections per frame.
df = (
    df.sort_values("quality", ascending=False)
      .groupby("frame", group_keys=False)
      .head(2)
)

detections_by_frame = {
    int(frame): group.to_dict("records")
    for frame, group in df.groupby("frame")
}

print("Filtered detections:", len(df))
print("Frames containing detections:", len(detections_by_frame))

# --------------------------------------------------
# Open video
# --------------------------------------------------
cap = cv2.VideoCapture(INPUT_VIDEO)

if not cap.isOpened():
    raise RuntimeError("Cannot open original video.")

fps = cap.get(cv2.CAP_PROP_FPS) or 24.0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

writer = cv2.VideoWriter(
    TEMP_VIDEO,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

if not writer.isOpened():
    cap.release()
    raise RuntimeError("Cannot create temporary video.")

# --------------------------------------------------
# Exactly two persistent track slots
# --------------------------------------------------
# Slot IDs are 1 and 2. They are not recycled as new IDs.
tracks = {
    1: {"center": None, "missed": 9999},
    2: {"center": None, "missed": 9999}
}

MAX_DISTANCE = 180
MAX_MISSED = 30

records = []
frame_number = 0

while True:
    ok, frame = cap.read()
    if not ok:
        break

    frame_number += 1
    candidates = detections_by_frame.get(frame_number, [])

    # Age tracks before matching.
    for tid in tracks:
        tracks[tid]["missed"] += 1

    assigned = {}
    used = set()

    # Match current detections to known IDs.
    active_ids = [
        tid for tid in tracks
        if tracks[tid]["center"] is not None
        and tracks[tid]["missed"] <= MAX_MISSED
    ]

    if active_ids and candidates:
        cost = np.zeros((len(active_ids), len(candidates)))

        for i, tid in enumerate(active_ids):
            old_center = tracks[tid]["center"]

            for j, det in enumerate(candidates):
                new_center = (
                    float(det["center_x"]),
                    float(det["center_y"])
                )
                cost[i, j] = np.linalg.norm(
                    np.array(old_center) - np.array(new_center)
                )

        rows, cols = linear_sum_assignment(cost)

        for i, j in zip(rows, cols):
            if cost[i, j] <= MAX_DISTANCE:
                tid = active_ids[i]
                assigned[j] = tid
                used.add(j)

    # Assign unmatched detections to an unused slot.
    # Prefer unused or longest-missing slots.
    for j, det in enumerate(candidates):
        if j in used:
            continue

        available = [
            tid for tid in tracks
            if tid not in assigned.values()
        ]

        if not available:
            continue

        # Prefer a slot that has never been initialized,
        # otherwise the slot missing for the longest time.
        tid = max(
            available,
            key=lambda t: (
                tracks[t]["center"] is None,
                tracks[t]["missed"]
            )
        )

        assigned[j] = tid

    # Draw only detections that fit one of the two slots.
    for j, det in enumerate(candidates):
        if j not in assigned:
            continue

        tid = assigned[j]

        x = int(det["x"])
        y = int(det["y"])
        w = int(det["width"])
        h = int(det["height"])

        center = (
            float(det["center_x"]),
            float(det["center_y"])
        )

        tracks[tid]["center"] = center
        tracks[tid]["missed"] = 0

        cv2.rectangle(
            frame, (x, y), (x + w, y + h),
            (0, 255, 0), 3
        )

        cv2.putText(
            frame, f"Egg {tid}",
            (x, max(25, y - 10)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8, (0, 255, 0), 2
        )

        records.append({
            "frame": frame_number,
            "timestamp_seconds": round((frame_number - 1) / fps, 3),
            "track_id": tid,
            "center_x": center[0],
            "center_y": center[1],
            "x": x,
            "y": y,
            "width": w,
            "height": h,
            "area": float(det["area"]),
            "circularity": float(det["circularity"])
        })

    # Intentionally no ROI rectangle.
    writer.write(frame)

    if frame_number % 240 == 0:
        print(f"Processed {frame_number}/{total_frames} frames")

cap.release()
writer.release()

# --------------------------------------------------
# Convert to H.264 for compatibility
# --------------------------------------------------
result = subprocess.run(
    [
        "ffmpeg", "-y",
        "-i", TEMP_VIDEO,
        "-c:v", "libx264",
        "-preset", "fast",
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
        "-an",
        OUTPUT_VIDEO
    ],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print(result.stderr[-2500:])
    raise RuntimeError("Video conversion failed.")

result_df = pd.DataFrame(records)
result_df.to_csv(OUTPUT_CSV, index=False)

print("\nCOMPLETE")
print("Video:", OUTPUT_VIDEO)
print("CSV:", OUTPUT_CSV)
print("Frames processed:", frame_number)
print("Detection records:", len(result_df))

if not result_df.empty:
    print("Track IDs:", sorted(result_df["track_id"].unique()))
    print("\nObservations per ID:")
    print(result_df.groupby("track_id").size())

In [ ]:
import cv2
import matplotlib.pyplot as plt

video_path = (
    "/content/drive/MyDrive/egg_task/video/"
    "final_tracking/egg_tracking_two_ids.mp4"
)

cap = cv2.VideoCapture(video_path)

for seconds in [12, 20, 25, 30, 40]:
    cap.set(cv2.CAP_PROP_POS_MSEC, seconds * 1000)
    ok, frame = cap.read()

    if ok:
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(12, 6))
        plt.imshow(frame)
        plt.title(f"Tracking at {seconds} seconds")
        plt.axis("off")
        plt.show()

cap.release()

In [ ]:
import cv2
import matplotlib.pyplot as plt
import os

video_path = "/content/drive/MyDrive/egg_task/video/final_tracking/egg_tracking_two_ids.mp4"

if not os.path.exists(video_path):
    raise FileNotFoundError(f"Video not found: {video_path}")

cap = cv2.VideoCapture(video_path)

print("FPS:", cap.get(cv2.CAP_PROP_FPS))
print("Total frames:", int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))

for sec in [12, 20, 25, 30, 40]:
    cap.set(cv2.CAP_PROP_POS_MSEC, sec * 1000)
    success, frame = cap.read()

    if success:
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(14, 7))
        plt.imshow(frame)
        plt.title(f"Tracking output at {sec} seconds")
        plt.axis("off")
        plt.show()
    else:
        print(f"Could not read frame at {sec} seconds")

cap.release()

In [ ]:
import os

base = "/content/drive/MyDrive/egg_task"

files_to_check = [
    "final_evaluation/evaluation_metrics.csv",
    "final_evaluation/evaluation_summary.json",
    "video/final_tracking/egg_tracking_two_ids.mp4",
    "video/final_tracking/egg_tracks_two_ids.csv",
    "egg_segmentation_final_submission.zip",
]

print("FINAL SUBMISSION FILE CHECK\n")

for file in files_to_check:
    path = os.path.join(base, file)
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"✅ {file} ({size_mb:.2f} MB)")
    else:
        print(f"❌ MISSING: {file}")